<a href="https://colab.research.google.com/github/NidaKhaan/intelligent-body-temperature-monitoring/blob/main/notebooks/research_log.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [ ]:
from google.colab import drive
drive.mount('/content/drive')

from pathlib import Path
ROOT = Path('/content/drive/MyDrive/Intelligent_Temperature_Monitor')
for d in ['datasets/charlotte_thermal','datasets/whu_iip','datasets/llvip','datasets/fda_physionet',
          'checkpoints','models','experiments','logs','metadata','outputs',
          'repo/src/utils','repo/src/detection','repo/src/alignment','repo/src/temperature',
          'repo/src/monitoring','repo/notebooks','repo/docs','repo/configs','repo/tests']:
    (ROOT/d).mkdir(parents=True, exist_ok=True)
print("Ready:", ROOT)

In [ ]:
%%writefile /content/drive/MyDrive/Intelligent_Temperature_Monitor/repo/src/utils/persistence.py
import json, os, time
from pathlib import Path

def atomic_write_json(path, data):
    """Temp file mein likho, phir replace: crash par purani file safe rehti hai."""
    path = Path(path)
    tmp = path.with_suffix(path.suffix + ".tmp")
    with open(tmp, "w") as f:
        json.dump(data, f, indent=2)
        f.flush(); os.fsync(f.fileno())
    os.replace(tmp, path)

class StageTracker:
    """Har stage ka status Drive par save karta hai; reconnect par done stages skip."""
    def __init__(self, path):
        self.path = Path(path)
        self.state = json.loads(self.path.read_text()) if self.path.exists() else {}

    def is_done(self, stage):
        return self.state.get(stage, {}).get("status") == "done"

    def mark_done(self, stage, **meta):
        self.state[stage] = {"status": "done",
                             "time": time.strftime("%Y-%m-%d %H:%M:%S"), **meta}
        atomic_write_json(self.path, self.state)

    def run(self, stage, fn, *args, **kwargs):
        if self.is_done(stage):
            print(f"[skip] {stage} (already done)")
            return
        print(f"[run]  {stage}")
        fn(*args, **kwargs)
        self.mark_done(stage)

In [ ]:
import sys
sys.path.insert(0, str(ROOT/'repo'))
from src.utils.persistence import StageTracker

t = StageTracker(ROOT/'metadata/progress.json')
t.run("test_stage", lambda: print("working..."))
t.run("test_stage", lambda: print("working..."))

In [ ]:
%%bash
cd /content/drive/MyDrive/Intelligent_Temperature_Monitor/repo
touch src/__init__.py src/utils/__init__.py
cat > .gitignore << 'EOF'
__pycache__/
*.pt
*.pth
*.onnx
datasets/
.env
*.tmp
EOF
cat > requirements.txt << 'EOF'
torch
torchvision
opencv-python-headless
mediapipe
numpy
pandas
matplotlib
fastapi
uvicorn
streamlit
EOF
git init -b main
git config user.name "Nida Sheraz"
git config user.email "nidasheraz1122@gmail.com"
git add .
git commit -m "chore: initialize project structure and persistence utilities"

In [ ]:
import subprocess
from google.colab import userdata

token = userdata.get('GITHUB_TOKEN')
repo = "NidaKhaan/intelligent-body-temperature-monitoring"
cwd = f"{ROOT}/repo"

subprocess.run(["git", "branch", "-M", "main"], cwd=cwd)
r = subprocess.run(["git", "push", f"https://{token}@github.com/{repo}.git", "main"],
                   cwd=cwd, capture_output=True, text=True)
print("exit code:", r.returncode)
print((r.stdout + r.stderr).replace(token, "***"))

In [ ]:
from google.colab import drive
drive.mount('/content/drive')
import sys
from pathlib import Path
ROOT = Path('/content/drive/MyDrive/Intelligent_Temperature_Monitor')
sys.path.insert(0, str(ROOT/'repo'))
from src.utils.persistence import StageTracker
tracker = StageTracker(ROOT/'metadata/progress.json')

In [ ]:
import subprocess, shutil
DEST = ROOT/'datasets/fda_physionet/raw'

def download_fda():
    if DEST.exists():
        shutil.rmtree(DEST)
    subprocess.run(["git","clone","--depth","1",
        "https://github.com/dbp-osel/Facial-and-oral-temperature-data.git", str(DEST)],
        check=True)

tracker.run("fda_download", download_fda)
for p in sorted(DEST.rglob("*")):
    if p.is_file() and ".git" not in p.parts:
        print(p.relative_to(DEST), f"{p.stat().st_size/1024:.0f} KB")

In [ ]:
import pandas as pd
csvs = sorted(p for p in DEST.rglob("*.csv"))
df = pd.read_csv(next(p for p in csvs if "FLIR" in p.name and "1and2" in p.name))
print(df.shape)
print(list(df.columns))
print(df.isna().sum().sort_values(ascending=False).head(8))
print(df.head(3).T.head(40))

In [ ]:
import pandas as pd, numpy as np, re
from src.utils.persistence import atomic_write_json

FDA = ROOT/'datasets/fda_physionet'
NAME_ROW, DATA_START = 2, 3          # file layout: row 2 = actual name, row 3 = data

def load_fda(csv_path):
    raw = pd.read_csv(csv_path, header=None)
    names = [str(n).strip() if pd.notna(n) else None for n in raw.iloc[NAME_ROW]]
    assert "SubjectID" in names, f"SubjectID header row mein nahi mila: {names[:5]}"
    data = raw.iloc[DATA_START:].reset_index(drop=True)
    dup = {n for n in names if n and names.count(n) > 1}
    assert not dup, f"duplicate column names: {dup}"
    data.columns = [n if n else f"_blank{i}" for i, n in enumerate(names)]
    data = data[[c for c in data.columns if not c.startswith("_blank")]]
    for c in data.columns:
        if c == "SubjectID":
            continue
        conv = pd.to_numeric(data[c], errors="coerce")
        if conv.notna().sum() >= 0.9 * data[c].notna().sum():
            data[c] = conv
    return data

def clean_all():
    for cam in ["FLIR", "ICI"]:
        d = load_fda(FDA/f"raw/{cam}_groups1and2.csv")
        d.to_csv(FDA/f"clean_{cam}_groups1and2.csv", index=False)
        print(cam, d.shape)


tracker.state.pop("fda_clean", None)
atomic_write_json(tracker.path, tracker.state)
tracker.run("fda_clean", clean_all)

flir = pd.read_csv(FDA/"clean_FLIR_groups1and2.csv")
print(flir.shape)
non_round = [c for c in flir.columns if not re.search(r'[1-4]$', c)]
print("Round-independent columns:", non_round)
print(flir[non_round].head(3).T)

In [ ]:
from sklearn.linear_model import LinearRegression
from sklearn.model_selection import KFold, cross_val_predict

def metrics(err):
    return dict(bias=err.mean(), MAE=err.abs().mean(), RMSE=np.sqrt((err**2).mean()))

oral_cols = [c for c in flir.columns if c.lower().startswith("aveoral")]
print("Oral reference columns found:", oral_cols)

if oral_cols:
    target = next((c for c in oral_cols if c.endswith("M")), oral_cols[0])
    print("Using target:", target)
    rows = []
    for prefix in ["canthiMax", "T_FH_Max", "T_Max"]:
        cols = [f"{prefix}{k}" for k in range(1, 5) if f"{prefix}{k}" in flir.columns]
        x = flir[cols].mean(axis=1)
        ok = x.notna() & flir[target].notna()
        x, y = x[ok], flir.loc[ok, target]
        pred = cross_val_predict(LinearRegression(), x.to_frame(), y,
                                 cv=KFold(5, shuffle=True, random_state=42))
        rows.append({"feature": prefix, "n": int(ok.sum()),
                     **{f"raw_{k}": v for k, v in metrics(x - y).items()},
                     **{f"cal_{k}": v for k, v in metrics(pd.Series(pred, index=y.index) - y).items()}})
    print(pd.DataFrame(rows).round(3).to_string(index=False))
else:
    print("Oral column couldnt be found: Please check Cell 9 'Round-independent columns' output.")

In [ ]:
%%writefile /content/drive/MyDrive/Intelligent_Temperature_Monitor/repo/src/temperature/fda_baseline.py
import numpy as np, pandas as pd
from sklearn.linear_model import LinearRegression
from sklearn.model_selection import KFold, cross_val_predict

FEVER_C = 37.5

def metrics(err):
    err = pd.Series(err)
    return dict(bias=err.mean(), MAE=err.abs().mean(), RMSE=float(np.sqrt((err ** 2).mean())))

def facial_feature(df, prefix):
    cols = [f"{prefix}{k}" for k in range(1, 5) if f"{prefix}{k}" in df.columns]
    return df[cols].mean(axis=1)

def cv_calibrate(x, y, seed=42):
    pred = cross_val_predict(LinearRegression(), x.to_frame(), y,
                             cv=KFold(5, shuffle=True, random_state=seed))
    return pd.Series(pred, index=y.index)

def bootstrap_ci(values, stat=np.mean, n=2000, seed=0):
    rng = np.random.default_rng(seed)
    v = np.asarray(values)
    return np.percentile([stat(v[rng.integers(0, len(v), len(v))]) for _ in range(n)], [2.5, 97.5])

def screening(pred, y, thr):
    pos, flag = y >= FEVER_C, pred >= thr
    tp, fn = (pos & flag).sum(), (pos & ~flag).sum()
    tn, fp = (~pos & ~flag).sum(), (~pos & flag).sum()
    return dict(n_pos=int(pos.sum()), sensitivity=float(tp / max(tp + fn, 1)),
                specificity=float(tn / max(tn + fp, 1)))

In [ ]:
import numpy as np, pandas as pd, matplotlib.pyplot as plt, importlib, json
(ROOT/'repo/src/temperature/__init__.py').touch()
import src.temperature.fda_baseline as fb; importlib.reload(fb)
from src.utils.persistence import atomic_write_json

def baseline_eval():
    flir = pd.read_csv(FDA/"clean_FLIR_groups1and2.csv")
    out = {"target": "aveOralM", "cv": "5-fold, shuffle, seed 42"}
    for prefix in ["canthiMax", "T_Max"]:
        x, y = fb.facial_feature(flir, prefix), flir["aveOralM"]
        ok = x.notna() & y.notna()
        x, y = x[ok], y[ok]
        pred = fb.cv_calibrate(x, y)
        err, fever = pred - y, y >= fb.FEVER_C
        lo, hi = fb.bootstrap_ci(err.abs())
        grp1 = flir.loc[y.index, "T_atm"] < 24
        out[prefix] = {
            "n": int(len(y)), "n_febrile": int(fever.sum()),
            **{k: float(v) for k, v in fb.metrics(err).items()},
            "MAE_CI95": [float(lo), float(hi)],
            "MAE_afebrile": float(err[~fever].abs().mean()),
            "MAE_febrile": float(err[fever].abs().mean()),
            "bias_febrile": float(err[fever].mean()),
            "MAE_ambient_lt24": float(err[grp1].abs().mean()),
            "MAE_ambient_ge24": float(err[~grp1].abs().mean()),
            "screening_thr_37.5": fb.screening(pred, y, 37.5),
            "screening_thr_37.2_exploratory": fb.screening(pred, y, 37.2),
        }
        if prefix == "T_Max":
            plt.figure(figsize=(6, 4))
            plt.scatter(y, err, s=8, alpha=.5)
            plt.axhline(0, c="k", lw=1); plt.axvline(37.5, c="r", ls="--", lw=1)
            plt.xlabel("Oral temperature (C)"); plt.ylabel("Calibrated prediction - oral (C)")
            plt.title("FDA FLIR: calibration error vs oral temperature")
            plt.tight_layout(); plt.savefig(ROOT/"outputs/fda_error_vs_oral.png", dpi=150); plt.close()
    atomic_write_json(ROOT/"experiments/fda_baseline_v1.json", out)

tracker.run("fda_baseline_v1", baseline_eval)
print(json.dumps(json.load(open(ROOT/"experiments/fda_baseline_v1.json")), indent=2))

In [ ]:
import subprocess
from google.colab import userdata
token = userdata.get('GITHUB_TOKEN'); cwd = f"{ROOT}/repo"
def sh(*a): return subprocess.run(a, cwd=cwd, capture_output=True, text=True)
sh("git", "add", "."); print(sh("git", "commit", "-m", "feat: add FDA facial-to-oral temperature baseline evaluation").stdout)
r = sh("git", "push", f"https://{token}@github.com/NidaKhaan/intelligent-body-temperature-monitoring.git", "main")
print(r.returncode, (r.stdout + r.stderr).replace(token, "***"))

In [ ]:
def single_round_eval():
    flir = pd.read_csv(FDA/"clean_FLIR_groups1and2.csv")
    y_all, out = flir["aveOralM"], {}
    for prefix in ["canthiMax", "T_Max"]:
        for k in range(1, 5):
            col = f"{prefix}{k}"
            ok = flir[col].notna() & y_all.notna()
            x, y = flir.loc[ok, col], y_all[ok]
            m = fb.metrics(fb.cv_calibrate(x, y) - y)
            out[col] = {"n": int(len(y)), "MAE": float(m["MAE"]), "RMSE": float(m["RMSE"])}
    atomic_write_json(ROOT/"experiments/fda_single_round_v1.json", out)

tracker.run("fda_single_round_v1", single_round_eval)
print(pd.DataFrame(json.load(open(ROOT/"experiments/fda_single_round_v1.json"))).T.round(3))

In [ ]:
import shutil, subprocess
from google.colab import userdata
res = ROOT/"repo/docs/results"; res.mkdir(parents=True, exist_ok=True)
for f in ["fda_baseline_v1.json", "fda_single_round_v1.json"]:
    shutil.copy(ROOT/"experiments"/f, res/f)
shutil.copy(ROOT/"outputs/fda_error_vs_oral.png", res/"fda_error_vs_oral.png")

token = userdata.get('GITHUB_TOKEN'); cwd = f"{ROOT}/repo"
def sh(*a): return subprocess.run(a, cwd=cwd, capture_output=True, text=True)
sh("git", "add", ".")
print(sh("git", "commit", "-m", "docs: add FDA baseline and single-round evaluation results").stdout)
r = sh("git", "push", f"https://{token}@github.com/NidaKhaan/intelligent-body-temperature-monitoring.git", "main")
print(r.returncode, (r.stdout + r.stderr).replace(token, "***"))

In [ ]:
import subprocess, shutil
CH = ROOT/'datasets/charlotte_thermal'

def clone_charlotte_repo():
    dest = CH/'repo_meta'
    if dest.exists():
        shutil.rmtree(dest)
    subprocess.run(["git", "clone", "--depth", "1",
                    "https://github.com/TeCSAR-UNCC/UNCC-ThermalFace.git", str(dest)], check=True)

tracker.run("charlotte_repo_meta", clone_charlotte_repo)

meta = CH/'repo_meta'
for p in sorted(meta.rglob("*")):
    if p.is_file() and ".git" not in p.parts:
        print(p.relative_to(meta), f"{p.stat().st_size/1024:.0f} KB")

readme = next(meta.glob("README*"))
print("\n----- README -----\n")
print(readme.read_text(errors="ignore")[:8000])

In [ ]:
import gdown, json, collections
from pathlib import Path
from src.utils.persistence import atomic_write_json

URL = "https://drive.google.com/drive/folders/1UE4BsRsJBSEyLtcAkMQ_jYIBKwKjwyQc"

def list_charlotte():
    files = gdown.download_folder(url=URL, skip_download=True, quiet=True, remaining_ok=True)
    manifest = [{"id": f.id, "path": f.path} for f in files]
    atomic_write_json(ROOT/"metadata/charlotte_manifest.json", manifest)

tracker.run("charlotte_listing_v1", list_charlotte)

m = json.load(open(ROOT/"metadata/charlotte_manifest.json"))
print("files listed (lower bound):", len(m))
print(collections.Counter(Path(x["path"]).suffix.lower() for x in m))
tops = collections.Counter(Path(x["path"]).parts[0] if len(Path(x["path"]).parts) > 1 else "<root>" for x in m)
print(tops.most_common(20))
for x in m[:15]:
    print(x["path"])

In [ ]:
import gdown, json, pandas as pd
from pathlib import Path

m = json.load(open(ROOT/"metadata/charlotte_manifest.json"))
CH = ROOT/'datasets/charlotte_thermal'
(CH/'annotations').mkdir(parents=True, exist_ok=True)

def dl_csvs():
    for x in m:
        name = Path(x["path"]).name
        if not name.endswith(".csv"):
            continue
        out = CH/'annotations'/name
        if out.exists() and out.stat().st_size > 0:
            continue                      # pehle se download hai
        gdown.download(id=x["id"], output=str(out), quiet=True)
        assert out.exists() and out.stat().st_size > 0, f"download failed: {name}"

tracker.run("charlotte_csvs_v1", dl_csvs)

for p in sorted((CH/'annotations').glob("S*.csv"), key=lambda p: int(p.stem[1:])):
    print(p.name, f"{p.stat().st_size/1024:.0f} KB")
s1 = pd.read_csv(CH/'annotations/S1.csv')
print(s1.shape)
print(list(s1.columns))
print(s1.head(3).T)

In [ ]:
import gdown, shutil, zipfile, collections
from pathlib import Path

zip_id = next(x["id"] for x in m if x["path"].endswith(".zip"))
LOCAL = Path("/content/cache"); LOCAL.mkdir(exist_ok=True)
local_zip = LOCAL/"Charlotte-ThermalFace.zip"
drive_zip = CH/"Charlotte-ThermalFace.zip"

def fetch_zip():
    if not local_zip.exists():
        gdown.download(id=zip_id, output=str(local_zip), quiet=False, resume=True)
    assert zipfile.is_zipfile(local_zip), "zip mukammal/valid nahi, dobara chalayein"
    size = local_zip.stat().st_size
    free = shutil.disk_usage(ROOT).free
    print(f"zip size: {size/1e9:.2f} GB | Drive free: {free/1e9:.2f} GB")
    if free < size * 1.1:
        raise RuntimeError("Drive mein jagah kam hai: copy nahi kiya. Size paste karein, hal nikalte hain.")
    tmp = drive_zip.with_suffix(".zip.part")
    shutil.copyfile(local_zip, tmp)
    tmp.replace(drive_zip)                # copy poori hone par hi final naam

tracker.run("charlotte_zip_v1", fetch_zip)

zf = zipfile.ZipFile(drive_zip)
names = zf.namelist()
print("entries:", len(names))
print(collections.Counter(Path(n).suffix.lower() for n in names).most_common(8))
print(collections.Counter(n.split("/")[0] for n in names).most_common(12))
for n in names[:12]:
    print(n)

In [ ]:
import cv2, numpy as np, pandas as pd, matplotlib.pyplot as plt
from pathlib import Path

OUT = Path("/content/cache/charlotte")
S1 = next(p for p in OUT.rglob("S1") if p.is_dir())
tiffs = {p.stem: p for p in S1.glob("*.tiff")}
ann = pd.read_csv(CH/"annotations/S1.csv")
xc, yc = [f"x{i}" for i in range(73)], [f"y{i}" for i in range(73)]

n_valid = ann[xc].notna().sum(axis=1)
print("landmarks per row:\n", n_valid.value_counts().to_string())
print("duplicate IDs:", int(ann["ID"].duplicated().sum()),
      "| IDs without tiff:", sorted(set(ann["ID"]) - set(tiffs))[:5])

def face_stats(row, scale=0.01):
    img = cv2.imread(str(tiffs[row["ID"]]), cv2.IMREAD_UNCHANGED)
    xs, ys = row[xc].to_numpy(float), row[yc].to_numpy(float)
    x0, x1 = max(int(np.nanmin(xs)), 0), int(np.nanmax(xs))
    y0, y1 = max(int(np.nanmin(ys)), 0), int(np.nanmax(ys))
    face = img[y0:y1, x0:x1].astype(float) * scale - 273.15
    return img, (xs, ys), (x0, y0, x1, y1), face

fig, axes = plt.subplots(1, 2, figsize=(10, 4.5))
for ax, n in zip(axes, [73, 43]):
    sel = ann[n_valid == n]
    if sel.empty:
        ax.set_title(f"no rows with {n} landmarks"); continue
    row = next(r for _, r in sel.iterrows() if r["ID"] in tiffs)
    img, (xs, ys), box, face = face_stats(row)
    print(f"\n[{n} landmarks] {row['ID']} | bbox {box} | env {row['env-temp']} C | dist {row['Distance']} m")
    print(f"  face median {np.median(face):.1f} C, max {face.max():.1f} C, "
          f"image min {img.min()*0.01-273.15:.1f} C  (assumed 10 mK/count)")
    ax.imshow(img, cmap="gray"); ax.scatter(xs, ys, s=6, c="r")
    ax.set_title(f"{n} landmarks"); ax.axis("off")
plt.tight_layout(); plt.show()

In [ ]:
%%writefile /content/drive/MyDrive/Intelligent_Temperature_Monitor/repo/src/temperature/radiometric.py
"""Raw 16-bit thermal counts -> degrees Celsius.

SCALE_K_PER_COUNT = 0.01 is an assumption: it gives plausible skin temperatures on
Charlotte-ThermalFace and agrees with FDA raw T_Max readings. It is NOT validated
against a blackbody reference.
"""
import numpy as np

KELVIN_OFFSET = 273.15
SCALE_K_PER_COUNT = 0.01

def raw_to_celsius(raw, scale=SCALE_K_PER_COUNT):
    return np.asarray(raw, dtype=np.float64) * scale - KELVIN_OFFSET

In [ ]:
%%writefile /content/drive/MyDrive/Intelligent_Temperature_Monitor/repo/tests/test_radiometric.py
import numpy as np
from src.temperature.radiometric import raw_to_celsius

def test_known_value():
    assert abs(raw_to_celsius(29815) - 25.0) < 1e-9

def test_array_and_dtype():
    out = raw_to_celsius(np.array([29315, 30915], dtype=np.uint16))
    assert np.allclose(out, [20.0, 36.0])

def test_no_uint16_overflow():

    assert raw_to_celsius(np.uint16(65535)) > 300

In [ ]:
%%bash
pip -q install pytest
cd /content/drive/MyDrive/Intelligent_Temperature_Monitor/repo && python -m pytest -q

In [ ]:
import cv2, zipfile, numpy as np, pandas as pd, importlib
from pathlib import Path
import src.temperature.radiometric as rd; importlib.reload(rd)

ROOT = Path('/content/drive/MyDrive/Intelligent_Temperature_Monitor')
CH = ROOT/'datasets/charlotte_thermal'
from src.utils.persistence import StageTracker
tracker = StageTracker(ROOT/'metadata/progress.json')

OUT = Path("/content/cache/charlotte")
if not (OUT/".extracted_ok").exists():
    zipfile.ZipFile(CH/"Charlotte-ThermalFace.zip").extractall(OUT); (OUT/".extracted_ok").touch()
xc, yc = [f"x{i}" for i in range(73)], [f"y{i}" for i in range(73)]

def charlotte_scan():
    rows, skipped = [], {"no_tiff": 0, "odd_landmarks": 0, "tiny_box": 0}
    for s in range(1, 11):
        sdir = next(p for p in OUT.rglob(f"S{s}") if p.is_dir())
        ann = pd.read_csv(CH/f"annotations/S{s}.csv").drop_duplicates("ID", keep="first")
        for _, r in ann.iterrows():
            tif = sdir/f"{r['ID']}.tiff"
            if not tif.exists(): skipped["no_tiff"] += 1; continue
            xs, ys = r[xc].to_numpy(float), r[yc].to_numpy(float)
            n = int(np.isfinite(xs).sum())
            if n not in (43, 73): skipped["odd_landmarks"] += 1; continue
            img = cv2.imread(str(tif), cv2.IMREAD_UNCHANGED); h, w = img.shape
            x0, x1 = max(int(np.nanmin(xs)), 0), min(int(np.nanmax(xs)), w)
            y0, y1 = max(int(np.nanmin(ys)), 0), min(int(np.nanmax(ys)), h)
            if x1 - x0 < 5 or y1 - y0 < 5: skipped["tiny_box"] += 1; continue
            face = rd.raw_to_celsius(img[y0:y1, x0:x1])
            rows.append(dict(subject=s, id=r["ID"], n_lm=n, distance=r["Distance"], env_temp=r["env-temp"],
                             rh=r["RH"], sensation=r["Sensation"], box_w=x1-x0, box_h=y1-y0,
                             face_max=face.max(), face_p95=np.percentile(face, 95), face_median=np.median(face)))
    pd.DataFrame(rows).to_csv(ROOT/"experiments/charlotte_scan_v1.csv", index=False)
    print("skipped:", skipped)

tracker.run("charlotte_scan_v1", charlotte_scan)
df = pd.read_csv(ROOT/"experiments/charlotte_scan_v1.csv")
print(df.shape, "| subjects:", df.subject.nunique())
by_dist = df.groupby(df.distance.round(1)).agg(n=("id", "size"), box_h=("box_h", "mean"),
                                               max_c=("face_max", "mean"), p95_c=("face_p95", "mean"),
                                               median_c=("face_median", "mean"))
print(by_dist.round(2).to_string())
print("\ncorr(face_max, env_temp) within distance<=1.5:",
      round(df[df.distance <= 1.5][["face_max", "env_temp"]].corr().iloc[0, 1], 3))

In [ ]:
import subprocess
from google.colab import userdata
token = userdata.get('GITHUB_TOKEN'); cwd = f"{ROOT}/repo"
def sh(*a): return subprocess.run(a, cwd=cwd, capture_output=True, text=True)
sh("git", "add", "src", "tests")
print(sh("git", "commit", "-m", "feat: add radiometric conversion with tests").stdout)
r = sh("git", "push", f"https://{token}@github.com/NidaKhaan/intelligent-body-temperature-monitoring.git", "main")
print(r.returncode, (r.stdout + r.stderr).replace(token, "***"))

In [ ]:
import pandas as pd
from pathlib import Path
OUT = Path("/content/cache/charlotte")

for s in range(1, 11):
    ann = pd.read_csv(CH/f"annotations/S{s}.csv")
    stems = {p.stem for d in OUT.rglob(f"S{s}") if d.is_dir() for p in d.glob("*.tiff")}
    raw_ids = ann["ID"]
    ids = set(raw_ids.dropna().astype(str).str.strip())
    dist = pd.to_numeric(ann["Distance"], errors="coerce")
    bad_d = int(((dist < 0.9) | (dist > 7.0) | dist.isna()).sum())
    print(f"S{s}: rows={len(ann)} nan_id={int(raw_ids.isna().sum())} "
          f"dup_id={int(raw_ids.dropna().duplicated().sum())} uniq_ids={len(ids)} "
          f"tiffs={len(stems)} matched={len(ids & stems)} bad_dist_rows={bad_d} id_dtype={raw_ids.dtype}")
    if len(ids & stems) < 0.9 * len(ids):
        print("   ids  :", sorted(ids)[:4])
        print("   tiffs:", sorted(stems)[:4])

s2 = pd.read_csv(CH/"annotations/S2.csv")
d2 = pd.to_numeric(s2["Distance"], errors="coerce")
print("\nS2 rows with Distance > 7:")
print(s2.loc[d2 > 7, ["ID", "Distance", "RH", "Airflow", "env-temp", "Sensation"]].head(6).to_string())

In [ ]:
import cv2, re, zipfile, numpy as np, pandas as pd, importlib
import matplotlib.pyplot as plt
from pathlib import Path
import src.temperature.radiometric as rd; importlib.reload(rd)

OUT = Path("/content/cache/charlotte")
if not (OUT/".extracted_ok").exists():
    zipfile.ZipFile(CH/"Charlotte-ThermalFace.zip").extractall(OUT); (OUT/".extracted_ok").touch()
xc, yc = [f"x{i}" for i in range(73)], [f"y{i}" for i in range(73)]
MIN_MATCH = 0.95

def candidates(raw_id):
    s = re.sub(r"\.(jpe?g|tiff?|png)$", "", str(raw_id).strip(), flags=re.I)
    yield s, "exact"
    if re.match(r"^RS\d", s): yield "R" + s[2:], "drop_S"
    if re.match(r"^[A-QS-Z]\d", s): yield "R" + s[1:], "first_letter_to_R"

def charlotte_scan():
    kept, quar, report = [], [], []
    for s in range(1, 11):
        sdir = next(p for p in OUT.rglob(f"S{s}") if p.is_dir())
        tiffs = {p.stem: p for p in sdir.glob("*.tiff")}
        ann = pd.read_csv(CH/f"annotations/S{s}.csv")
        n_nan = int(ann["ID"].isna().sum())
        ann = ann[ann["ID"].notna()].copy()
        ann["raw_id"] = ann["ID"].astype(str).str.strip()
        conflict = set(ann.groupby("raw_id")[xc + yc].nunique(dropna=False).max(axis=1)
                          .loc[lambda v: v > 1].index)           # duplicate IDs jinke landmarks alag hain
        ann = ann.drop_duplicates("raw_id", keep="first")

        matched, rules = [], {}
        for _, r in ann.iterrows():
            if r["raw_id"] in conflict:
                quar.append((s, r["raw_id"], "conflicting_duplicate")); continue
            hit = next(((c, rule) for c, rule in candidates(r["raw_id"]) if c in tiffs), None)
            if hit is None:
                quar.append((s, r["raw_id"], "no_tiff")); continue
            rules[hit[1]] = rules.get(hit[1], 0) + 1
            matched.append((r, hit[0]))
        rate = len(matched) / max(len(ann), 1)
        report.append(dict(subject=s, ann_unique=len(ann), nan_id=n_nan, conflict=len(conflict),
                           matched=len(matched), match_rate=round(rate, 3), rules=rules))
        if rate < MIN_MATCH:
            unm = [q[1] for q in quar if q[0] == s and q[2] == "no_tiff"][:6]
            print(f"WARN S{s}: match_rate {rate:.2f} < {MIN_MATCH}, subject EXCLUDED. unmatched ids: {unm}")
            print(f"     tiff stems not annotated (sample): {sorted(set(tiffs) - {m[1] for m in matched})[:6]}")
            continue

        for r, stem in matched:
            dist = pd.to_numeric(r["Distance"], errors="coerce")
            if not (0.9 <= dist <= 7.0):
                quar.append((s, r["raw_id"], "bad_distance")); continue
            xs, ys = r[xc].to_numpy(float), r[yc].to_numpy(float)
            ok = np.isfinite(xs) & np.isfinite(ys); n = int(ok.sum())
            if n not in (43, 73):
                quar.append((s, r["raw_id"], f"odd_landmarks_{n}")); continue
            img = cv2.imread(str(tiffs[stem]), cv2.IMREAD_UNCHANGED)
            if img is None or img.dtype != np.uint16:
                quar.append((s, r["raw_id"], "unreadable_or_not_uint16")); continue
            h, w = img.shape
            pts = np.stack([np.clip(xs[ok], 0, w - 1), np.clip(ys[ok], 0, h - 1)], 1).astype(np.int32)
            mask = np.zeros((h, w), np.uint8); cv2.fillConvexPoly(mask, cv2.convexHull(pts), 1)
            if mask.sum() < 25:
                quar.append((s, r["raw_id"], "tiny_hull")); continue
            hull_t = rd.raw_to_celsius(img[mask.astype(bool)])
            x0, y0 = pts.min(0); x1, y1 = pts.max(0)
            rect_t = rd.raw_to_celsius(img[y0:y1 + 1, x0:x1 + 1])
            kept.append(dict(subject=s, id=r["raw_id"], n_lm=n, distance=float(dist),
                             env_temp=pd.to_numeric(r["env-temp"], errors="coerce"),
                             hull_px=int(mask.sum()), box_h=int(y1 - y0 + 1),
                             max_hull=hull_t.max(), p95_hull=np.percentile(hull_t, 95),
                             median_hull=np.median(hull_t), max_rect=rect_t.max()))
    pd.DataFrame(kept).to_csv(ROOT/"experiments/charlotte_scan_v2.csv", index=False)
    pd.DataFrame(quar, columns=["subject", "id", "reason"]).to_csv(ROOT/"experiments/charlotte_scan_v2_quarantine.csv", index=False)
    pd.DataFrame(report).to_csv(ROOT/"experiments/charlotte_scan_v2_report.csv", index=False)

tracker.run("charlotte_scan_v2", charlotte_scan)

print(pd.read_csv(ROOT/"experiments/charlotte_scan_v2_report.csv").to_string(index=False))
q = pd.read_csv(ROOT/"experiments/charlotte_scan_v2_quarantine.csv")
print("\nquarantine reasons:\n", q.reason.value_counts().to_string())

df = pd.read_csv(ROOT/"experiments/charlotte_scan_v2.csv"); df["d"] = df.distance.round(1)
print("\nkept:", df.shape, "| subjects:", sorted(df.subject.unique()))
print("\nn_lm by subject:\n", pd.crosstab(df.subject, df.n_lm).to_string())
print("\nhull vs rectangle max (mean diff, C):", round((df.max_rect - df.max_hull).mean(), 3))
print(df.groupby(["n_lm", "d"]).agg(n=("id", "size"), hull_px=("hull_px", "mean"),
      max_hull=("max_hull", "mean"), median_hull=("median_hull", "mean")).round(2).to_string())
sub = df[df.d.isin([1.0, 6.6])].pivot_table(index="subject", columns="d", values="max_hull")
sub["drop_1m_to_6.6m"] = sub[1.0] - sub[6.6]
print("\nper-subject:\n", sub.round(2).to_string())

In [ ]:
import cv2, numpy as np, pandas as pd, matplotlib.pyplot as plt, re
fig, axes = plt.subplots(1, 3, figsize=(13, 4))
for ax, s in zip(axes, [6, 7, 8]):
    sdir = next(p for p in OUT.rglob(f"S{s}") if p.is_dir())
    tiffs = {p.stem: p for p in sdir.glob("*.tiff")}
    ann = pd.read_csv(CH/f"annotations/S{s}.csv").dropna(subset=["ID"])
    for _, r in ann.iloc[::97].iterrows():
        hit = next((c for c, _ in candidates(r["ID"]) if c in tiffs), None)
        if hit:
            img = cv2.imread(str(tiffs[hit]), cv2.IMREAD_UNCHANGED)
            xs, ys = r[xc].to_numpy(float), r[yc].to_numpy(float)
            ax.imshow(img, cmap="gray"); ax.scatter(xs, ys, s=5, c="r")
            ax.set_title(f"S{s}: {r['ID']} -> {hit}"); ax.axis("off"); break
plt.tight_layout(); plt.show()

In [ ]:
import cv2, numpy as np, pandas as pd, re
from pathlib import Path
import src.temperature.radiometric as rd

OUT = Path("/content/cache/charlotte")
df = pd.read_csv(ROOT/"experiments/charlotte_scan_v2.csv"); df["d"] = df.distance.round(1)
xc, yc = [f"x{i}" for i in range(73)], [f"y{i}" for i in range(73)]

def hull_max(img, xs, ys):
    h, w = img.shape
    ok = np.isfinite(xs) & np.isfinite(ys)
    pts = np.stack([np.clip(xs[ok], 0, w - 1), np.clip(ys[ok], 0, h - 1)], 1).astype(np.int32)
    mask = np.zeros((h, w), np.uint8); cv2.fillConvexPoly(mask, cv2.convexHull(pts), 1)
    return rd.raw_to_celsius(img[mask.astype(bool)]).max() if mask.sum() else np.nan

def synth():
    out = []
    for s in sorted(df[df.d == 6.6].subject.unique()):
        sd = df[(df.subject == s) & (df.n_lm == 73)]
        near, far = sd[sd.d == 1.0], sd[sd.d == 6.6]
        if len(near) < 20 or len(far) < 20: continue
        f = far.box_h.mean() / near.box_h.mean()
        sdir = next(p for p in OUT.rglob(f"S{s}") if p.is_dir())
        tiffs = {p.stem: p for p in sdir.glob("*.tiff")}
        ann = pd.read_csv(CH/f"annotations/S{s}.csv").dropna(subset=["ID"])
        ann["raw_id"] = ann["ID"].astype(str).str.strip()
        full, small = [], []
        for rid in near.id.sample(min(60, len(near)), random_state=0):
            r = ann[ann.raw_id == rid].iloc[0]
            stem = next(c for c in [re.sub(r"\.(jpe?g|tiff?)$", "", rid), "R" + re.sub(r"\.(jpe?g|tiff?)$", "", rid)[1:],
                                    "R" + re.sub(r"\.(jpe?g|tiff?)$", "", rid)[2:]] if c in tiffs)
            img = cv2.imread(str(tiffs[stem]), cv2.IMREAD_UNCHANGED)
            xs, ys = r[xc].to_numpy(float), r[yc].to_numpy(float)
            sm = cv2.resize(img, None, fx=f, fy=f, interpolation=cv2.INTER_AREA)
            full.append(hull_max(img, xs, ys)); small.append(hull_max(sm, xs * f, ys * f))
        out.append(dict(subject=s, scale=round(f, 3),
                        synthetic_drop=np.nanmean(full) - np.nanmean(small),
                        real_drop=near.max_hull.mean() - far.max_hull.mean()))
    pd.DataFrame(out).to_csv(ROOT/"experiments/charlotte_synthetic_downsample_v1.csv", index=False)

tracker.run("charlotte_synth_v1", synth)
print(pd.read_csv(ROOT/"experiments/charlotte_synthetic_downsample_v1.csv").round(3).to_string(index=False))

In [ ]:
import subprocess, shutil
from google.colab import userdata
res = ROOT/"repo/docs/results"
agg = df.groupby(["n_lm", "d"]).agg(n=("id", "size"), mean_hull_px=("hull_px", "mean"),
        mean_max_c=("max_hull", "mean"), mean_median_c=("median_hull", "mean")).round(2).reset_index()
agg.to_csv(res/"charlotte_distance_summary.csv", index=False)
shutil.copy(ROOT/"experiments/charlotte_synthetic_downsample_v1.csv", res/"charlotte_synthetic_downsample_v1.csv")

token = userdata.get('GITHUB_TOKEN'); cwd = f"{ROOT}/repo"
def sh(*a): return subprocess.run(a, cwd=cwd, capture_output=True, text=True)
sh("git", "add", "docs")
print(sh("git", "commit", "-m", "docs: add Charlotte distance effect summary and downsampling experiment").stdout)
r = sh("git", "push", f"https://{token}@github.com/NidaKhaan/intelligent-body-temperature-monitoring.git", "main")
print(r.returncode, (r.stdout + r.stderr).replace(token, "***"))

In [ ]:
import subprocess, shutil
LL = ROOT/'datasets/llvip'

def clone_llvip_meta():
    dest = LL/'repo_meta'
    if dest.exists():
        shutil.rmtree(dest)
    subprocess.run(["git", "clone", "--depth", "1", "https://github.com/bupt-ai-cz/LLVIP.git", str(dest)], check=True)

tracker.run("llvip_repo_meta", clone_llvip_meta)
meta = LL/'repo_meta'
f = next(meta.rglob("download_dataset.md"), None)
print(f.read_text(errors="ignore")[:6000] if f else "download_dataset.md nahi mili: files:\n" +
      "\n".join(str(p.relative_to(meta)) for p in sorted(meta.glob("*"))[:40]))

In [ ]:
import zipfile, shutil, collections
from pathlib import Path

LOCAL = Path("/content/cache")
LOCAL.mkdir(exist_ok=True)
llvip_zip = LOCAL/"LLVIP.zip"

# Google Drive sources to look for the pre-existing zip
drive_sources = [
    ROOT/'datasets/llvip/LLVIP.zip',
    ROOT/'LLVIP.zip',
    Path('/content/drive/MyDrive/LLVIP.zip')
]

if not (llvip_zip.exists() and zipfile.is_zipfile(llvip_zip)):
    found = False
    for src in drive_sources:
        if src.exists() and zipfile.is_zipfile(src):
            print(f"Found ZIP on Drive: {src} ({src.stat().st_size/1e9:.2f} GB). Copying to local cache...")
            shutil.copyfile(src, llvip_zip)
            found = True
            break
    if not found:
        raise FileNotFoundError("LLVIP.zip aapki Drive par nahi mili. Meherbani karke use 'Intelligent_Temperature_Monitor/datasets/llvip/' folder mein upload karein.")

assert zipfile.is_zipfile(llvip_zip), "ZIP file mukammal ya valid nahi hai!"
print(f"Local ZIP ready: {llvip_zip.stat().st_size/1e9:.2f} GB | Local free: {shutil.disk_usage('/content').free/1e9:.0f} GB")

zf = zipfile.ZipFile(llvip_zip); names = zf.namelist()
print("Entries in ZIP:", len(names))
print(collections.Counter(Path(n).suffix.lower() for n in names).most_common(6))
print(collections.Counter("/".join(Path(n).parts[:2]) for n in names).most_common(12))
for n in names[:6] + names[len(names)//2:len(names)//2 + 3]:
    print(n)

In [ ]:
%%bash
pip -q install mediapipe 2>&1 | tail -3
python -c "import mediapipe as mp; print('mediapipe', mp.__version__)"
python -c "import torch, torchvision; print('torch', torch.__version__, torchvision.__version__, 'cuda:', torch.cuda.is_available())"

In [ ]:
%%writefile /content/drive/MyDrive/Intelligent_Temperature_Monitor/repo/src/detection/person_detector.py
"""COCO-pretrained torchvision Faster R-CNN, filtered to the 'person' class."""
import numpy as np
import torch
from torchvision.models.detection import (
    fasterrcnn_mobilenet_v3_large_fpn, FasterRCNN_MobileNet_V3_Large_FPN_Weights,
    fasterrcnn_resnet50_fpn_v2, FasterRCNN_ResNet50_FPN_V2_Weights,
)

PERSON_LABEL = 1  # COCO category id for "person" in torchvision's label map

class PersonDetector:
    def __init__(self, variant="mobilenet", score_thr=0.5, device=None):
        self.device = device or ("cuda" if torch.cuda.is_available() else "cpu")
        if variant == "mobilenet":
            self.model = fasterrcnn_mobilenet_v3_large_fpn(
                weights=FasterRCNN_MobileNet_V3_Large_FPN_Weights.DEFAULT)
        elif variant == "resnet50":
            self.model = fasterrcnn_resnet50_fpn_v2(
                weights=FasterRCNN_ResNet50_FPN_V2_Weights.DEFAULT)
        else:
            raise ValueError(f"unknown variant: {variant}")
        self.model.to(self.device).eval()
        self.score_thr = score_thr

    @torch.inference_mode()
    def detect(self, rgb: np.ndarray):
        """rgb: HxWx3 uint8 (RGB order). Returns (boxes Nx4 xyxy float32, scores N)."""
        assert rgb.ndim == 3 and rgb.shape[2] == 3 and rgb.dtype == np.uint8, "expected HxWx3 uint8 RGB"
        x = torch.from_numpy(rgb).permute(2, 0, 1).float().div(255).to(self.device)
        out = self.model([x])[0]
        keep = (out["labels"] == PERSON_LABEL) & (out["scores"] >= self.score_thr)
        return out["boxes"][keep].cpu().numpy(), out["scores"][keep].cpu().numpy()

In [ ]:
import zipfile, os
from pathlib import Path
from src.utils.persistence import StageTracker

LOCAL_EXTRACT = Path("/content/cache/llvip_extracted")
llvip_zip = Path("/content/cache/LLVIP.zip")

def extract_llvip():
    if LOCAL_EXTRACT.exists():
        print("Local extraction already exists.")
        return
    LOCAL_EXTRACT.mkdir(parents=True, exist_ok=True)
    print("Extracting full LLVIP dataset locally...")
    with zipfile.ZipFile(llvip_zip, 'r') as zip_ref:
        zip_ref.extractall(LOCAL_EXTRACT)
    print("Full extraction complete!")

tracker.run("llvip_local_extract", extract_llvip)


In [ ]:
import xml.etree.ElementTree as ET
import pandas as pd
from pathlib import Path
import shutil

# Source extracted directories
ann_dir = LOCAL_EXTRACT / "LLVIP" / "Annotations"
ir_test_dir = LOCAL_EXTRACT / "LLVIP" / "infrared" / "test"
vis_test_dir = LOCAL_EXTRACT / "LLVIP" / "visible" / "test"

# Destination directories on Drive
drive_test_root = ROOT / "datasets/llvip/test_subset"
for d in ["infrared", "visible", "annotations"]:
    (drive_test_root / d).mkdir(parents=True, exist_ok=True)

def prepare_and_save_subset():
    # Get list of IR test images
    ir_images = sorted(ir_test_dir.glob("*.jpg"))
    print(f"Total test images available locally: {len(ir_images)}")

    # Pick first 500 images as subset
    subset_images = ir_images[:500]
    records = []

    for img_path in subset_images:
        stem = img_path.stem
        xml_path = ann_dir / f"{stem}.xml"
        vis_path = vis_test_dir / f"{stem}.jpg"

        if not (xml_path.exists() and vis_path.exists()):
            continue

        # Copy to Google Drive
        shutil.copyfile(img_path, drive_test_root / "infrared" / img_path.name)
        shutil.copyfile(vis_path, drive_test_root / "visible" / vis_path.name)
        shutil.copyfile(xml_path, drive_test_root / "annotations" / xml_path.name)

        # Parse XML bounding boxes for meta stats
        tree = ET.parse(xml_path)
        root = tree.getroot()
        num_persons = 0
        for obj in root.findall("object"):
            name = obj.find("name").text
            if name == "person":
                num_persons += 1

        records.append({"id": stem, "num_persons": num_persons})

    df_meta = pd.DataFrame(records)
    df_meta.to_csv(ROOT / "metadata/llvip_test_subset_meta.csv", index=False)
    print(f"Subset generation complete! {len(records)} pairs copied to Google Drive.")
    print(f"Total annotated persons in subset: {df_meta['num_persons'].sum()}")

tracker.run("llvip_subset_prep", prepare_and_save_subset)


In [ ]:
import sys, time, torch, numpy as np, importlib
(ROOT/'repo/src/detection/__init__.py').touch()
import src.detection.person_detector as pdm; importlib.reload(pdm)
from skimage import data

img = data.astronaut()                      # 512x512x3 uint8 RGB
for variant in ["mobilenet", "resnet50"]:
    det = pdm.PersonDetector(variant)
    det.detect(img)                         # warm-up
    if torch.cuda.is_available(): torch.cuda.synchronize()
    t0 = time.perf_counter()
    for _ in range(5): boxes, scores = det.detect(img)
    if torch.cuda.is_available(): torch.cuda.synchronize()
    ms = (time.perf_counter() - t0) / 5 * 1000
    print(f"{variant}: {len(boxes)} person(s), scores={np.round(scores, 3)}, boxes={np.round(boxes).astype(int).tolist()}, {ms:.0f} ms/frame on {det.device}")

In [ ]:
import xml.etree.ElementTree as ET
import cv2, torch, numpy as np, pandas as pd
import matplotlib.pyplot as plt
from pathlib import Path
import src.detection.person_detector as pdm

def calculate_iou(boxA, boxB):
    xA = max(boxA[0], boxB[0])
    yA = max(boxA[1], boxB[1])
    xB = min(boxA[2], boxB[2])
    yB = min(boxA[3], boxB[3])
    interArea = max(0, xB - xA) * max(0, yB - yA)
    boxAArea = (boxA[2] - boxA[0]) * (boxA[3] - boxA[1])
    boxBArea = (boxB[2] - boxB[0]) * (boxB[3] - boxB[1])
    iou = interArea / float(boxAArea + boxBArea - interArea) if (boxAArea + boxBArea - interArea) > 0 else 0
    return iou

def parse_xml_annotations(xml_path):
    tree = ET.parse(xml_path)
    root = tree.getroot()
    boxes = []
    for obj in root.findall("object"):
        name = obj.find("name").text
        if name == "person":
            bnd = obj.find("bndbox")
            xmin = float(bnd.find("xmin").text)
            ymin = float(bnd.find("ymin").text)
            xmax = float(bnd.find("xmax").text)
            ymax = float(bnd.find("ymax").text)
            boxes.append([xmin, ymin, xmax, ymax])
    return np.array(boxes) if boxes else np.empty((0, 4))

subset_root = ROOT / "datasets/llvip/test_subset"
subset_ir_dir = subset_root / "infrared"
subset_vis_dir = subset_root / "visible"
subset_ann_dir = subset_root / "annotations"


In [ ]:
detector = pdm.PersonDetector(variant="resnet50", score_thr=0.5)
subset_images = sorted(subset_ir_dir.glob("*.jpg"))[:20]
fig, axes = plt.subplots(5, 4, figsize=(16, 20))
axes = axes.flatten()

for idx, img_path in enumerate(subset_images):
    stem = img_path.stem
    xml_path = subset_ann_dir / f"{stem}.xml"
    ir_bgr = cv2.imread(str(img_path))
    ir_rgb = cv2.cvtColor(ir_bgr, cv2.COLOR_BGR2RGB)
    pred_boxes, pred_scores = detector.detect(ir_rgb)
    gt_boxes = parse_xml_annotations(xml_path)

    ax = axes[idx]
    ax.imshow(ir_rgb)
    for gt in gt_boxes:
        rect = plt.Rectangle((gt[0], gt[1]), gt[2]-gt[0], gt[3]-gt[1], fill=False, color='lime', linewidth=1.5)
        ax.add_patch(rect)
    for pred, score in zip(pred_boxes, pred_scores):
        rect = plt.Rectangle((pred[0], pred[1]), pred[2]-pred[0], pred[3]-pred[1], fill=False, color='red', linewidth=1.5)
        ax.add_patch(rect)
    ax.set_title(f"{stem} (GT:{len(gt_boxes)}|Pr:{len(pred_boxes)})")
    ax.axis('off')

plt.tight_layout()
plt.savefig(ROOT / "outputs/llvip_sanity_check_20.png", dpi=150)
plt.show()


In [ ]:
all_images = sorted(subset_ir_dir.glob("*.jpg"))
tp_total, fp_total, fn_total = 0, 0, 0
num_gt_total = 0

for img_path in all_images:
    stem = img_path.stem
    xml_path = subset_ann_dir / f"{stem}.xml"
    gt_boxes = parse_xml_annotations(xml_path)
    num_gt_total += len(gt_boxes)

    ir_bgr = cv2.imread(str(img_path))
    ir_rgb = cv2.cvtColor(ir_bgr, cv2.COLOR_BGR2RGB)
    pred_boxes, pred_scores = detector.detect(ir_rgb)

    matched_gt = set()
    sorted_indices = np.argsort(pred_scores)[::-1]
    for i in sorted_indices:
        p_box = pred_boxes[i]
        best_iou = 0
        best_gt_idx = -1
        for gt_idx, g_box in enumerate(gt_boxes):
            if gt_idx in matched_gt:
                continue
            iou = calculate_iou(p_box, g_box)
            if iou > best_iou:
                best_iou = iou
                best_gt_idx = gt_idx
        if best_iou >= 0.5:
            matched_gt.add(best_gt_idx)
            tp_total += 1
        else:
            fp_total += 1
    fn_total += (len(gt_boxes) - len(matched_gt))

precision = tp_total / max(tp_total + fp_total, 1)
recall = tp_total / max(num_gt_total, 1)
f1 = 2 * (precision * recall) / max(precision + recall, 1e-9)

print(f"Results on {len(all_images)} pairs:")
print(f"Ground-Truth Persons: {num_gt_total}")
print(f"Precision: {precision:.4f}")
print(f"Recall: {recall:.4f}")
print(f"F1 Score: {f1:.4f}")


In [ ]:
import xml.etree.ElementTree as ET
import cv2, torch, numpy as np, pandas as pd
import matplotlib.pyplot as plt
from pathlib import Path
import src.detection.person_detector as pdm

# Bounding box metric utils
def calculate_iou(boxA, boxB):
    xA = max(boxA[0], boxB[0])
    yA = max(boxA[1], boxB[1])
    xB = min(boxA[2], boxB[2])
    yB = min(boxA[3], boxB[3])
    interArea = max(0, xB - xA) * max(0, yB - yA)
    boxAArea = (boxA[2] - boxA[0]) * (boxA[3] - boxA[1])
    boxBArea = (boxB[2] - boxB[0]) * (boxB[3] - boxB[1])
    iou = interArea / float(boxAArea + boxBArea - interArea) if (boxAArea + boxBArea - interArea) > 0 else 0
    return iou

def parse_xml_annotations(xml_path):
    tree = ET.parse(xml_path)
    root = tree.getroot()
    boxes = []
    for obj in root.findall("object"):
        name = obj.find("name").text
        if name == "person":
            bnd = obj.find("bndbox")
            xmin = float(bnd.find("xmin").text)
            ymin = float(bnd.find("ymin").text)
            xmax = float(bnd.find("xmax").text)
            ymax = float(bnd.find("ymax").text)
            boxes.append([xmin, ymin, xmax, ymax])
    return np.array(boxes) if boxes else np.empty((0, 4))

# Paths setup
subset_root = ROOT / "datasets/llvip/test_subset"
subset_ir_dir = subset_root / "infrared"
subset_vis_dir = subset_root / "visible"
subset_ann_dir = subset_root / "annotations"


In [ ]:
# Cell 30: Run Sanity Check on first 20 images
detector = pdm.PersonDetector(variant="resnet50", score_thr=0.5)

subset_images = sorted(subset_ir_dir.glob("*.jpg"))[:20]
fig, axes = plt.subplots(5, 4, figsize=(16, 20))
axes = axes.flatten()

for idx, img_path in enumerate(subset_images):
    stem = img_path.stem
    # Visible counterpart
    vis_img_path = subset_vis_dir / f"{stem}.jpg"
    xml_path = subset_ann_dir / f"{stem}.xml"

    ir_bgr = cv2.imread(str(img_path))
    ir_rgb = cv2.cvtColor(ir_bgr, cv2.COLOR_BGR2RGB)

    # Detect on infrared image
    pred_boxes, pred_scores = detector.detect(ir_rgb)
    gt_boxes = parse_xml_annotations(xml_path)

    ax = axes[idx]
    ax.imshow(ir_rgb)

    # Plot ground truth in green
    for gt in gt_boxes:
        rect = plt.Rectangle((gt[0], gt[1]), gt[2]-gt[0], gt[3]-gt[1], fill=False, color='lime', linewidth=1.5)
        ax.add_patch(rect)

    # Plot predictions in red
    for pred, score in zip(pred_boxes, pred_scores):
        rect = plt.Rectangle((pred[0], pred[1]), pred[2]-pred[0], pred[3]-pred[1], fill=False, color='red', linewidth=1.5)
        ax.add_patch(rect)

    ax.set_title(f"{stem} (GT: {len(gt_boxes)} | Pred: {len(pred_boxes)})")
    ax.axis('off')

plt.tight_layout()
plt.savefig(ROOT / "outputs/llvip_sanity_check_20.png", dpi=150)
plt.show()


In [ ]:
# Full evaluation on all 500 pairs
all_images = sorted(subset_ir_dir.glob("*.jpg"))
tp_total, fp_total, fn_total = 0, 0, 0
all_pred_scores = []
all_pred_matches = []
num_gt_total = 0

for img_path in all_images:
    stem = img_path.stem
    xml_path = subset_ann_dir / f"{stem}.xml"
    gt_boxes = parse_xml_annotations(xml_path)
    num_gt = len(gt_boxes)
    num_gt_total += num_gt

    ir_bgr = cv2.imread(str(img_path))
    ir_rgb = cv2.cvtColor(ir_bgr, cv2.COLOR_BGR2RGB)
    pred_boxes, pred_scores = detector.detect(ir_rgb)

    # IoU Match Tracker
    matched_gt = set()
    image_preds_matched = []

    # Sort predictions by score descending for matching
    sorted_indices = np.argsort(pred_scores)[::-1]
    for i in sorted_indices:
        p_box = pred_boxes[i]
        score = pred_scores[i]
        all_pred_scores.append(score)

        best_iou = 0
        best_gt_idx = -1
        for gt_idx, g_box in enumerate(gt_boxes):
            if gt_idx in matched_gt:
                continue
            iou = calculate_iou(p_box, g_box)
            if iou > best_iou:
                best_iou = iou
                best_gt_idx = gt_idx

        if best_iou >= 0.5:
            matched_gt.add(best_gt_idx)
            all_pred_matches.append(1)  # True Positive
            tp_total += 1
        else:
            all_pred_matches.append(0)  # False Positive
            fp_total += 1

    fn_total += (num_gt - len(matched_gt))

precision = tp_total / max(tp_total + fp_total, 1)
recall = tp_total / max(num_gt_total, 1)
f1 = 2 * (precision * recall) / max(precision + recall, 1e-9)

print(f"Evaluation finished on {len(all_images)} pairs!")
print(f"Total Ground-Truth Objects: {num_gt_total}")
print(f"Precision: {precision:.4f}")
print(f"Recall: {recall:.4f}")
print(f"F1 Score: {f1:.4f}")


In [ ]:
import subprocess
from pathlib import Path

SF_DIR = ROOT/'datasets/speaking_faces'
(SF_DIR/'raw').mkdir(parents=True, exist_ok=True)
(SF_DIR/'processed').mkdir(parents=True, exist_ok=True)
print("SpeakingFaces setup directory created at:", SF_DIR)

In [ ]:
import urllib.request
import zipfile
from pathlib import Path

SF_SAMPLE_URL = "https://huggingface.co/datasets/issai/Speaking_Faces/resolve/main/image_only/sub_100_io.zip"
local_sf_zip = Path("/content/cache/SpeakingFaces_sample.zip")
local_sf_zip.parent.mkdir(parents=True, exist_ok=True)

def download_speaking_faces_sample():
    if not local_sf_zip.exists():
        print("Downloading SpeakingFaces structural sample subset...")
        try:
            # User-agent header to avoid potential bot blocks
            opener = urllib.request.build_opener()
            opener.addheaders = [('User-Agent', 'Mozilla/5.0')]
            urllib.request.install_opener(opener)
            urllib.request.urlretrieve(SF_SAMPLE_URL, local_sf_zip)
            print("Sample download completed successfully!")
        except Exception as e:
            raise RuntimeError(f"Hugging Face download failed! Error: {e}")

    extract_dest = Path("/content/cache/speaking_faces_extracted")
    if not (extract_dest/".extracted_ok").exists():
        print("Extracting sample subset...")
        with zipfile.ZipFile(local_sf_zip, 'r') as zip_ref:
            zip_ref.extractall(extract_dest)
        (extract_dest/".extracted_ok").touch()
    print("SpeakingFaces structural sample ready in local cache!")

tracker.run("sf_sample_download", download_speaking_faces_sample)

In [ ]:
import cv2
import matplotlib.pyplot as plt
from pathlib import Path

extract_dest = Path("/content/cache/speaking_faces_extracted")

# Pura file system search karein nested structure dhoondhne ke liye
all_files = [p for p in extract_dest.rglob("*") if p.is_file()]
print(f"Total files found recursively: {len(all_files)}")

# First 15 files print karein structural hierarchy samajhne ke liye
print("\nSample file paths structure:")
for f in all_files[:15]:
    print(f.relative_to(extract_dest), f"({f.stat().st_size / 1024:.1f} KB)")

In [ ]:
import cv2
import matplotlib.pyplot as plt
from pathlib import Path

# Source directories from extracted cache
extract_dest = Path("/content/cache/speaking_faces_extracted")
sample_dir = extract_dest / "sub_100_io" / "trial_1"

# Let's locate an aligned image from the list
aligned_dir = sample_dir / "rgb_image_aligned"
all_pngs = sorted(aligned_dir.glob("*.png"))

if all_pngs:
    sample_img_path = all_pngs[0]
    print(f"Displaying sample image: {sample_img_path.name}")

    # Read and convert color from BGR to RGB for matplotlib
    img_bgr = cv2.imread(str(sample_img_path))
    img_rgb = cv2.cvtColor(img_bgr, cv2.COLOR_BGR2RGB)

    # Display the image
    plt.figure(figsize=(6, 6))
    plt.imshow(img_rgb)
    plt.title(f"Speaking_Faces Aligned RGB Sample: {sample_img_path.name}")
    plt.axis("off")
    plt.show()
else:
    print("No PNG images found in the aligned path!")

In [ ]:
%%writefile /content/drive/MyDrive/Intelligent_Temperature_Monitor/repo/src/alignment/face_aligner.py
"""Face alignment module using 2D affine transformation on facial landmarks."""
import cv2
import numpy as np

class FaceAligner:
    def __init__(self, target_size=(256, 256), eye_target_ratio=0.35):
        self.target_size = target_size
        self.eye_target_ratio = eye_target_ratio

    def align_face_by_eyes(self, img, left_eye_center, right_eye_center):
        """Performs 2D affine transformation to align and crop face based on eye centers."""
        dy = right_eye_center[1] - left_eye_center[1]
        dx = right_eye_center[0] - left_eye_center[0]
        angle = np.degrees(np.arctan2(dy, dx))

        # Desired coordinates of eyes in output crop
        desired_left_x = self.target_size[0] * self.eye_target_ratio
        desired_right_x = self.target_size[0] * (1.0 - self.eye_target_ratio)
        desired_y = self.target_size[1] * self.eye_target_ratio

        dist = np.sqrt(dx**2 + dy**2)
        desired_dist = desired_right_x - desired_left_x
        scale = desired_dist / max(dist, 1e-6)

        # Center between eyes
        eyes_center = ((left_eye_center[0] + right_eye_center[0]) / 2.0,
                       (left_eye_center[1] + right_eye_center[1]) / 2.0)

        # Get Rotation Matrix
        M = cv2.getRotationMatrix2D(eyes_center, angle, scale)

        # Translate eyes center to target center
        t_x = self.target_size[0] * 0.5
        t_y = desired_y
        M[0, 2] += (t_x - eyes_center[0])
        M[1, 2] += (t_y - eyes_center[1])

        # Apply affine transformation
        aligned = cv2.warpAffine(img, M, self.target_size, flags=cv2.INTER_LINEAR)
        return aligned, M


In [ ]:
%%writefile /content/drive/MyDrive/Intelligent_Temperature_Monitor/repo/tests/test_alignment.py
import numpy as np
from src.alignment.face_aligner import FaceAligner

def test_dummy_alignment():
    # Generate dummy image with a white square mimicking eyes
    img = np.zeros((100, 100, 3), dtype=np.uint8)
    aligner = FaceAligner(target_size=(64, 64))

    left_eye = (30.0, 40.0)
    right_eye = (70.0, 40.0)

    aligned, M = aligner.align_face_by_eyes(img, left_eye, right_eye)
    assert aligned.shape == (64, 64, 3)
    assert M.shape == (2, 3)


In [ ]:
# Run Pytest to verify alignment implementation
!cd /content/drive/MyDrive/Intelligent_Temperature_Monitor/repo && python -m pytest tests/test_alignment.py -q

In [ ]:
import subprocess
from google.colab import userdata
token = userdata.get('GITHUB_TOKEN')
cwd = f"{ROOT}/repo"

def sh(*a):
    return subprocess.run(a, cwd=cwd, capture_output=True, text=True)

sh("git", "add", "src/alignment/face_aligner.py", "tests/test_alignment.py")
print(sh("git", "commit", "-m", "feat: add FaceAligner class and corresponding unit test").stdout)
r = sh("git", "push", f"https://{token}@github.com/NidaKhaan/intelligent-body-temperature-monitoring.git", "main")
print("Push exit code:", r.returncode)
print((r.stdout + r.stderr).replace(token, "***"))

In [ ]:
import subprocess
from google.colab import userdata
token = userdata.get('GITHUB_TOKEN')
cwd = f"{ROOT}/repo"

def sh(*a):
    return subprocess.run(a, cwd=cwd, capture_output=True, text=True)

sh("git", "add", "src/alignment", "tests/test_alignment.py")
print(sh("git", "commit", "-m", "feat: add FaceAligner class and corresponding unit test").stdout)
r = sh("git", "push", f"https://{token}@github.com/NidaKhaan/intelligent-body-temperature-monitoring.git", "main")
print("Push exit code:", r.returncode)
print((r.stdout + r.stderr).replace(token, "***"))

In [ ]:
import subprocess
from google.colab import userdata
token = userdata.get('GITHUB_TOKEN')
cwd = f"{ROOT}/repo"

def sh(*a):
    return subprocess.run(a, cwd=cwd, capture_output=True, text=True)

sh("git", "add", "src/detection")
print(sh("git", "commit", "-m", "feat: add person detector class and sub-components").stdout)
r = sh("git", "push", f"https://{token}@github.com/NidaKhaan/intelligent-body-temperature-monitoring.git", "main")
print("Push exit code:", r.returncode)
print((r.stdout + r.stderr).replace(token, "***"))

In [ ]:
import cv2
import numpy as np
import matplotlib.pyplot as plt
from pathlib import Path
from src.alignment.face_aligner import FaceAligner

# Path of SpeakingFaces samples
extract_dest = Path('/content/cache/speaking_faces_extracted')
aligned_dir = extract_dest / 'sub_100_io' / 'trial_1' / 'rgb_image_aligned'
all_pngs = sorted(aligned_dir.glob('*.png'))

if all_pngs:
    # Sample Image Load karein
    img_path = all_pngs[0]
    img_bgr = cv2.imread(str(img_path))
    img_rgb = cv2.cvtColor(img_bgr, cv2.COLOR_BGR2RGB)

    # Simulate eye centers based on the image size
    # (In production, these landmarks are predicted by MediaPipe or customized landmark detectors)
    h, w, _ = img_rgb.shape
    left_eye = (w * 0.42, h * 0.43)    # Approximate left eye coordinate
    right_eye = (w * 0.58, h * 0.43)   # Approximate right eye coordinate

    # Aligner initialize aur run karein
    aligner = FaceAligner(target_size=(256, 256), eye_target_ratio=0.35)
    aligned_face, M = aligner.align_face_by_eyes(img_rgb, left_eye, right_eye)

    # Visualizations
    fig, axes = plt.subplots(1, 2, figsize=(12, 6))

    # Plot Original Image with Landmarks
    axes[0].imshow(img_rgb)
    axes[0].scatter([left_eye[0], right_eye[0]], [left_eye[1], right_eye[1]], color='red', s=40, label='Eye Centers')
    axes[0].set_title("Original Image (with simulated eyes)")
    axes[0].axis('off')
    axes[0].legend()

    # Plot Aligned Face Crop
    axes[1].imshow(aligned_face)
    axes[1].set_title("Aligned & Cropped Face (256x256)")
    axes[1].axis('off')

    plt.tight_layout()
    plt.show()
else:
    print("No SpeakingFaces samples found to visualize!")

In [ ]:
import cv2
import numpy as np
import matplotlib.pyplot as plt
from pathlib import Path
from src.alignment.face_aligner import FaceAligner
from src.temperature.radiometric import raw_to_celsius

# SpeakingFaces paths setup
extract_dest = Path('/content/cache/speaking_faces_extracted')
sample_dir = extract_dest / 'sub_100_io' / 'trial_1'

# RGB and Thermal matching file structures
rgb_dir = sample_dir / 'rgb_image_aligned'
# Let's locate the raw thermal file path (which is typically in 'thermal_image' or we map the name)
thermal_dir = sample_dir / 'thermal_image'

all_rgbs = sorted(rgb_dir.glob('*.png'))

if all_rgbs:
    # Sample matching frames
    rgb_path = all_rgbs[0]
    # Corresponding thermal file shares the base ID but ends with .tiff
    thermal_path = thermal_dir / (rgb_path.stem.replace('_3', '_1') + '.tiff') # standard SpeakingFaces ID mapping

    if not thermal_path.exists():
        # Fallback search in case of nested structure
        all_tiffs = list(sample_dir.rglob('*.tiff'))
        if all_tiffs:
            thermal_path = all_tiffs[0]

    print(f"Loading RGB: {rgb_path.name}")
    print(f"Loading Thermal: {thermal_path.name if thermal_path.exists() else 'Not found'}")

    # Load RGB
    img_rgb = cv2.cvtColor(cv2.imread(str(rgb_path)), cv2.COLOR_BGR2RGB)
    h, w, _ = img_rgb.shape

    # Setup Simulated eye centers on RGB
    left_eye = (w * 0.42, h * 0.43)
    right_eye = (w * 0.58, h * 0.43)

    # Initialize Aligner and align RGB
    aligner = FaceAligner(target_size=(256, 256), eye_target_ratio=0.35)
    aligned_rgb, M = aligner.align_face_by_eyes(img_rgb, left_eye, right_eye)

    # If Thermal image exists, load, convert to Celsius and apply same Transformation Matrix
    if thermal_path.exists():
        img_thermal_raw = cv2.imread(str(thermal_path), cv2.IMREAD_UNCHANGED)
        # Convert raw 16-bit counts to Celsius
        img_thermal_c = raw_to_celsius(img_thermal_raw)

        # Apply the exact same affine matrix 'M' to the Thermal Image
        aligned_thermal = cv2.warpAffine(img_thermal_c, M, (256, 256), flags=cv2.INTER_LINEAR)

        # Visualizing both side-by-side
        fig, axes = plt.subplots(2, 2, figsize=(12, 10))

        # Row 1: RGB Original & Aligned
        axes[0, 0].imshow(img_rgb)
        axes[0, 0].scatter([left_eye[0], right_eye[0]], [left_eye[1], right_eye[1]], color='red', s=40)
        axes[0, 0].set_title("Original RGB Image (with simulated eyes)")
        axes[0, 0].axis('off')

        axes[0, 1].imshow(aligned_rgb)
        axes[0, 1].set_title("Aligned RGB Face (256x256)")
        axes[0, 1].axis('off')

        # Row 2: Thermal Original & Aligned (using 'inferno' colormap for thermal output)
        im_therm_orig = axes[1, 0].imshow(img_thermal_c, cmap='inferno')
        fig.colorbar(im_therm_orig, ax=axes[1, 0], label='Temperature (C)')
        axes[1, 0].set_title("Original Thermal Temperature Map (C)")
        axes[1, 0].axis('off')

        im_therm_aligned = axes[1, 1].imshow(aligned_thermal, cmap='inferno')
        fig.colorbar(im_therm_aligned, ax=axes[1, 1], label='Temperature (C)')
        axes[1, 1].set_title("Aligned Thermal Face (256x256)")
        axes[1, 1].axis('off')

        plt.tight_layout()
        plt.show()
    else:
        print("Thermal TIFF matching sample nahi mila!")
else:
    print("No SpeakingFaces RGB samples found to visualize!")

In [ ]:
import cv2, numpy as np, collections
from pathlib import Path
SFX = Path('/content/cache/speaking_faces_extracted')
print("exists:", SFX.exists())
if SFX.exists():
    files = [p for p in SFX.rglob("*") if p.is_file() and not p.name.startswith(".")]
    print("files:", len(files))
    print(collections.Counter((p.parent.name, p.suffix.lower()) for p in files).most_common(10))
    subs = sorted({p.relative_to(SFX).parts[0] for p in files}); print("subject dirs:", len(subs), subs[:5])
    for m in sorted({p.parent for p in files if p.relative_to(SFX).parts[0] == subs[0]}):
        fs = sorted(q for q in m.iterdir() if q.is_file())
        im = cv2.imread(str(fs[0]), cv2.IMREAD_UNCHANGED)
        info = None if im is None else (im.shape, str(im.dtype), int(im.min()), int(im.max()))
        print(m.relative_to(SFX), "| n:", len(fs), "| ex:", fs[0].name, "|", info)

In [ ]:
import numpy as np, cv2, shutil, time, xml.etree.ElementTree as ET, pandas as pd
from pathlib import Path
import src.detection.person_detector as pdm
from src.utils.persistence import atomic_write_json

LOCAL = Path("/content/cache/llvip_extracted/LLVIP")
V1, V2 = ROOT/"datasets/llvip/test_subset", ROOT/"datasets/llvip/test_subset_v2"

def make_v2():
    ir, vis, ann = LOCAL/"infrared/test", LOCAL/"visible/test", LOCAL/"Annotations"
    ids = sorted(p.stem for p in ir.glob("*.jpg"))[::7][:500]
    for d in ["infrared", "visible", "annotations"]: (V2/d).mkdir(parents=True, exist_ok=True)
    k = 0
    for i in ids:
        if (vis/f"{i}.jpg").exists() and (ann/f"{i}.xml").exists():
            shutil.copyfile(ir/f"{i}.jpg", V2/"infrared"/f"{i}.jpg")
            shutil.copyfile(vis/f"{i}.jpg", V2/"visible"/f"{i}.jpg")
            shutil.copyfile(ann/f"{i}.xml", V2/"annotations"/f"{i}.xml"); k += 1
    print("v2 pairs:", k)

if LOCAL.exists():
    tracker.run("llvip_subset_v2", make_v2, verify=lambda: (V2/"annotations").exists()); SUB = V2
else:
    print("LOCAL LLVIP cache nahi: purana first-500 subset (consecutive frames) use ho raha hai: yeh limitation hai")
    SUB = V1

def load_gt(xml):
    r = ET.parse(xml).getroot()
    return np.array([[float(o.find("bndbox").find(k).text) for k in ("xmin", "ymin", "xmax", "ymax")]
                     for o in r.findall("object") if o.find("name").text == "person"], float).reshape(-1, 4)

def iou_matrix(a, b):
    if len(a) == 0 or len(b) == 0: return np.zeros((len(a), len(b)))
    x0 = np.maximum(a[:, None, 0], b[None, :, 0]); y0 = np.maximum(a[:, None, 1], b[None, :, 1])
    x1 = np.minimum(a[:, None, 2], b[None, :, 2]); y1 = np.minimum(a[:, None, 3], b[None, :, 3])
    inter = np.clip(x1 - x0, 0, None) * np.clip(y1 - y0, 0, None)
    area = lambda z: (z[:, 2] - z[:, 0]) * (z[:, 3] - z[:, 1])
    return inter / (area(a)[:, None] + area(b)[None, :] - inter + 1e-9)

def score(dets, gts, thr=0.5):
    recs, n_gt = [], sum(len(g) for g in gts)
    for (b, s), g in zip(dets, gts):
        o = np.argsort(-s); b, s = b[o], s[o]; ious = iou_matrix(b, g); used = set()
        for k in range(len(b)):
            cand = [(ious[k, m], m) for m in range(len(g)) if m not in used and ious[k, m] >= 0.5]
            j = max(cand)[1] if cand else -1
            if j >= 0: used.add(j)
            recs.append((float(s[k]), j >= 0))
    recs.sort(key=lambda r: -r[0])
    sc = np.array([r[0] for r in recs]); tpf = np.array([r[1] for r in recs], float)
    tp, fp = np.cumsum(tpf), np.cumsum(1 - tpf)
    rec, prec = tp / max(n_gt, 1), tp / np.maximum(tp + fp, 1)
    mrec, mpre = np.r_[0, rec, 1], np.r_[0, prec, 0]
    for k in range(len(mpre) - 2, -1, -1): mpre[k] = max(mpre[k], mpre[k + 1])
    idx = np.where(mrec[1:] != mrec[:-1])[0]
    sel = sc >= thr
    return dict(AP50=float(np.sum((mrec[idx + 1] - mrec[idx]) * mpre[idx + 1])),
                P=float(tpf[sel].sum() / max(sel.sum(), 1)), R=float(tpf[sel].sum() / max(n_gt, 1)), n_gt=int(n_gt))

ids = sorted(p.stem for p in (SUB/"infrared").glob("*.jpg"))
gts = [load_gt(SUB/"annotations"/f"{i}.xml") for i in ids]

def llvip_eval():
    res = {"subset": SUB.name, "n_pairs": len(ids)}
    for variant in ["mobilenet", "resnet50"]:
        det = pdm.PersonDetector(variant, score_thr=0.05)
        for mod in ["visible", "infrared"]:
            dets, ts = [], []
            for i in ids:
                rgb = cv2.cvtColor(cv2.imread(str(SUB/mod/f"{i}.jpg")), cv2.COLOR_BGR2RGB)
                t0 = time.perf_counter(); b, s = det.detect(rgb); ts.append(time.perf_counter() - t0); dets.append((b, s))
            res[f"{variant}/{mod}"] = {**score(dets, gts), "median_ms": float(np.median(ts) * 1000)}
    atomic_write_json(ROOT/f"experiments/llvip_eval_v2_{SUB.name}.json", res)

tracker.run(f"llvip_eval_v2_{SUB.name}", llvip_eval)
import json
r = json.load(open(ROOT/f"experiments/llvip_eval_v2_{SUB.name}.json"))
print("subset:", r["subset"], "| pairs:", r["n_pairs"])
print(pd.DataFrame({k: v for k, v in r.items() if "/" in k}).T.round(3).to_string())

In [ ]:
import importlib
p = ROOT/'repo/src/utils/persistence.py'
s = p.read_text()
old = "def run(self, stage, fn, *args, **kwargs):\n        if self.is_done(stage):"
new = "def run(self, stage, fn, *args, verify=None, **kwargs):\n        if self.is_done(stage) and (verify is None or verify()):"
assert old in s, "persistence.py ka format alag hai"
p.write_text(s.replace(old, new))
import src.utils.persistence as P; importlib.reload(P)
tracker = P.StageTracker(ROOT/'metadata/progress.json')
# istemal: tracker.run("sf_sample_download", fn, verify=lambda: SFX.exists())

In [ ]:
from src.utils.persistence import atomic_write_json
for k in [k for k in tracker.state if k.startswith("sf_")]:
    tracker.state.pop(k)
atomic_write_json(tracker.path, tracker.state)
print("remaining stages:", list(tracker.state))

In [ ]:
from src.utils.persistence import atomic_write_json
tracker.state.pop("llvip_local_extract", None)
atomic_write_json(tracker.path, tracker.state)

In [ ]:
import cv2, numpy as np, matplotlib.pyplot as plt
from pathlib import Path

SFX = Path('/content/cache/speaking_faces_extracted/sub_100_io')
def stem_of(p): return p.stem.rsplit("_", 1)[0]        # last digit = modality code (hypothesis)
def index(trial, folder): return {stem_of(p): p for p in (SFX/trial/folder).glob("*.png")}

thr, raw, ali = (index("trial_1", f) for f in ["thr_image", "rgb_image", "rgb_image_aligned"])
common = sorted(set(thr) & set(raw) & set(ali))
print("frames in all 3:", len(common), "| thr/raw/ali:", len(thr), len(raw), len(ali))

t = cv2.imread(str(thr[common[0]]), cv2.IMREAD_UNCHANGED)
gray = bool((t[..., 0] == t[..., 1]).all() and (t[..., 1] == t[..., 2]).all())
print("thermal: 3 channels identical (grayscale):", gray, "| distinct gray levels:", len(np.unique(t[..., 0])))

sift, bf = cv2.SIFT_create(), cv2.BFMatcher()
def est_H(raw_img, ali_img):
    g1, g2 = (cv2.cvtColor(i, cv2.COLOR_BGR2GRAY) for i in (raw_img, ali_img))
    k1, d1 = sift.detectAndCompute(g1, None); k2, d2 = sift.detectAndCompute(g2, None)
    if d1 is None or d2 is None: return None, 0
    m = [p[0] for p in bf.knnMatch(d1, d2, k=2) if len(p) == 2 and p[0].distance < 0.75 * p[1].distance]
    if len(m) < 12: return None, len(m)
    src = np.float32([k1[x.queryIdx].pt for x in m]); dst = np.float32([k2[x.trainIdx].pt for x in m])
    H, inl = cv2.findHomography(src, dst, cv2.RANSAC, 3.0)
    return H, (int(inl.sum()) if inl is not None else 0)


grid = np.float32([[x, y] for x in (60, 232, 404) for y in (50, 174, 298)]).reshape(-1, 1, 2)
mapped, inliers, first = [], [], None
for s in common[::max(len(common) // 40, 1)][:40]:
    r_i, a_i = cv2.imread(str(raw[s])), cv2.imread(str(ali[s]))
    H, n = est_H(r_i, a_i)
    if H is None: continue
    mapped.append(cv2.perspectiveTransform(grid, np.linalg.inv(H)).reshape(-1, 2)); inliers.append(n)
    first = first or (s, H)
mapped = np.array(mapped)
print(f"\nframes with a valid H: {len(mapped)} | RANSAC inliers median: {np.median(inliers):.0f}")
sd = mapped.std(0)
print(f"std of mapped points across frames (px): median {np.median(sd):.2f}, max {sd.max():.2f}")

s, H = first
warped = cv2.warpPerspective(cv2.imread(str(raw[s])), H, (464, 348))
th = cv2.imread(str(thr[s]))
blend = cv2.addWeighted(cv2.cvtColor(warped, cv2.COLOR_BGR2GRAY), 0.5, cv2.cvtColor(th, cv2.COLOR_BGR2GRAY), 0.5, 0)
fig, ax = plt.subplots(1, 4, figsize=(16, 4))
for a, im, ttl in zip(ax, [th, cv2.imread(str(ali[s])), warped, blend],
                      ["thermal (thr_image)", "dataset's aligned RGB", "raw RGB warped by our H", "50% blend: warped RGB + thermal"]):
    a.imshow(cv2.cvtColor(im, cv2.COLOR_BGR2RGB) if im.ndim == 3 else im, cmap="gray"); a.set_title(ttl, fontsize=9); a.axis("off")
plt.tight_layout(); plt.show()

In [ ]:
import collections
def pos_of(s): return s.split("_")[3]            # hypothesis: camera position 1-9
def frame_of(s): return int(s.split("_")[4])

def mapped_pts(s):
    H, n = est_H(cv2.imread(str(raw[s])), cv2.imread(str(ali[s])))
    return None if H is None else cv2.perspectiveTransform(grid, np.linalg.inv(H)).reshape(-1, 2)

by = collections.defaultdict(list)
for s in common[::max(len(common) // 120, 1)][:120]:
    m = mapped_pts(s)
    if m is not None: by[pos_of(s)].append(m)
print("frames per position field:", {p: len(v) for p, v in sorted(by.items())})
for p, v in sorted(by.items()):
    if len(v) >= 3:
        sd = np.array(v).std(0); print(f"position {p}: n={len(v)} | std px median {np.median(sd):.2f}, max {sd.max():.2f}")

p0 = max(by, key=lambda p: len(by[p]))
cons = sorted([s for s in common if pos_of(s) == p0], key=frame_of)[:15]
ms = [m for m in (mapped_pts(s) for s in cons) if m is not None]
sd = np.array(ms).std(0)
print(f"consecutive frames, position {p0}: n={len(ms)} | std px median {np.median(sd):.2f}, max {sd.max():.2f}")

In [ ]:
!pip install -q mediapipe
import urllib.request
from pathlib import Path
import cv2
import numpy as np
import time
import mediapipe as mp
from mediapipe.tasks import python
from mediapipe.tasks.python import vision

# Model weights file download for Tasks API
model_path = Path("/content/cache/detector.tflite")
if not model_path.exists():
    model_path.parent.mkdir(parents=True, exist_ok=True)
    url = "https://storage.googleapis.com/mediapipe-models/face_detector/blaze_face_short_range/float16/1/blaze_face_short_range.tflite"
    urllib.request.urlretrieve(url, model_path)

# Initialize MediaPipe Tasks Face Detector
base_options = python.BaseOptions(model_asset_path=str(model_path))
options = vision.FaceDetectorOptions(base_options=base_options, min_detection_confidence=0.3)
detector = vision.FaceDetector.create_from_options(options)

def detect_face(rgb):
    h, w = rgb.shape[:2]
    mp_image = mp.Image(image_format=mp.ImageFormat.SRGB, data=rgb)
    res = detector.detect(mp_image)
    out = []
    for detection in res.detections:
        box = detection.bounding_box
        score = detection.categories[0].score
        xmin = max(0, box.origin_x)
        ymin = max(0, box.origin_y)
        xmax = min(w, box.origin_x + box.width)
        ymax = min(h, box.origin_y + box.height)
        out.append((xmin, ymin, xmax, ymax, float(score)))
    return sorted(out, key=lambda t: -t[4])

def iou(a, b):
    x0, y0, x1, y1 = max(a[0], b[0]), max(a[1], b[1]), min(a[2], b[2]), min(a[3], b[3])
    i = max(0, x1 - x0) * max(0, y1 - y0)
    return i / ((a[2]-a[0])*(a[3]-a[1]) + (b[2]-b[0])*(b[3]-b[1]) - i + 1e-9)

sample = common[::81][:100]
stats = {"rgb": 0, "thermal_color": 0, "thermal_gray": 0}
iou_c, iou_g, ts = [], [], []

for s in sample:
    rgb = cv2.cvtColor(cv2.imread(str(ali[s])), cv2.COLOR_BGR2RGB)
    thc = cv2.cvtColor(cv2.imread(str(thr[s])), cv2.COLOR_BGR2RGB)
    thg = cv2.cvtColor(cv2.cvtColor(thc, cv2.COLOR_RGB2GRAY), cv2.COLOR_GRAY2RGB)

    t0 = time.perf_counter()
    r = detect_face(rgb)
    ts.append(time.perf_counter() - t0)

    c, g = detect_face(thc), detect_face(thg)
    stats["rgb"] += bool(r)
    stats["thermal_color"] += bool(c)
    stats["thermal_gray"] += bool(g)

    if r and c:
        iou_c.append(iou(r[0][:4], c[0][:4]))
    if r and g:
        iou_g.append(iou(r[0][:4], g[0][:4]))

n = len(sample)
print({k: f"{v}/{n}" for k, v in stats.items()}, f"| {np.median(ts)*1000:.1f} ms/frame (RGB, Colab CPU)")
for name, v in [("colorized thermal", iou_c), ("grayscale thermal", iou_g)]:
    if v:
        print(f"{name}: n={len(v)} | IoU with RGB box: median {np.median(v):.2f}, >=0.5: {np.mean(np.array(v) >= 0.5):.0%}")

In [ ]:
import shutil
DST = ROOT/"datasets/speaking_faces/processed/sample_trial1_every81"
def save_subset():
    for s in common[::81]:
        for folder, d in [("thr_image", thr), ("rgb_image", raw), ("rgb_image_aligned", ali)]:
            (DST/folder).mkdir(parents=True, exist_ok=True)
            shutil.copyfile(d[s], DST/folder/d[s].name)
    print("saved frames:", len(list((DST/"thr_image").glob("*.png"))))
tracker.run("sf_subset_to_drive_v1", save_subset)

In [ ]:
import cv2, zipfile, numpy as np, pandas as pd, mediapipe as mp
from pathlib import Path
import src.temperature.radiometric as rd
from src.utils.persistence import atomic_write_json
from mediapipe.tasks import python
from mediapipe.tasks.python import vision

CH = ROOT/'datasets/charlotte_thermal'
OUT = Path("/content/cache/charlotte")
if not (OUT/".extracted_ok").exists():
    zipfile.ZipFile(CH/"Charlotte-ThermalFace.zip").extractall(OUT); (OUT/".extracted_ok").touch()
xc, yc = [f"x{i}" for i in range(73)], [f"y{i}" for i in range(73)]
SUBJ, FRACS = [1, 4, 5, 9], [0.025, 0.05, 0.10, 0.20]
T_LO, T_HI = 20.0, 38.0            # fixed 8-bit window (assumption): thermal -> grayscale for MediaPipe

kept = pd.read_csv(ROOT/"experiments/charlotte_scan_v2.csv")
kept = kept[kept.subject.isin(SUBJ)].copy(); kept["d"] = kept.distance.round(1)
samp = kept.sample(frac=1, random_state=0).groupby(["subject", "d"]).head(15)
ann = {s: pd.read_csv(CH/f"annotations/S{s}.csv").drop_duplicates("ID").set_index("ID") for s in SUBJ}
tiffs = {s: {p.stem: p for p in next(q for q in OUT.rglob(f"S{s}") if q.is_dir()).glob("*.tiff")} for s in SUBJ}

# Initialize modernized MediaPipe Tasks Face Detector
model_path = Path("/content/cache/detector.tflite")
base_options = python.BaseOptions(model_asset_path=str(model_path))
options = vision.FaceDetectorOptions(base_options=base_options, min_detection_confidence=0.3)
fd = vision.FaceDetector.create_from_options(options)

def iou(a, b):
    x0, y0, x1, y1 = max(a[0], b[0]), max(a[1], b[1]), min(a[2], b[2]), min(a[3], b[3])
    i = max(0, x1 - x0) * max(0, y1 - y0)
    return i / ((a[2]-a[0])*(a[3]-a[1]) + (b[2]-b[0])*(b[3]-b[1]) - i + 1e-9)

def face_box(g8):
    h, w = g8.shape
    # Use 3-channel grayscale for MediaPipe Tasks API
    rgb = np.dstack([g8] * 3)
    mp_image = mp.Image(image_format=mp.ImageFormat.SRGB, data=rgb)
    res = fd.detect(mp_image)
    if not res.detections: return None

    # Select detection with highest score
    best = max(res.detections, key=lambda d: d.categories[0].score)
    b = best.bounding_box
    x0, y0 = max(0, b.origin_x), max(0, b.origin_y)
    x1, y1 = min(w, b.origin_x + b.width), min(h, b.origin_y + b.height)
    return (x0, y0, x1, y1) if (x1 - x0 >= 3 and y1 - y0 >= 3) else None

def one(r):
    row = ann[r.subject].loc[r.id]
    img = cv2.imread(str(tiffs[r.subject][r.id]), cv2.IMREAD_UNCHANGED); h, w = img.shape
    c = rd.raw_to_celsius(img)
    xs, ys = row[xc].to_numpy(float), row[yc].to_numpy(float); ok = np.isfinite(xs) & np.isfinite(ys)
    pts = np.stack([np.clip(xs[ok], 0, w - 1), np.clip(ys[ok], 0, h - 1)], 1).astype(np.int32)
    mask = np.zeros((h, w), np.uint8); cv2.fillConvexPoly(mask, cv2.convexHull(pts), 1)
    ref = c[mask.astype(bool)].max()
    rec = dict(subject=r.subject, d=r.d, box_h=r.box_h, ref_max=ref)
    for f in FRACS:
        mag, dl = f * r.box_h, []
        for dx, dy in [(mag, 0), (-mag, 0), (0, mag), (0, -mag)]:
            m2 = cv2.warpAffine(mask, np.float32([[1, 0, dx], [0, 1, dy]]), (w, h), flags=cv2.INTER_NEAREST).astype(bool)
            dl.append(abs(c[m2].max() - ref) if m2.any() else np.nan)
        rec[f"shift_{f}"] = float(np.nanmean(dl))
    box = face_box(np.clip((c - T_LO) / (T_HI - T_LO) * 255, 0, 255).astype(np.uint8))
    rec["detected"] = box is not None
    if box:
        rec["box_minus_ref"] = float(c[box[1]:box[3], box[0]:box[2]].max() - ref)
        rec["iou"] = iou(box, (*pts.min(0), *pts.max(0)))
    return rec

def roi_experiment():
    pd.DataFrame([one(r) for r in samp.itertuples()]).to_csv(ROOT/"experiments/charlotte_roi_sensitivity_v1.csv", index=False)

tracker.run("charlotte_roi_v1", roi_experiment)
res = pd.read_csv(ROOT/"experiments/charlotte_roi_sensitivity_v1.csv")
res["box_abs"] = res.box_minus_ref.abs()
agg = {"n": ("ref_max", "size"), "box_h": ("box_h", "mean"), **{f"shift{f}": (f"shift_{f}", "mean") for f in FRACS},
       "detected": ("detected", "mean"), "iou": ("iou", "median"), "auto-ref": ("box_minus_ref", "mean"), "|auto-ref|": ("box_abs", "mean")}
print(res.groupby("d").agg(**agg).round(2).to_string())
print("\noverall: images", len(res), "| subjects", sorted(res.subject.unique()),
      "| |shift effect| >0.1C at 10%:", f"{(res['shift_0.1'] > 0.1).mean():.0%}")

In [ ]:
import time, numpy as np
import src.detection.person_detector as pdm
from skimage import data
img = data.astronaut()
for variant in ["mobilenet", "resnet50"]:
    det = pdm.PersonDetector(variant, device="cpu"); det.detect(img)
    t0 = time.perf_counter()
    for _ in range(3): det.detect(img)
    print(variant, f"{(time.perf_counter() - t0) / 3 * 1000:.0f} ms/frame on Colab CPU")

In [ ]:
import pandas as pd, shutil, subprocess
from pathlib import Path
from google.colab import userdata

# Define the missing path variables
ROOT = Path('/content/drive/MyDrive/Intelligent_Temperature_Monitor')
FDA = ROOT/'datasets/fda_physionet'

fl = pd.read_csv(FDA/"clean_FLIR_groups1and2.csv")
ok = fl["T_Max1"].notna() & fl["aveOralM"].notna()
print(f"FDA (FLIR, ~0.8 m): T_Max round1 mean {fl.loc[ok,'T_Max1'].mean():.2f} C | oral mean {fl.loc[ok,'aveOralM'].mean():.2f} C "
      f"| T_atm mean {fl['T_atm'].mean():.1f} C")
sc = pd.read_csv(ROOT/"experiments/charlotte_scan_v2.csv"); sc["d"] = sc.distance.round(1)
for d in (1.0, 1.4):
    s = sc[sc.d == d]
    print(f"Charlotte {d} m: face max mean {s.max_hull.mean():.2f} C | env {s.env_temp.mean():.1f} C | by subject:",
          s.groupby("subject").max_hull.mean().round(2).to_dict())

res = ROOT/"repo/docs/results"
shutil.copy(ROOT/"experiments/llvip_eval_v2_test_subset.json", res)
shutil.copy(ROOT/"experiments/charlotte_roi_sensitivity_v1.csv", res)
token = userdata.get('GITHUB_TOKEN'); cwd = f"{ROOT}/repo"
def sh(*a): return subprocess.run(a, cwd=cwd, capture_output=True, text=True)
sh("git", "add", "docs")
print(sh("git", "commit", "-m", "docs: add LLVIP detector evaluation and Charlotte ROI sensitivity results").stdout)
r = sh("git", "push", f"https://{token}@github.com/NidaKhaan/intelligent-body-temperature-monitoring.git", "main")
print(r.returncode, (r.stdout + r.stderr).replace(token, "***"))

In [ ]:
import json, numpy as np, pandas as pd
from sklearn.linear_model import LinearRegression

fl = pd.read_csv(FDA/"clean_FLIR_groups1and2.csv")
x = pd.concat([fl[f"T_Max{k}"] for k in range(1, 5)], ignore_index=True)
y = pd.concat([fl["aveOralM"]] * 4, ignore_index=True)
ok = x.notna() & y.notna()
lr = LinearRegression().fit(x[ok].to_frame(), y[ok])
cfg = dict(name="fda_flir_tmax_single_frame",
           source="FDA/PhysioNet FLIR groups1and2: single-round T_Max vs aveOralM, 4 rounds pooled",
           slope=float(lr.coef_[0]), intercept=float(lr.intercept_),
           residual_std_c=float((y[ok] - lr.predict(x[ok].to_frame())).std()), n=int(ok.sum()),
           validated_for="FDA camera and FDA ROI definition only; not transferable")
(ROOT/"repo/configs").mkdir(exist_ok=True)
json.dump(cfg, open(ROOT/"repo/configs/calibration_fda_flir_tmax.json", "w"), indent=2)
print(cfg)

In [ ]:
%%writefile /content/drive/MyDrive/Intelligent_Temperature_Monitor/repo/src/detection/face_detector.py
import numpy as np
from pathlib import Path
import urllib.request
import mediapipe as mp
from mediapipe.tasks import python
from mediapipe.tasks.python import vision

def mediapipe_face_detector(min_conf=0.3):
    """Returns detect(gray8 HxW uint8) -> list of (x0, y0, x1, y1) in pixels using the modern Tasks API."""
    model_path = Path("/content/cache/detector.tflite")
    if not model_path.exists():
        model_path.parent.mkdir(parents=True, exist_ok=True)
        url = "https://storage.googleapis.com/mediapipe-models/face_detector/blaze_face_short_range/float16/1/blaze_face_short_range.tflite"
        urllib.request.urlretrieve(url, model_path)

    base_options = python.BaseOptions(model_asset_path=str(model_path))
    options = vision.FaceDetectorOptions(base_options=base_options, min_detection_confidence=min_conf)
    detector = vision.FaceDetector.create_from_options(options)

    def detect(gray8):
        h, w = gray8.shape
        rgb = np.dstack([gray8] * 3)
        mp_image = mp.Image(image_format=mp.ImageFormat.SRGB, data=rgb)
        res = detector.detect(mp_image)
        out = []
        for detection in res.detections:
            box = detection.bounding_box
            xmin = max(0, box.origin_x)
            ymin = max(0, box.origin_y)
            xmax = min(w, box.origin_x + box.width)
            ymax = min(h, box.origin_y + box.height)
            out.append((xmin, ymin, xmax, ymax))
        return out
    return detect

In [ ]:
from pathlib import Path

# Ensure the target directory exists before writing
Path('/content/drive/MyDrive/Intelligent_Temperature_Monitor/repo/src/pipeline').mkdir(parents=True, exist_ok=True)

code_content = """\"\"\"Thermal-only screening: radiometric frame -> face box -> face max -> optional calibration -> alert.

Fail-safe rules: no face, tiny ROI, invalid input, or no calibration => no alert, only a status.
\"\"\"
from dataclasses import dataclass
from typing import Optional
import numpy as np
from src.temperature.radiometric import raw_to_celsius

@dataclass(frozen=True)
class Calibration:
    slope: float
    intercept: float
    def apply(self, t_c: float) -> float:
        return self.slope * t_c + self.intercept

@dataclass(frozen=True)
class ScreenerConfig:
    window_c: tuple = (20.0, 38.0)        # fixed thermal -> 8-bit window for the face detector (assumption)
    min_face_height_px: int = 100         # provisional, from Charlotte distance analysis (one camera)
    alert_threshold_c: float = 37.5       # applied to calibrated (oral-equivalent) temperature
    calibration: Optional[Calibration] = None

@dataclass
class FaceReading:
    box: Optional[tuple]
    face_max_c: Optional[float]
    status: str                            # ok | no_face | unreliable_roi_small | invalid_input
    oral_equiv_c: Optional[float] = None
    alert: Optional[bool] = None

class ThermalScreener:
    def __init__(self, face_detector, config: ScreenerConfig = ScreenerConfig()):
        self.detect, self.cfg = face_detector, config

    def to_gray8(self, temp_c):
        lo, hi = self.cfg.window_c
        return np.clip((temp_c - lo) / (hi - lo) * 255, 0, 255).astype(np.uint8)

    def process_celsius(self, temp_c):
        if temp_c.ndim != 2 or not np.isfinite(temp_c).all():
            return [FaceReading(None, None, 'invalid_input')]
        h, w = temp_c.shape
        out = []
        for x0, y0, x1, y1 in self.detect(self.to_gray8(temp_c)):
            x0, y0, x1, y1 = max(int(x0), 0), max(int(y0), 0), min(int(x1), w), min(int(y1), h)
            if x1 - x0 < 2 or y1 - y0 < 2:
                continue
            r = FaceReading((x0, y0, x1, y1), float(temp_c[y0:y1, x0:x1].max()), 'ok')
            if (y1 - y0) < self.cfg.min_face_height_px:
                r.status = 'unreliable_roi_small'
            elif self.cfg.calibration is not None:
                r.oral_equiv_c = self.cfg.calibration.apply(r.face_max_c)
                r.alert = bool(r.oral_equiv_c >= self.cfg.alert_threshold_c)
            out.append(r)
        return out or [FaceReading(None, None, 'no_face')]

    def process_raw(self, raw16):
        if raw16.dtype != np.uint16:
            return [FaceReading(None, None, 'invalid_input')]
        return self.process_celsius(raw_to_celsius(raw16))
"""

Path('/content/drive/MyDrive/Intelligent_Temperature_Monitor/repo/src/pipeline/screener.py').write_text(code_content)
print('File written successfully!')

In [ ]:
%%writefile /content/drive/MyDrive/Intelligent_Temperature_Monitor/repo/tests/test_screener.py
import numpy as np
from src.pipeline.screener import ThermalScreener, ScreenerConfig, Calibration

def det(box):
    return lambda g: [box] if box else []

def frame(max_c=36.0):
    t = np.full((200, 300), 24.0); t[50:150, 100:200] = 34.0; t[80, 150] = max_c
    return t

BOX = (100, 50, 200, 150)   # 100 px tall

def test_no_face():
    r = ThermalScreener(det(None)).process_celsius(frame())[0]
    assert r.status == "no_face" and r.alert is None

def test_small_roi_flagged_not_alerted():
    cfg = ScreenerConfig(calibration=Calibration(1.0, 5.0))
    r = ThermalScreener(det((100, 50, 140, 90)), cfg).process_celsius(frame())[0]
    assert r.status == "unreliable_roi_small" and r.alert is None

def test_uncalibrated_measures_without_alert():
    r = ThermalScreener(det(BOX)).process_celsius(frame(36.5))[0]
    assert r.status == "ok" and abs(r.face_max_c - 36.5) < 1e-9 and r.alert is None

def test_alert_threshold():
    s = ThermalScreener(det(BOX), ScreenerConfig(calibration=Calibration(1.0, 1.0)))
    assert s.process_celsius(frame(36.0))[0].alert is False
    assert s.process_celsius(frame(36.6))[0].alert is True

def test_wrong_dtype_rejected():
    assert ThermalScreener(det(None)).process_raw(np.zeros((10, 10), np.uint8))[0].status == "invalid_input"

def test_nan_rejected():
    t = frame(); t[0, 0] = np.nan
    assert ThermalScreener(det((1, 1, 50, 50))).process_celsius(t)[0].status == "invalid_input"

In [ ]:
%%bash
cd /content/drive/MyDrive/Intelligent_Temperature_Monitor/repo
touch src/pipeline/__init__.py 2>/dev/null || (mkdir -p src/pipeline && touch src/pipeline/__init__.py)
python -m pytest -q

In [ ]:
import sys, subprocess
try:
    import mediapipe as mp
except ImportError:
    subprocess.run([sys.executable, "-m", "pip", "install", "-q", "mediapipe"])

import cv2, zipfile, time, importlib, numpy as np, pandas as pd
from pathlib import Path
import src.pipeline.screener as sm; importlib.reload(sm)
import src.detection.face_detector as fdm; importlib.reload(fdm)

OUT = Path("/content/cache/charlotte")
if not (OUT/".extracted_ok").exists():
    zipfile.ZipFile(ROOT/'datasets/charlotte_thermal/Charlotte-ThermalFace.zip').extractall(OUT); (OUT/".extracted_ok").touch()
t5 = {p.stem: p for p in next(q for q in OUT.rglob("S5") if q.is_dir()).glob("*.tiff")}
kept = pd.read_csv(ROOT/"experiments/charlotte_scan_v2.csv"); kept["d"] = kept.distance.round(1)

screener = sm.ThermalScreener(fdm.mediapipe_face_detector())
screener.process_raw(cv2.imread(str(next(iter(t5.values()))), cv2.IMREAD_UNCHANGED))   # warm-up
rows = []
for d in (1.0, 2.2, 4.0, 6.6):
    for r in kept[(kept.subject == 5) & (kept.d == d)].sample(3, random_state=0).itertuples():
        img = cv2.imread(str(t5[r.id]), cv2.IMREAD_UNCHANGED)
        t0 = time.perf_counter(); x = screener.process_raw(img)[0]; ms = (time.perf_counter() - t0) * 1000
        rows.append(dict(d=d, status=x.status, box_h=None if x.box is None else x.box[3] - x.box[1],
                         face_max=x.face_max_c, landmark_ref=r.max_hull, alert=x.alert, ms=ms))
print(pd.DataFrame(rows).round(2).to_string(index=False))

In [ ]:
import subprocess
from google.colab import userdata
token = userdata.get('GITHUB_TOKEN'); cwd = f"{ROOT}/repo"
def sh(*a): return subprocess.run(a, cwd=cwd, capture_output=True, text=True)
sh("git", "add", "src", "tests", "configs")
print(sh("git", "commit", "-m", "feat: add thermal screening pipeline with fail-safe status handling").stdout)
r = sh("git", "push", f"https://{token}@github.com/NidaKhaan/intelligent-body-temperature-monitoring.git", "main")
print(r.returncode, (r.stdout + r.stderr).replace(token, "***"))

In [ ]:
import numpy as np
import pandas as pd
from pathlib import Path
from sklearn.linear_model import LinearRegression

# Re-initialize path variables
ROOT = Path('/content/drive/MyDrive/Intelligent_Temperature_Monitor')
FDA = ROOT/'datasets/fda_physionet'

# Load clean FLIR data
fl = pd.read_csv(FDA/'clean_FLIR_groups1and2.csv')

# Reconstruct pooled features (rounds 1-4) and target
x = pd.concat([fl[f'T_Max{k}'] for k in range(1, 5)], ignore_index=True)
y = pd.concat([fl['aveOralM']] * 4, ignore_index=True)
ok = x.notna() & y.notna()

# Retrain linear regression baseline
lr = LinearRegression().fit(x[ok].to_frame(), y[ok])

# Map each pooled observation back to its original round (1-4)
rnd = pd.Series(np.repeat([1, 2, 3, 4], len(fl)), index=x.index)
xr, yr, rr = x[ok], y[ok], rnd[ok]
res = yr - lr.predict(xr.to_frame())

# Display residual stats per round and pooled standard deviation
print(res.groupby(rr).agg(['count', 'mean', 'std']).round(3))
print('pooled std:', round(res.std(), 3))

In [ ]:
import os
from pathlib import Path
# Ensure parent directory exists
Path('/content/drive/MyDrive/Intelligent_Temperature_Monitor/repo/backend').mkdir(parents=True, exist_ok=True)

code_content = """\"\"\"Thermal screening API. Images are processed in memory and never stored.\"\"\"
import os
os.environ.setdefault(\"OPENCV_IO_MAX_IMAGE_PIXELS\", str(2 ** 25))   # decompression-bomb guard (set before cv2 import)
import json
from dataclasses import asdict
from functools import lru_cache

import cv2
import numpy as np
from fastapi import Depends, FastAPI, File, HTTPException, UploadFile

from src.pipeline.screener import Calibration, ScreenerConfig, ThermalScreener

MAX_BYTES = 10 * 1024 * 1024
app = FastAPI(title=\"Thermal Screening API\", version=\"0.1.0\")


@lru_cache(maxsize=1)
def get_screener() -> ThermalScreener:
    from src.detection.face_detector import mediapipe_face_detector
    cal = None
    path = os.getenv(\"CALIBRATION_JSON\")        # optional; default = uncalibrated (no alerts)
    if path:
        c = json.load(open(path))
        cal = Calibration(c[\"slope\"], c[\"intercept\"])
    return ThermalScreener(mediapipe_face_detector(), ScreenerConfig(calibration=cal))


@app.get(\"/health\")
def health():
    return {\"status\": \"ok\"}


@app.post(\"/screen\")
async def screen(file: UploadFile = File(...), screener: ThermalScreener = Depends(get_screener)):
    data = await file.read(MAX_BYTES + 1)
    if len(data) > MAX_BYTES:
        raise HTTPException(413, \"file too large (max 10 MB)\")
    img = cv2.imdecode(np.frombuffer(data, np.uint8), cv2.IMREAD_UNCHANGED)
    if img is None:
        raise HTTPException(422, \"could not decode image\")
    if img.ndim != 2 or img.dtype != np.uint16:
        raise HTTPException(422, \"expected a single-channel 16-bit radiometric image (uint16)\")
    return {
        \"calibrated\": screener.cfg.calibration is not None,
        \"scale_assumption\": \"0.01 K/count, plausible on Charlotte-ThermalFace only, not validated per camera\",
        \"readings\": [asdict(r) for r in screener.process_raw(img)],
    }
"""

with open('/content/drive/MyDrive/Intelligent_Temperature_Monitor/repo/backend/app.py', 'w') as f:
    f.write(code_content)
print('File written successfully after creating the backend directory!')

In [ ]:
%%writefile /content/drive/MyDrive/Intelligent_Temperature_Monitor/repo/tests/test_api.py
import cv2, numpy as np, pytest
from fastapi.testclient import TestClient
import backend.app as be
from src.pipeline.screener import ThermalScreener, ScreenerConfig, Calibration

def screener(cal=None):
    return ThermalScreener(lambda g: [(100, 50, 200, 150)], ScreenerConfig(calibration=cal))

def frame16(max_c=36.5):
    t = np.full((200, 300), 24.0); t[50:150, 100:200] = 34.0; t[80, 150] = max_c
    return np.round((t + 273.15) / 0.01).astype(np.uint16)

def post(client, arr):
    ok, buf = cv2.imencode(".tiff", arr)
    return client.post("/screen", files={"file": ("f.tiff", buf.tobytes(), "image/tiff")})

@pytest.fixture
def make_client():
    def _make(s):
        be.app.dependency_overrides[be.get_screener] = lambda: s
        return TestClient(be.app)
    yield _make
    be.app.dependency_overrides.clear()

def test_health(make_client):
    assert make_client(screener()).get("/health").json() == {"status": "ok"}

def test_uncalibrated_measures_without_alert(make_client):
    j = post(make_client(screener()), frame16(36.5)).json()
    r = j["readings"][0]
    assert j["calibrated"] is False and r["status"] == "ok"
    assert abs(r["face_max_c"] - 36.5) < 0.01 and r["alert"] is None

def test_calibrated_alert(make_client):
    c = make_client(screener(Calibration(1.0, 1.0)))
    assert post(c, frame16(36.0)).json()["readings"][0]["alert"] is False
    assert post(c, frame16(36.6)).json()["readings"][0]["alert"] is True

def test_rejects_8bit_image(make_client):
    assert post(make_client(screener()), np.zeros((50, 50), np.uint8)).status_code == 422

def test_rejects_garbage(make_client):
    r = make_client(screener()).post("/screen", files={"file": ("x.tiff", b"not an image", "image/tiff")})
    assert r.status_code == 422

def test_rejects_oversize(make_client, monkeypatch):
    monkeypatch.setattr(be, "MAX_BYTES", 10)
    assert post(make_client(screener()), frame16()).status_code == 413

In [ ]:
%%bash
cd /content/drive/MyDrive/Intelligent_Temperature_Monitor/repo
mkdir -p backend && touch backend/__init__.py
cat > requirements.txt << 'EOF'
torch
torchvision
opencv-python-headless
mediapipe
numpy
pandas
matplotlib
scikit-learn
fastapi
uvicorn
python-multipart
httpx
streamlit
pytest
EOF
pip -q install python-multipart httpx 2>&1 | tail -1
python -m pytest -q

In [ ]:
import subprocess
from google.colab import userdata
token = userdata.get('GITHUB_TOKEN'); cwd = f"{ROOT}/repo"
def sh(*a): return subprocess.run(a, cwd=cwd, capture_output=True, text=True)
sh("git", "add", "backend", "tests", "requirements.txt")
print(sh("git", "commit", "-m", "feat: add FastAPI screening endpoint with input validation and tests").stdout)
r = sh("git", "push", f"https://{token}@github.com/NidaKhaan/intelligent-body-temperature-monitoring.git", "main")
print(r.returncode, (r.stdout + r.stderr).replace(token, "***"))

In [ ]:
import json, numpy as np, pandas as pd
from sklearn.linear_model import LinearRegression
fl = pd.read_csv(FDA/"clean_FLIR_groups1and2.csv")
xs = pd.concat([fl[f"T_Max{k}"] for k in range(1, 5)], ignore_index=True)
ys = pd.concat([fl["aveOralM"]] * 4, ignore_index=True)
rd_ = np.repeat([1, 2, 3, 4], len(fl)); ok = (xs.notna() & ys.notna()).to_numpy()
lr = LinearRegression().fit(xs[ok].to_frame(), ys[ok]); r = ys[ok] - lr.predict(xs[ok].to_frame())
g = r.groupby(rd_[ok])
p = ROOT/"repo/configs/calibration_fda_flir_tmax.json"; cfg = json.load(open(p))
cfg["residual_std_c_pooled_in_sample"] = cfg.pop("residual_std_c")
cfg["per_round_residual_mean_c"] = {int(k): round(float(v), 3) for k, v in g.mean().items()}
cfg["per_round_residual_std_c"] = {int(k): round(float(v), 3) for k, v in g.std().items()}
cfg["note"] = "Round 4 differs by ~0.4 C from rounds 1-3; pooled std is inflated by this between-round offset. Cause not investigated."
json.dump(cfg, open(p, "w"), indent=2); print(cfg)

import importlib.metadata as md
for pkg in ["streamlit", "mediapipe", "opencv-python-headless", "numpy", "fastapi", "scikit-learn", "pandas"]:
    try: print(pkg, md.version(pkg))
    except Exception as e: print(pkg, "not installed")

In [ ]:
from pathlib import Path

# Programmatically ensure the frontend directory exists
Path('/content/drive/MyDrive/Intelligent_Temperature_Monitor/repo/frontend').mkdir(parents=True, exist_ok=True)

code_content = """import cv2
import numpy as np

MODE_16 = \"Upload 16-bit radiometric image\"
MODE_8 = \"Upload 8-bit thermal image (detection only)\"
MODE_SYN = \"Synthetic frame (logic demo)\"


def synthetic_frame(face_max_c=36.5, face_h_px=160, shape=(348, 464), ambient_c=24.0, face_c=34.0):
    \"\"\"Synthetic thermal frame: warm rectangle (face) + one hot pixel. Returns (temp_c, box).\"\"\"
    h, w = shape
    fh = int(min(max(face_h_px, 10), h - 2)); fw = int(fh * 0.75)
    y0, x0 = (h - fh) // 2, (w - fw) // 2
    t = np.full(shape, ambient_c, dtype=np.float64)
    t[y0:y0 + fh, x0:x0 + fw] = face_c
    t[y0 + fh // 4, x0 + fw // 3] = face_max_c
    return t, (x0, y0, x0 + fw, y0 + fh)


def render_overlay(temp_c, readings, lo=20.0, hi=38.0):
    g = np.clip((temp_c - lo) / (hi - lo) * 255, 0, 255).astype(np.uint8)
    rgb = cv2.applyColorMap(g, cv2.COLORMAP_INFERNO)[:, :, ::-1].copy()
    for r in readings:
        if r.box is None:
            continue
        color = (255, 0, 0) if r.alert else ((255, 255, 255) if r.status == \"ok\" else (0, 200, 255))
        x0, y0, x1, y1 = r.box
        cv2.rectangle(rgb, (x0, y0), (x1, y1), color, 2)
    return rgb


def reading_row(source, r, ts):
    return {\"time\": ts, \"source\": source, \"status\": r.status,
            \"face_max_c\": None if r.face_max_c is None else round(r.face_max_c, 2),
            \"roi_height_px\": None if r.box is None else r.box[3] - r.box[1],
            \"alert\": r.alert}
"""

with open('/content/drive/MyDrive/Intelligent_Temperature_Monitor/repo/frontend/helpers.py', 'w') as f:
    f.write(code_content)
print('File written successfully after creating the frontend directory!')

In [ ]:
%%writefile /content/drive/MyDrive/Intelligent_Temperature_Monitor/repo/frontend/app.py
import sys, pathlib
sys.path.insert(0, str(pathlib.Path(__file__).resolve().parents[1]))   # repo root, for `streamlit run frontend/app.py`

import datetime as dt
import cv2, numpy as np, pandas as pd, streamlit as st
from frontend.helpers import MODE_16, MODE_8, MODE_SYN, synthetic_frame, render_overlay, reading_row
from src.pipeline.screener import Calibration, ScreenerConfig, ThermalScreener
from src.temperature.radiometric import raw_to_celsius

MAX_BYTES = 10 * 1024 * 1024
st.set_page_config(page_title="Thermal Screening Demo", layout="wide")
st.title("Thermal face-temperature screening (research demo)")
st.warning("Research prototype, not a medical device. Face surface temperature is not core body temperature. "
           "The 0.01 K/count scale is an assumption checked on one dataset only; accuracy is not validated for your camera. "
           "Images are processed in memory and not stored.")


@st.cache_resource
def get_detector():
    from src.detection.face_detector import mediapipe_face_detector
    return mediapipe_face_detector()


def decode(upload):
    data = upload.getvalue()
    if len(data) > MAX_BYTES:
        st.error("File too large (max 10 MB)."); return None
    img = cv2.imdecode(np.frombuffer(data, np.uint8), cv2.IMREAD_UNCHANGED)
    if img is None:
        st.error("Could not decode the image."); return None
    return img


if "history" not in st.session_state:
    st.session_state.history = []

mode = st.sidebar.radio("Input", [MODE_16, MODE_8, MODE_SYN], key="mode")
thr = st.sidebar.number_input("Alert threshold on measured face max (°C, uncalibrated)", 30.0, 45.0, 37.5, 0.1, key="thr")
min_h = st.sidebar.number_input("Minimum face height for a reliable reading (px)", 20, 400, 100, 5, key="minh")
sim = st.sidebar.slider("Simulated ΔT added to the frame (°C)", 0.0, 3.0, 0.0, 0.1, key="sim")
if sim > 0:
    st.error(f"SIMULATION: +{sim:.1f} °C added to the frame. This is not a measurement.")


def screen(temp_c, detector):
    cfg = ScreenerConfig(min_face_height_px=int(min_h), alert_threshold_c=float(thr), calibration=Calibration(1.0, 0.0))
    return ThermalScreener(detector, cfg).process_celsius(temp_c)


temp_c, readings, source = None, None, ""
if mode == MODE_16:
    up = st.file_uploader("16-bit single-channel radiometric TIFF/PNG", type=["tif", "tiff", "png"], key="up16")
    if up is None:
        st.info("Upload a 16-bit radiometric image (assumed 0.01 K per count).")
    else:
        img = decode(up)
        if img is not None:
            if img.ndim != 2 or img.dtype != np.uint16:
                st.error("Expected a single-channel 16-bit (uint16) image.")
            else:
                temp_c = raw_to_celsius(img) + sim
                readings, source = screen(temp_c, get_detector()), "upload_16bit"
elif mode == MODE_8:
    up = st.file_uploader("8-bit thermal image (grayscale or colorized)", type=["png", "jpg", "jpeg", "tif", "tiff"], key="up8")
    if up is None:
        st.info("Face detection only: an 8-bit image carries no temperature information.")
    else:
        img = decode(up)
        if img is not None:
            gray = cv2.cvtColor(img, cv2.COLOR_BGR2GRAY) if img.ndim == 3 else img
            if gray.dtype != np.uint8:
                st.error("Expected an 8-bit image.")
            else:
                vis = cv2.cvtColor(gray, cv2.COLOR_GRAY2RGB)
                boxes = get_detector()(gray)
                for x0, y0, x1, y1 in boxes:
                    cv2.rectangle(vis, (int(x0), int(y0)), (int(x1), int(y1)), (255, 255, 255), 2)
                st.image(vis, caption=f"{len(boxes)} face(s) detected. No temperature available.")
else:
    st.caption("Synthetic frame: a warm rectangle and one hot pixel. Tests ROI, status and alert logic without a face detector.")
    fmax = st.slider("Hottest pixel in face (°C)", 33.0, 40.0, 36.5, 0.1, key="fmax")
    fh = st.slider("Face height (px)", 20, 300, 160, 5, key="fh")
    temp_c, box = synthetic_frame(fmax + sim, fh)
    readings, source = screen(temp_c, lambda g: [box]), "synthetic"

if readings:
    left, right = st.columns([3, 2])
    left.image(render_overlay(temp_c, readings), caption="Colormap: 20 to 38 °C (white box = reading, orange = unreliable, red = alert)")
    now = dt.datetime.now().strftime("%Y-%m-%d %H:%M:%S")
    rows = [reading_row(source, r, now) for r in readings]
    right.dataframe(pd.DataFrame(rows), use_container_width=True)
    if any(r.alert for r in readings):
        right.error("Alert: measured face max is at or above the threshold.")
    elif any(r.status == "unreliable_roi_small" for r in readings):
        right.warning("Face too small for a reliable reading at this resolution.")
    if right.button("Log this reading", key="log"):
        st.session_state.history.extend(rows)

if st.session_state.history:
    st.subheader("Session history (in memory only)")
    hist = pd.DataFrame(st.session_state.history)
    st.dataframe(hist, use_container_width=True)
    st.download_button("Download report (CSV)", hist.to_csv(index=False).encode(), "screening_report.csv", "text/csv")
    if st.button("Clear history", key="clear"):
        st.session_state.history = []
        st.rerun()

In [ ]:
%%writefile /content/drive/MyDrive/Intelligent_Temperature_Monitor/repo/tests/test_frontend.py
from pathlib import Path
from frontend.helpers import MODE_SYN, synthetic_frame, reading_row
from src.pipeline.screener import ThermalScreener, ScreenerConfig, Calibration

def test_synthetic_frame_box_and_max():
    t, box = synthetic_frame(36.7, 160)
    x0, y0, x1, y1 = box
    assert 0 <= x0 < x1 <= t.shape[1] and 0 <= y0 < y1 <= t.shape[0]
    assert abs(t[y0:y1, x0:x1].max() - 36.7) < 1e-9

def test_screener_on_synthetic_alerts_only_above_threshold():
    cfg = ScreenerConfig(calibration=Calibration(1.0, 0.0), alert_threshold_c=37.5)
    for fmax, expect in [(37.0, False), (38.0, True)]:
        t, box = synthetic_frame(fmax, 160)
        assert ThermalScreener(lambda g: [box], cfg).process_celsius(t)[0].alert is expect

def test_reading_row_has_no_image_data():
    t, box = synthetic_frame(36.5, 160)
    r = ThermalScreener(lambda g: [box]).process_celsius(t)[0]
    assert set(reading_row("synthetic", r, "t")) == {"time", "source", "status", "face_max_c", "roi_height_px", "alert"}

def test_app_runs_default_and_synthetic():
    from streamlit.testing.v1 import AppTest
    app = str(Path(__file__).resolve().parents[1] / "frontend" / "app.py")
    at = AppTest.from_file(app, default_timeout=30).run()
    assert not at.exception
    at.radio(key="mode").set_value(MODE_SYN).run()
    assert not at.exception and len(at.dataframe) >= 1

In [ ]:
%%bash
cd /content/drive/MyDrive/Intelligent_Temperature_Monitor/repo
mkdir -p frontend && touch frontend/__init__.py
pip -q install streamlit 2>&1 | tail -1
python -m pytest -q

In [ ]:
import subprocess
from google.colab import userdata
token = userdata.get('GITHUB_TOKEN'); cwd = f"{ROOT}/repo"
def sh(*a): return subprocess.run(a, cwd=cwd, capture_output=True, text=True)
sh("git", "add", "frontend", "tests", "configs")
print(sh("git", "commit", "-m", "feat: add Streamlit dashboard with synthetic, 8-bit and 16-bit modes").stdout)
r = sh("git", "push", f"https://{token}@github.com/NidaKhaan/intelligent-body-temperature-monitoring.git", "main")
print(r.returncode, (r.stdout + r.stderr).replace(token, "***"))

In [ ]:
import sys, importlib.metadata as md, subprocess
from pathlib import Path

REPO = ROOT/"repo"
(REPO/"deploy").mkdir(exist_ok=True); (REPO/"scripts").mkdir(exist_ok=True)
py = f"{sys.version_info.major}.{sys.version_info.minor}"
st_ver = md.version("streamlit")
pins = {p: md.version(p) for p in ["numpy", "pandas", "opencv-python-headless", "mediapipe"]}

(REPO/"deploy/space_requirements.txt").write_text("".join(f"{k}=={v}\n" for k, v in pins.items()))
(REPO/"deploy/packages.txt").write_text("libgl1\nlibglib2.0-0\n")
(REPO/"deploy/SPACE_README.md").write_text(f"""---
title: Thermal Face Screening Demo
emoji: 🌡️
colorFrom: red
colorTo: yellow
sdk: streamlit
sdk_version: {st_ver}
python_version: "{py}"
app_file: frontend/app.py
pinned: false
---

# Thermal face-temperature screening (research demo)

Research prototype, not a medical device. Face surface temperature is not core body temperature,
and accuracy is not validated for arbitrary cameras. Images are processed in memory and not stored.
Source code and evaluation: https://github.com/NidaKhaan/intelligent-body-temperature-monitoring
""", encoding="utf-8")

(REPO/"scripts/deploy_space.py").write_text(r'''"""Upload a whitelisted subset of this repo to a Hugging Face Space.
Usage: python scripts/deploy_space.py <user/space> [--dry-run]   (HF_TOKEN in the environment)"""
import os, shutil, sys, tempfile
from pathlib import Path

ROOT = Path(__file__).resolve().parents[1]
INCLUDE = ["src", "frontend", "configs"]
IGNORE = shutil.ignore_patterns("__pycache__", "*.pyc", ".ipynb_checkpoints")

def build(dest: Path):
    for d in INCLUDE:
        shutil.copytree(ROOT / d, dest / d, ignore=IGNORE)
    shutil.copyfile(ROOT / "deploy" / "space_requirements.txt", dest / "requirements.txt")
    shutil.copyfile(ROOT / "deploy" / "packages.txt", dest / "packages.txt")
    shutil.copyfile(ROOT / "deploy" / "SPACE_README.md", dest / "README.md")

if __name__ == "__main__":
    repo_id, dry = sys.argv[1], "--dry-run" in sys.argv
    with tempfile.TemporaryDirectory() as tmp:
        tmp = Path(tmp); build(tmp)
        files = sorted(str(p.relative_to(tmp)) for p in tmp.rglob("*") if p.is_file())
        print(len(files), "files:"); print("\n".join(files))
        if not dry:
            from huggingface_hub import HfApi
            HfApi().upload_folder(folder_path=str(tmp), repo_id=repo_id, repo_type="space",
                                  token=os.environ["HF_TOKEN"], commit_message="Deploy from GitHub repo")
''', encoding="utf-8")

# public upload ke liye decompression-bomb guard (cv2 import se pehle)
p = REPO/"frontend/app.py"; s = p.read_text()
if "OPENCV_IO_MAX_IMAGE_PIXELS" not in s:
    old = "import datetime as dt\nimport cv2"
    assert old in s, "app.py ka format alag hai"
    p.write_text(s.replace(old, 'import os\nos.environ.setdefault("OPENCV_IO_MAX_IMAGE_PIXELS", str(2 ** 25))\n' + old))

print("python", py, "| streamlit", st_ver, "| pins", pins)
print(subprocess.run([sys.executable, "-m", "pytest", "-q"], cwd=REPO, capture_output=True, text=True).stdout[-300:])
print(subprocess.run([sys.executable, "scripts/deploy_space.py", "x/y", "--dry-run"], cwd=REPO, capture_output=True, text=True).stdout)

In [ ]:
import subprocess, sys, os
from google.colab import userdata
SPACE = "NidaaKhaan/thermal-face-screening"
env = {**os.environ, "HF_TOKEN": userdata.get("HF_TOKEN")}
r = subprocess.run([sys.executable, "scripts/deploy_space.py", SPACE], cwd=REPO, env=env,
                   capture_output=True, text=True)
print("exit code:", r.returncode)
print((r.stdout + r.stderr).replace(env["HF_TOKEN"], "***")[-600:])

In [ ]:
%%writefile /content/drive/MyDrive/Intelligent_Temperature_Monitor/repo/README.md
# Thermal Face-Temperature Screening (research prototype)

> **Not a medical device.** Face surface temperature is not core body temperature. The 0.3 °C figure from the
> original brief was reached only under the conditions listed below; end-to-end accuracy with automatic
> face detection could not be validated on any public dataset.

**Live demo:** https://huggingface.co/spaces/NidaaKhaan/thermal-face-screening
(synthetic frame, 8-bit detection-only and 16-bit radiometric upload modes; images are processed in memory and not stored)

## What it does

A thermal-only pipeline: 16-bit radiometric frame, face box, maximum temperature inside the face ROI,
optional calibration to an oral-equivalent value, alert. It fails safe: no face, a face too small to
measure reliably, invalid input, or no calibration produces a status instead of an alert.
It ships as a Streamlit dashboard and a FastAPI endpoint. All numbers below come from public datasets;
the evaluation scripts and result files are in `docs/results/`.

## Scope against the original brief

| Brief item | Status |
|---|---|
| Person detection | Evaluated on LLVIP (below); optional module, not wired into the dashboard |
| Face detection | MediaPipe on the thermal frame, in the pipeline |
| RGB-thermal alignment | **Not implemented in the pipeline.** Registration was only analysed on one SpeakingFaces subject |
| Face-person association | Not implemented |
| Temperature from thermal data | Implemented (16-bit counts to °C, assumed 0.01 K/count) |
| Error <= 0.3 °C | MAE met, RMSE borderline, only with expert ROI at 0.8 m on one camera model |
| <= 1 s per frame | Face detection, ROI and reading take 4 to 22 ms per frame on a Colab CPU (350x465 frames) |
| Live camera input | Not implemented or tested |
| History and report export | Session history in memory only, CSV download |

## Architecture

```mermaid
flowchart LR
    subgraph Research["Research (Colab + Google Drive)"]
        D1["FDA / PhysioNet"] --> E1["Calibration and error analysis"]
        D2["Charlotte-ThermalFace"] --> E2["Distance and ROI sensitivity"]
        D3["LLVIP"] --> E3["Person detector evaluation"]
        D4["SpeakingFaces sample"] --> E4["Registration analysis"]
    end
    subgraph App["Application"]
        UI["Streamlit dashboard"] --> S["ThermalScreener"]
        API["FastAPI /screen"] --> S
    end
    E1 --> CFG["configs/calibration_fda_flir_tmax.json"]
    CFG --> S
```

## Pipeline

```mermaid
flowchart TD
    A["16-bit radiometric frame"] --> B["Counts to Celsius (0.01 K per count)"]
    B --> C["Fixed 20-38 C window to 8-bit"]
    C --> D["MediaPipe face detector"]
    D --> E{"Face found?"}
    E -- no --> F["status: no_face"]
    E -- yes --> G{"ROI height >= minimum px?"}
    G -- no --> H["status: unreliable_roi_small"]
    G -- yes --> I["Face max inside ROI"]
    I --> J{"Calibration loaded?"}
    J -- no --> K["Measured value only, no alert"]
    J -- yes --> L["Oral-equivalent value and alert"]
```

## Results

### 1. Facial temperature vs oral thermometer (FDA dataset, FLIR camera, about 0.8 m)

Features are the dataset's own extracted values (expert ROI), not images. Linear calibration, 5-fold cross-validation, 1,020 subjects.

| Feature | Raw bias (°C) | Calibrated MAE | Calibrated RMSE |
|---|---|---|---|
| T_Max, mean of 4 rounds | -0.94 | 0.210 (95% CI 0.200 to 0.220) | 0.271 |
| Inner canthi max, mean of 4 rounds | -1.24 | 0.248 (95% CI 0.236 to 0.260) | 0.319 |
| T_Max, single round | n/a | 0.234 to 0.259 | 0.304 to 0.333 |
| Inner canthi max, single round | n/a | 0.250 to 0.251 | 0.322 to 0.327 |

* Febrile subjects (oral >= 37.5 °C, n = 111) are under-estimated: MAE 0.313 °C, bias -0.27 °C. At a 37.5 °C threshold sensitivity is 0.76 and specificity 0.98, so about one in four fevers is missed.
* A threshold of 37.2 °C gives sensitivity 0.95 and specificity 0.84, but it was chosen on the same data and is exploratory only.
* Round 4 sits about 0.4 °C below rounds 1 to 3 (per-round residual means +0.15, +0.07, +0.06, -0.29). The cause was not investigated, so one calibration can drift between sessions.
* The saved calibration (slope 0.587, intercept 15.84) is valid for the FDA camera and ROI definition only. Applied to Charlotte-ThermalFace it would flag healthy volunteers, so the application ships uncalibrated.

### 2. Distance effect (Charlotte-ThermalFace, radiometric, 9 of 10 subjects usable)

Mean face-max inside the landmark hull, all subjects pooled (subject mix varies by distance):

| Distance | Face pixels | Face max (°C) |
|---|---|---|
| 1.0 m | ~14,900 | 36.47 |
| 1.4 m | ~7,600 | 36.42 |
| 1.8 m | ~4,600 | 36.12 |
| 2.8 m | ~1,900 | 35.97 |
| 4.0 m | ~900 | 35.75 |
| 6.6 m | ~375 | 35.33 |

* For the 5 subjects recorded at both 1 m and 6.6 m the face max falls by 1.15 to 1.49 °C.
* Downsampling the 1 m images to the 6.6 m face size reproduces only about 19 to 32% of that drop. The remainder is unexplained (optics, camera distance compensation, landmark error and subject cooling are untested hypotheses).
* Consequence: readings are flagged unreliable below a minimum face height (provisional 100 px, which is roughly 1.4 m on this sensor).

### 3. ROI placement (Charlotte-ThermalFace, 600 images, 4 subjects)

| Distance | Shift of ROI by 10% of face height | Automatic box vs landmark ROI, mean abs. difference | IoU with landmark box |
|---|---|---|---|
| 1.0 m | 0.05 °C | 0.15 °C | 0.70 |
| 2.8 m | 0.04 °C | 0.08 °C | 0.59 |
| 6.6 m | 0.01 °C | 0.05 °C | 0.40 |

* Shifting the ROI barely changes the face max (above 0.1 °C in 9% of images at a 10% shift). This holds for a single warm face on a cold background and says nothing about crowds, rotation, scale error or camera parallax.
* The automatic box reads about 0.15 °C higher than the landmark ROI at 1 m, a bias on top of the FDA calibration.

### 4. Person detection (LLVIP, first 500 consecutive test pairs, IoU 0.5, torchvision Faster R-CNN, COCO weights)

| Model / modality | AP50 | Precision | Recall | Median time (T4 GPU) |
|---|---|---|---|---|
| MobileNet / visible | 0.707 | 0.791 | 0.639 | 59 ms |
| MobileNet / infrared | 0.431 | 0.426 | 0.535 | 59 ms |
| ResNet50 / visible | 0.783 | 0.814 | 0.735 | 156 ms |
| ResNet50 / infrared | 0.836 | 0.829 | 0.780 | 155 ms |

On a Colab CPU MobileNet takes about 0.56 s and ResNet50 about 6.9 s per frame, so ResNet50 does not meet the 1 s target on CPU. The 500 frames come from a few scenes and there are no confidence intervals.

### 5. RGB-thermal registration (SpeakingFaces, one subject, two trials)

* The thermal images are 8-bit colorized PNGs, not radiometric, so this dataset is used for registration and detection only.
* The homography between raw and aligned RGB is stable within one camera position (std 0.10 to 0.27 px, 0.16 px over consecutive frames) and differs between the nine positions.
* MediaPipe boxes on RGB and on grayscale thermal frames overlap with median IoU 0.81 (100 frames, one subject), a proxy and not ground truth.

### 6. Error budget (estimate, not measured)

Calibration RMSE 0.30 to 0.33 °C, automatic ROI bias about 0.15 °C and the 1.4 m distance effect about 0.05 °C combine to roughly 0.35 °C RMSE at 1.4 m or closer, assuming independent errors. No public dataset pairs images with an oral reference, so this estimate cannot be checked end to end.

## Limitations

* The 0.01 K/count scale is an assumption: it gives plausible skin temperatures on Charlotte-ThermalFace and FDA face maxima are close (35.8 °C vs 36.4 °C at about 1 m), but absolute accuracy was not checked against a blackbody reference.
* The two cameras differ by about 0.6 °C at comparable distance, so calibrations do not transfer between cameras.
* Public datasets are mostly controlled indoor scenes with a single subject; crowds, outdoor conditions, glasses, masks and fever-range subjects other than FDA's 111 are untested.
* Close-range (about 1 to 1.4 m) screening is the validated regime. Distant crowd monitoring needs higher-resolution optics.
* Charlotte-ThermalFace: one annotation session (S3) is missing from the zip, and S2 has corrupted distance values. Affected rows were excluded.

## Datasets and licences

| Dataset | Used for | Terms |
|---|---|---|
| FDA / PhysioNet face-oral temperature data | Calibration and error analysis | CC0 |
| Charlotte-ThermalFace | Distance and ROI analysis | Research use only, no licence file; images are not redistributed |
| LLVIP | Person-detector evaluation | Non-commercial; images are not redistributed |
| SpeakingFaces (sample, one subject) | Registration analysis | CC BY 4.0 as stated by ISSAI (verify on the Hugging Face dataset card) |

No dataset image is stored in this repository. Pretrained COCO weights come from torchvision and their terms should be checked before any commercial use.

## Privacy and ethics

The tool performs no identification or face recognition. The dashboard keeps readings in session memory only and the API processes uploads in memory. Using thermal screening in public spaces raises consent, retention and discrimination questions that this prototype does not solve.

## Repository layout

src/pipeline/ ThermalScreener (fail-safe screening logic)
src/temperature/ radiometric conversion, FDA baseline analysis
src/detection/ face detector wrapper, person detector (evaluation)
src/utils/ resumable stage tracker (atomic JSON checkpoints)
src/alignment/ early eye-alignment experiment, not used by the pipeline
backend/ FastAPI app
frontend/ Streamlit dashboard
configs/ calibration for the FDA camera only
docs/results/ evaluation outputs
scripts/ Hugging Face Space deployment
tests/ 20 tests


## Run it

```bash
pip install -r requirements.txt
python -m pytest -q
streamlit run frontend/app.py
uvicorn backend.app:app          # then: curl -F "file=@frame.tiff" http://localhost:8000/screen
```

The API accepts single-channel 16-bit images up to 10 MB and returns the status and face max per face. `calibrated` is false unless `CALIBRATION_JSON` points to a calibration file.

## Persistence in Colab

Datasets, checkpoints and results live on Google Drive. A `StageTracker` records completed stages in `progress.json` with atomic writes, so a disconnected runtime resumes where it stopped. Stages whose output lives on the volatile Colab disk must be re-run after a reset (the tracker supports a `verify` check for this).

## Citations

* Charlotte-ThermalFace: "A Novel Fully Annotated Thermal Infrared Face Dataset: Recorded in Various Environment Conditions and Distances From The Camera", arXiv:2205.02093.
* LLVIP: Jia et al., "LLVIP: A Visible-infrared Paired Dataset for Low-light Vision", ICCV Workshops 2021.
* FDA / PhysioNet: https://physionet.org/content/face-oral-temp-data/1.0.0/
* SpeakingFaces: ISSAI dataset page and paper (citation to be verified against the official page).

## Licence

Code: MIT (see `LICENSE`). Dataset terms above apply separately.

In [ ]:
from datetime import date
(REPO/"LICENSE").write_text(f"""MIT License

Copyright (c) {date.today().year} Nida Sheraz

Permission is hereby granted, free of charge, to any person obtaining a copy
of this software and associated documentation files (the "Software"), to deal
in the Software without restriction, including without limitation the rights
to use, copy, modify, merge, publish, distribute, sublicense, and/or sell
copies of the Software, and to permit persons to whom the Software is
furnished to do so, subject to the following conditions:

The above copyright notice and this permission notice shall be included in all
copies or substantial portions of the Software.

THE SOFTWARE IS PROVIDED "AS IS", WITHOUT WARRANTY OF ANY KIND, EXPRESS OR
IMPLIED, INCLUDING BUT NOT LIMITED TO THE WARRANTIES OF MERCHANTABILITY,
FITNESS FOR A PARTICULAR PURPOSE AND NONINFRINGEMENT. IN NO EVENT SHALL THE
AUTHORS OR COPYRIGHT HOLDERS BE LIABLE FOR ANY CLAIM, DAMAGES OR OTHER
LIABILITY, WHETHER IN AN ACTION OF CONTRACT, TORT OR OTHERWISE, ARISING FROM,
OUT OF OR IN CONNECTION WITH THE SOFTWARE OR THE USE OR OTHER DEALINGS IN THE
SOFTWARE.
""")
from google.colab import userdata
import subprocess
token = userdata.get('GITHUB_TOKEN'); cwd = str(REPO)
def sh(*a): return subprocess.run(a, cwd=cwd, capture_output=True, text=True)
sh("git", "add", "README.md", "LICENSE", "deploy", "scripts", "frontend")
print(sh("git", "commit", "-m", "docs: add README with architecture, results and limitations; add MIT license").stdout)
r = sh("git", "push", f"https://{token}@github.com/NidaKhaan/intelligent-body-temperature-monitoring.git", "main")
print(r.returncode, (r.stdout + r.stderr).replace(token, "***"))

In [ ]:
%%writefile /content/drive/MyDrive/Intelligent_Temperature_Monitor/repo/src/alignment/registration.py
"""RGB -> thermal registration with a calibrated homography (fixed camera pair)."""
import numpy as np
import cv2


class HomographyRegistration:
    def __init__(self, H):
        H = np.asarray(H, dtype=np.float64)
        if H.shape != (3, 3) or not np.isfinite(H).all():
            raise ValueError("H must be a finite 3x3 matrix")
        self.H = H

    @classmethod
    def from_points(cls, src, dst, ransac_thr=3.0):
        """Returns (registration, mean reprojection error in px over inliers, number of inliers)."""
        src, dst = np.asarray(src, np.float32), np.asarray(dst, np.float32)
        if len(src) != len(dst) or len(src) < 4:
            raise ValueError("need at least 4 matching points")
        H, inl = cv2.findHomography(src, dst, cv2.RANSAC, ransac_thr)
        if H is None:
            raise ValueError("homography estimation failed")
        reg, mask = cls(H), inl.ravel().astype(bool)
        err = np.linalg.norm(reg.map_points(src[mask]) - dst[mask], axis=1)
        return reg, float(err.mean()), int(mask.sum())

    def map_points(self, pts):
        pts = np.asarray(pts, np.float32).reshape(-1, 1, 2)
        return cv2.perspectiveTransform(pts, self.H.astype(np.float64)).reshape(-1, 2)

    def map_box(self, box):
        x0, y0, x1, y1 = box
        c = self.map_points([[x0, y0], [x1, y0], [x1, y1], [x0, y1]])
        return (float(c[:, 0].min()), float(c[:, 1].min()), float(c[:, 0].max()), float(c[:, 1].max()))

In [ ]:
%%writefile /content/drive/MyDrive/Intelligent_Temperature_Monitor/repo/tests/test_registration.py
import numpy as np
import pytest
from src.alignment.registration import HomographyRegistration

def test_translation_maps_box():
    H = np.array([[1, 0, 10], [0, 1, -5], [0, 0, 1]], float)
    assert np.allclose(HomographyRegistration(H).map_box((0, 0, 20, 30)), (10, -5, 30, 25))

def test_from_points_recovers_known_transform():
    rng = np.random.default_rng(0)
    src = rng.uniform(0, 400, (40, 2))
    H = np.array([[0.6, 0.02, 12.0], [-0.01, 0.62, 7.0], [0, 0, 1]])
    truth = HomographyRegistration(H)
    reg, err, n = HomographyRegistration.from_points(src, truth.map_points(src))
    assert err < 1e-2 and n == 40
    assert np.allclose(reg.map_points([[100, 100]]), truth.map_points([[100, 100]]), atol=0.05)

def test_rejects_bad_input():
    with pytest.raises(ValueError):
        HomographyRegistration(np.eye(2))
    with pytest.raises(ValueError):
        HomographyRegistration.from_points([[0, 0]] * 3, [[0, 0]] * 3)

In [ ]:
%%bash
cd /content/drive/MyDrive/Intelligent_Temperature_Monitor/repo && python -m pytest -q

In [ ]:
import sys, json, cv2, numpy as np, pandas as pd
import matplotlib; matplotlib.use("Agg")
import matplotlib.pyplot as plt
import mediapipe as mp
from mediapipe.tasks import python
from mediapipe.tasks.python import vision
from pathlib import Path

REPO = ROOT/"repo"; sys.path.insert(0, str(REPO))
IMG = REPO/"docs/images"; IMG.mkdir(parents=True, exist_ok=True)
from src.alignment.registration import HomographyRegistration
from src.pipeline.screener import ThermalScreener, ScreenerConfig, Calibration
from frontend.helpers import synthetic_frame, render_overlay
import src.temperature.fda_baseline as fb

# ---------- Figure 1: detection + RGB->thermal registration (SpeakingFaces, CC BY 4.0) ----------
SUB = ROOT/"datasets/speaking_faces/processed/sample_trial1_every81"
def idx(f): return {p.stem.rsplit("_", 1)[0]: p for p in (SUB/f).glob("*.png")}
thr, raw, ali = idx("thr_image"), idx("rgb_image"), idx("rgb_image_aligned")
common = sorted(set(thr) & set(raw) & set(ali))
pos = lambda s: s.split("_")[3]
sift, bf = cv2.SIFT_create(), cv2.BFMatcher()

def calibrate(s):
    a, b = (cv2.cvtColor(cv2.imread(str(d[s])), cv2.COLOR_BGR2GRAY) for d in (raw, ali))
    k1, d1 = sift.detectAndCompute(a, None); k2, d2 = sift.detectAndCompute(b, None)
    if d1 is None or d2 is None: return None
    m = [p[0] for p in bf.knnMatch(d1, d2, k=2) if len(p) == 2 and p[0].distance < 0.75 * p[1].distance]
    if len(m) < 12: return None
    return HomographyRegistration.from_points([k1[x.queryIdx].pt for x in m], [k2[x.trainIdx].pt for x in m])

by_pos = {}
for s in common: by_pos.setdefault(pos(s), []).append(s)
cal, test = None, None
for p, frames in sorted(by_pos.items()):
    if len(frames) < 3: continue
    for s in frames[:-1]:
        r = calibrate(s)
        if r and r[2] >= 40: cal, test = r, frames[-1]; break
    if cal: break
assert cal, "calibration frame nahi mila (common frames: %d)" % len(common)
reg, reproj, n_inl = cal

# Initialize modern MediaPipe face detector
model_path = Path("/content/cache/detector.tflite")
if not model_path.exists():
    model_path.parent.mkdir(parents=True, exist_ok=True)
    import urllib.request
    url = "https://storage.googleapis.com/mediapipe-models/face_detector/blaze_face_short_range/float16/1/blaze_face_short_range.tflite"
    urllib.request.urlretrieve(url, model_path)

base_options = python.BaseOptions(model_asset_path=str(model_path))
options = vision.FaceDetectorOptions(base_options=base_options, min_detection_confidence=0.3)
detector = vision.FaceDetector.create_from_options(options)

def face(img_rgb):
    h, w = img_rgb.shape[:2]
    mp_image = mp.Image(image_format=mp.ImageFormat.SRGB, data=img_rgb)
    res = detector.detect(mp_image)
    if not res.detections: return None
    best = max(res.detections, key=lambda d: d.categories[0].score)
    b = best.bounding_box
    return (b.origin_x, b.origin_y, b.origin_x + b.width, b.origin_y + b.height)

def iou(a, b):
    x0, y0, x1, y1 = max(a[0], b[0]), max(a[1], b[1]), min(a[2], b[2]), min(a[3], b[3])
    i = max(0, x1 - x0) * max(0, y1 - y0)
    return i / ((a[2]-a[0])*(a[3]-a[1]) + (b[2]-b[0])*(b[3]-b[1]) - i + 1e-9)

raw_rgb = cv2.cvtColor(cv2.imread(str(raw[test])), cv2.COLOR_BGR2RGB)
thr_rgb = cv2.cvtColor(cv2.imread(str(thr[test])), cv2.COLOR_BGR2RGB)
b_rgb = face(raw_rgb); assert b_rgb, "RGB frame par chehra nahi mila"
b_map = reg.map_box(b_rgb)
thr_gray3 = cv2.cvtColor(cv2.cvtColor(thr_rgb, cv2.COLOR_RGB2GRAY), cv2.COLOR_GRAY2RGB)
b_thr = face(thr_gray3)
v = iou(b_map, b_thr) if b_thr else float("nan")
warped = cv2.warpPerspective(raw_rgb, reg.H, (thr_rgb.shape[1], thr_rgb.shape[0]))
blend = cv2.addWeighted(cv2.cvtColor(warped, cv2.COLOR_RGB2GRAY), 0.5, cv2.cvtColor(thr_rgb, cv2.COLOR_RGB2GRAY), 0.5, 0)

def rect(ax, b, c, ls="-"): ax.add_patch(plt.Rectangle((b[0], b[1]), b[2]-b[0], b[3]-b[1], fill=False, ec=c, lw=2, ls=ls))
fig, ax = plt.subplots(1, 3, figsize=(15, 4.4))
ax[0].imshow(raw_rgb); rect(ax[0], b_rgb, "lime"); ax[0].set_title("1. RGB camera: face detected", fontsize=10)
ax[1].imshow(thr_rgb); rect(ax[1], b_map, "white")
if b_thr: rect(ax[1], b_thr, "cyan", "--")
ax[1].set_title(f"2. Box mapped to thermal (white) vs box found on thermal (cyan)\nIoU {v:.2f}", fontsize=10)
ax[2].imshow(blend, cmap="gray"); ax[2].set_title("3. RGB warped onto thermal (50% blend)", fontsize=10)
for a in ax: a.axis("off")
fig.suptitle(f"Homography calibrated on one frame (mean reprojection error {reproj:.2f} px, {n_inl} inliers) and applied to a "
             f"different frame of the same camera position | SpeakingFaces (ISSAI), CC BY 4.0", fontsize=8)
plt.tight_layout(); plt.savefig(IMG/"alignment_demo.png", dpi=130); plt.close()
print("alignment: IoU", round(v, 3), "| reproj", round(reproj, 2), "| inliers", n_inl, "| position", pos(test))

# ---------- Figure 2: screening logic on synthetic frames ----------
cases = [("Close, no calibration loaded", 36.5, 160, False), ("Close, calibrated, above threshold", 38.0, 160, True),
         ("Face too small to measure", 38.0, 60, True)]
fig, axes = plt.subplots(1, 3, figsize=(15, 4.3))
for ax, (title, fmax, fh, use_cal) in zip(axes, cases):
    t, box = synthetic_frame(fmax, fh)
    cfg = ScreenerConfig(calibration=Calibration(1.0, 0.0) if use_cal else None, alert_threshold_c=37.5)
    r = ThermalScreener(lambda g, b=box: [b], cfg).process_celsius(t)[0]
    ax.imshow(render_overlay(t, [r])); ax.axis("off")
    ax.set_title(f"{title}\nstatus: {r.status} | face max {r.face_max_c:.1f} C | alert: {r.alert}", fontsize=9)
fig.suptitle("Synthetic frames (not real people): the screener measures, flags small ROIs, and alerts only when calibrated", fontsize=9)
plt.tight_layout(); plt.savefig(IMG/"screening_logic.png", dpi=130); plt.close()

# ---------- Figure 3: results overview ----------
RES = REPO/"docs/results"
dist = pd.read_csv(RES/"charlotte_distance_summary.csv")
roi = pd.read_csv(RES/"charlotte_roi_sensitivity_v1.csv")
ll = json.load(open(RES/"llvip_eval_v2_test_subset.json"))
flir = pd.read_csv(FDA/"clean_FLIR_groups1and2.csv")
x, y = fb.facial_feature(flir, "T_Max"), flir["aveOralM"]; ok = x.notna() & y.notna()
err = fb.cv_calibrate(x[ok], y[ok]) - y[ok]

fig, ax = plt.subplots(2, 2, figsize=(12, 8.5))
ax[0, 0].scatter(y[ok], err, s=7, alpha=.5); ax[0, 0].axhline(0, c="k", lw=1); ax[0, 0].axvline(37.5, c="r", ls="--", lw=1)
ax[0, 0].set_xlabel("Oral temperature (C)"); ax[0, 0].set_ylabel("Calibrated estimate - oral (C)")
ax[0, 0].set_title(f"FDA: calibration error (MAE {err.abs().mean():.2f} C, n={int(ok.sum())})\nfever subjects (right of red line) are under-estimated", fontsize=9)
for nl in sorted(dist.n_lm.unique()):
    d = dist[dist.n_lm == nl]; ax[0, 1].plot(d.d, d.mean_max_c, marker="o", label=f"{nl} landmarks")
ax[0, 1].set_xlabel("Distance (m)"); ax[0, 1].set_ylabel("Mean face max (C)"); ax[0, 1].legend()
ax[0, 1].set_title("Charlotte: measured face max falls with distance", fontsize=9)
g = roi.groupby("d")
ax[1, 0].plot(g["shift_0.1"].mean().index, g["shift_0.1"].mean().values, marker="o", label="ROI shifted by 10% of face height")
ab = roi.box_minus_ref.abs().groupby(roi.d).mean(); ax[1, 0].plot(ab.index, ab.values, marker="s", label="automatic box vs landmark ROI")
ax[1, 0].set_xlabel("Distance (m)"); ax[1, 0].set_ylabel("Change in face max (C)"); ax[1, 0].legend(fontsize=8)
ax[1, 0].set_title("ROI placement effect (600 images, 4 subjects)", fontsize=9)
keys = ["mobilenet/visible", "mobilenet/infrared", "resnet50/visible", "resnet50/infrared"]
ax[1, 1].bar(range(4), [ll[k]["AP50"] for k in keys]); ax[1, 1].set_xticks(range(4)); ax[1, 1].set_xticklabels(keys, rotation=15, fontsize=8)
ax[1, 1].set_ylabel("AP50"); ax[1, 1].set_title("LLVIP person detection (500 consecutive test frames)", fontsize=9)
plt.tight_layout(); plt.savefig(IMG/"results_overview.png", dpi=130); plt.close()
print("saved:", sorted(p.name for p in IMG.iterdir()))

In [ ]:
import subprocess
from google.colab import userdata
rd = REPO/"README.md"; s = rd.read_text(encoding="utf-8")

i = s.find("**Live demo:**")
if i != -1:
    j = s.find("\n\n", i); s = s[:i] + s[j + 2:]

block = """## At a glance

### Face detection and RGB-thermal registration (real data)
![Registration demo](docs/images/alignment_demo.png)

*SpeakingFaces (ISSAI, CC BY 4.0), one subject. A homography is estimated from one frame of a camera position and applied to another frame of the same position. The dataset's thermal images are 8-bit, so they carry no temperature. The IoU compares the mapped box with a box MediaPipe finds on the thermal image: a proxy, not ground truth.*

### Screening logic (synthetic frames, not real people)
![Screening logic](docs/images/screening_logic.png)

### Results overview (aggregate numbers from public datasets)
![Results overview](docs/images/results_overview.png)

"""
if "## At a glance" not in s:
    s = s.replace("## What it does", block + "## What it does", 1)
s = s.replace("| RGB-thermal alignment | **Not implemented in the pipeline.** Registration was only analysed on one SpeakingFaces subject |",
              "| RGB-thermal alignment | Homography registration module (`src/alignment/registration.py`, tested), demonstrated on one SpeakingFaces subject; not wired into the thermal-only pipeline |")
s = s.replace("src/alignment/      early eye-alignment experiment, not used by the pipeline",
              "src/alignment/      registration.py (RGB-to-thermal homography); face_aligner.py is an early unused experiment")
rd.write_text(s, encoding="utf-8")

token = userdata.get('GITHUB_TOKEN'); cwd = str(REPO)
def sh(*a): return subprocess.run(a, cwd=cwd, capture_output=True, text=True)
sh("git", "add", "README.md", "docs/images", "src/alignment/registration.py", "tests/test_registration.py")
print(sh("git", "commit", "-m", "feat: add homography registration module; docs: add figures to README").stdout)
r = sh("git", "push", f"https://{token}@github.com/NidaKhaan/intelligent-body-temperature-monitoring.git", "main")
print(r.returncode, (r.stdout + r.stderr).replace(token, "***"))

In [ ]:
%%bash
pip -q install playwright 2>&1 | tail -1
playwright install --with-deps chromium 2>&1 | tail -2

In [ ]:
import subprocess, sys, time, asyncio
p = REPO/"frontend/app.py"; s = p.read_text()
old = "    right.dataframe(pd.DataFrame(rows), use_container_width=True)"
new = ("    r0 = readings[0]; c1, c2 = right.columns(2)\n"
       "    c1.metric('Face max (measured)', 'n/a' if r0.face_max_c is None else f'{r0.face_max_c:.1f} °C')\n"
       "    c2.metric('Status', r0.status.replace('_', ' '))\n" + old)
if "Face max (measured)" not in s:
    assert old in s, "app.py ka format alag hai"; p.write_text(s.replace(old, new))
print(subprocess.run([sys.executable, "-m", "pytest", "-q"], cwd=REPO, capture_output=True, text=True).stdout[-200:])

proc = subprocess.Popen([sys.executable, "-m", "streamlit", "run", "frontend/app.py", "--server.headless", "true",
                         "--server.port", "8501"], cwd=REPO, stdout=subprocess.DEVNULL, stderr=subprocess.DEVNULL)
time.sleep(15)

from playwright.async_api import async_playwright
async def capture_screenshot():
    async with async_playwright() as pw:
        b = await pw.chromium.launch()
        pg = await b.new_page(viewport={"width": 1400, "height": 1000})
        await pg.goto("http://localhost:8501")
        await pg.wait_for_timeout(4000)
        await pg.get_by_text("Synthetic frame (logic demo)").click()
        await pg.wait_for_timeout(3500)
        await pg.screenshot(path=str(IMG/"dashboard_synthetic.png"))
        await b.close()

# Run the async capture task in the existing Jupyter event loop
await capture_screenshot()
proc.terminate()
print("saved", (IMG/"dashboard_synthetic.png").exists())

In [ ]:
import subprocess
from google.colab import userdata
rd = REPO/"README.md"; s = rd.read_text(encoding="utf-8")
add = """### Dashboard (synthetic mode)
![Dashboard](docs/images/dashboard_synthetic.png)

*Streamlit dashboard in synthetic mode: measured face max, status, a clearly labelled simulated temperature offset, session history and CSV export. Uploads of 16-bit radiometric images are supported. Nothing is stored.*

"""
if "### Dashboard (synthetic mode)" in s:
    print("dashboard section already present")
elif "### Results overview" in s:
    rd.write_text(s.replace("### Results overview", add + "### Results overview", 1), encoding="utf-8"); print("section added")
else:
    print("README mein 'At a glance' wala hissa nahi mila: Cell S pehle chalayein")

token = userdata.get('GITHUB_TOKEN'); cwd = str(REPO)
def sh(*a): return subprocess.run(a, cwd=cwd, capture_output=True, text=True)
print(sh("git", "status", "--short").stdout)
sh("git", "add", "README.md", "frontend", "docs/images")
print(sh("git", "commit", "-m", "feat: dashboard metric cards; docs: add dashboard screenshot").stdout)
r = sh("git", "push", f"https://{token}@github.com/NidaKhaan/intelligent-body-temperature-monitoring.git", "main")
print(r.returncode, (r.stdout + r.stderr).replace(token, "***"))

In [ ]:
import subprocess, sys, importlib.metadata as md
import pandas as pd
from pathlib import Path
from google.colab import userdata

REPO = ROOT/"repo"
def put(rel, text):
    p = REPO/rel; p.parent.mkdir(parents=True, exist_ok=True); p.write_text(text, encoding="utf-8")

# ======================= scripts/make_figures.py =======================
MAKE_FIGURES = r'''"""Generate README figures from result files and datasets on Google Drive.
Usage: python scripts/make_figures.py [drive_project_root]
Each figure is independent; failures are reported at the end."""
import json, re, sys, traceback, zipfile
from pathlib import Path

import cv2
import numpy as np
import pandas as pd
import matplotlib
matplotlib.use("Agg")
import matplotlib.pyplot as plt
from matplotlib.patches import FancyBboxPatch

REPO = Path(__file__).resolve().parents[1]
DRIVE = Path(sys.argv[1]) if len(sys.argv) > 1 else Path("/content/drive/MyDrive/Intelligent_Temperature_Monitor")
IMG, RES = REPO / "docs" / "images", REPO / "docs" / "results"
IMG.mkdir(parents=True, exist_ok=True); RES.mkdir(parents=True, exist_ok=True)
sys.path.insert(0, str(REPO))

GREY, BLUE, GRN, YEL, ORG, RED = "#e5e7eb", "#dbeafe", "#dcfce7", "#fef3c7", "#ffedd5", "#fee2e2"


def box(ax, x, y, w, h, text, fc, ls="-", fs=9):
    ax.add_patch(FancyBboxPatch((x, y), w, h, boxstyle="round,pad=0.02,rounding_size=0.08",
                                fc=fc, ec="#1f2937", lw=1.3, ls=ls))
    ax.text(x + w / 2, y + h / 2, text, ha="center", va="center", fontsize=fs, color="#111827")


def arrow(ax, p, q, txt=None, ls="-"):
    ax.annotate("", xy=q, xytext=p, arrowprops=dict(arrowstyle="-|>", lw=1.4, color="#374151", linestyle=ls))
    if txt:
        ax.text((p[0] + q[0]) / 2 + 0.1, (p[1] + q[1]) / 2 + 0.12, txt, fontsize=8, color="#374151")


def diagram_pipeline():
    fig, ax = plt.subplots(figsize=(14, 6.2)); ax.set_xlim(0, 14); ax.set_ylim(0, 7.2); ax.axis("off")
    W, H = 3.1, 1.3; xs = [0.3, 3.8, 7.3, 10.8]; top, mid, low = 5.4, 3.0, 0.6
    box(ax, xs[0], top, W, H, "16-bit radiometric\nthermal frame", BLUE)
    box(ax, xs[1], top, W, H, "Counts to degrees C\n(0.01 K per count, assumed)", BLUE)
    box(ax, xs[2], top, W, H, "Fixed 20-38 C window\nto an 8-bit image", BLUE)
    box(ax, xs[3], top, W, H, "MediaPipe\nface detector", BLUE)
    for i in range(3):
        arrow(ax, (xs[i] + W, top + H / 2), (xs[i + 1], top + H / 2))
    box(ax, xs[3], mid, W, H, "Face found?", YEL)
    box(ax, xs[2], mid, W, H, "ROI height at least\n100 px?", YEL)
    box(ax, xs[1], mid, W, H, "Face maximum\ninside the ROI", BLUE)
    box(ax, xs[0], mid, W, H, "Calibration\nloaded?", YEL)
    arrow(ax, (xs[3] + W / 2, top), (xs[3] + W / 2, mid + H))
    arrow(ax, (xs[3], mid + H / 2), (xs[2] + W, mid + H / 2), "yes")
    arrow(ax, (xs[2], mid + H / 2), (xs[1] + W, mid + H / 2), "yes")
    arrow(ax, (xs[1], mid + H / 2), (xs[0] + W, mid + H / 2))
    box(ax, xs[3], low, W, H, "status: no_face\n(no alert)", ORG)
    box(ax, xs[2], low, W, H, "status: unreliable_roi_small\n(no alert)", ORG)
    box(ax, xs[1], low, W, H, "calibrated: oral-equivalent\nvalue and alert flag", RED)
    box(ax, xs[0], low, W, H, "uncalibrated: measured\nvalue only, no alert", GRN)
    arrow(ax, (xs[3] + W / 2, mid), (xs[3] + W / 2, low + H), "no")
    arrow(ax, (xs[2] + W / 2, mid), (xs[2] + W / 2, low + H), "no")
    arrow(ax, (xs[0] + W / 2, mid), (xs[0] + W / 2, low + H), "no")
    arrow(ax, (xs[0] + W, mid + 0.2), (xs[1] + W / 2, low + H), "yes")
    ax.set_title("Thermal-only screening pipeline: every doubtful case yields a status, never an alert", fontsize=12)
    plt.tight_layout(); plt.savefig(IMG / "pipeline.png", dpi=140); plt.close()


def diagram_architecture():
    fig, ax = plt.subplots(figsize=(15, 7.8)); ax.set_xlim(0, 16); ax.set_ylim(0, 8.7); ax.axis("off")
    for x, t in [(2.1, "Public datasets"), (7.3, "Experiments (Colab + Google Drive)"), (13.3, "Application")]:
        ax.text(x, 8.3, t, ha="center", fontsize=11, fontweight="bold")
    rows = [("FDA / PhysioNet\n(CSV features + oral reference)", "Calibration and error analysis\n(regions, ROC)"),
            ("Charlotte-ThermalFace\n(16-bit radiometric)", "Distance effect and\nROI sensitivity"),
            ("LLVIP\n(visible + infrared)", "Person-detector evaluation"),
            ("SpeakingFaces sample\n(CC BY 4.0)", "RGB-thermal registration\nand face alignment")]
    ys = [6.5, 4.9, 3.3, 1.7]
    for (d, e), y in zip(rows, ys):
        box(ax, 0.3, y, 3.6, 1.2, d, GREY); box(ax, 5.2, y, 4.2, 1.2, e, BLUE)
        arrow(ax, (3.9, y + 0.6), (5.2, y + 0.6))
    box(ax, 11.0, 6.6, 4.6, 1.2, "configs/calibration_fda_flir_tmax.json\n(valid for the FDA camera only)", ORG)
    box(ax, 11.0, 4.5, 4.6, 1.4, "ThermalScreener\n(src/pipeline, fail-safe)", GRN, fs=10)
    box(ax, 11.0, 2.6, 2.2, 1.0, "FastAPI\n/screen", BLUE)
    box(ax, 13.4, 2.6, 2.2, 1.0, "Streamlit\ndashboard", BLUE)
    arrow(ax, (9.4, 7.1), (11.0, 7.2))
    arrow(ax, (13.3, 6.6), (13.3, 5.9))
    arrow(ax, (12.1, 4.5), (12.1, 3.6)); arrow(ax, (14.5, 4.5), (14.5, 3.6))
    arrow(ax, (9.4, 2.3), (11.0, 5.0), "optional", ls="--")
    box(ax, 0.3, 0.2, 15.3, 0.8, "StageTracker: resumable stages with atomic JSON checkpoints on Google Drive "
        "(datasets, results and progress survive Colab disconnects)", GREY)
    plt.tight_layout(); plt.savefig(IMG / "architecture.png", dpi=140); plt.close()


def fda_regions():
    from sklearn.linear_model import LinearRegression
    from sklearn.metrics import roc_auc_score, roc_curve
    from sklearn.model_selection import KFold, cross_val_predict
    fl = pd.read_csv(DRIVE / "datasets/fda_physionet/clean_FLIR_groups1and2.csv")
    y_all = fl["aveOralM"]; pat = re.compile(r"^(.*?)_?([1-4])$"); groups = {}
    for c in fl.columns:
        m = pat.match(c)
        if m:
            groups.setdefault(m.group(1), {})[int(m.group(2))] = c
    groups = {b: d for b, d in groups.items() if len(d) == 4 and b != "T_offset"
              and all(pd.api.types.is_numeric_dtype(fl[c]) for c in d.values())}
    cv = KFold(5, shuffle=True, random_state=42); rows, cache = [], {}
    for b, d in groups.items():
        maes, aucs = [], []
        for r in (1, 2, 3, 4):
            x = fl[d[r]]; ok = x.notna() & y_all.notna()
            if ok.sum() < 200:
                continue
            p = cross_val_predict(LinearRegression(), x[ok].to_frame(), y_all[ok], cv=cv)
            maes.append(float(np.abs(p - y_all[ok]).mean())); aucs.append(float(roc_auc_score(y_all[ok] >= 37.5, x[ok])))
            if r == 1:
                cache[b] = (x[ok].to_numpy(), y_all[ok].to_numpy(), p)
        if maes:
            rows.append(dict(feature=b, mean_MAE=np.mean(maes), mean_AUC=np.mean(aucs)))
    res = pd.DataFrame(rows).sort_values("mean_MAE").reset_index(drop=True)
    res.round(4).to_csv(RES / "fda_region_summary.csv", index=False)
    top = res.head(14)
    fig, ax = plt.subplots(1, 3, figsize=(18, 5.8), gridspec_kw={"width_ratios": [1.25, 1, 0.8]})
    colors = ["#ef4444" if f in ("T_Max", "canthiMax") else "#60a5fa" for f in top.feature]
    ax[0].barh(range(len(top)), top.mean_MAE, color=colors); ax[0].set_yticks(range(len(top)))
    ax[0].set_yticklabels(top.feature, fontsize=9); ax[0].invert_yaxis()
    for i, (m, a) in enumerate(zip(top.mean_MAE, top.mean_AUC)):
        ax[0].text(m + 0.002, i, f"AUC {a:.2f}", va="center", fontsize=8)
    ax[0].set_xlabel("Calibrated MAE vs oral thermometer (C); mean of 4 single rounds, 5-fold CV")
    ax[0].set_title("Which facial measurement tracks oral temperature best?\n(14 lowest-error features; red = T_Max, canthiMax)", fontsize=10)
    cand = [b for b in dict.fromkeys(list(top.feature.head(3)) + ["T_Max"]) if b in cache]
    for b in cand:
        x, y, _ = cache[b]; f, t, _ = roc_curve(y >= 37.5, x)
        ax[1].plot(f, t, label=f"{b} (AUC {roc_auc_score(y >= 37.5, x):.2f})")
    ax[1].plot([0, 1], [0, 1], "k--", lw=1); ax[1].legend(fontsize=8, loc="lower right")
    ax[1].set_xlabel("False positive rate"); ax[1].set_ylabel("True positive rate")
    ax[1].set_title("Detecting fever (oral >= 37.5 C), round 1", fontsize=10)
    x, y, p = cache["T_Max"]; fever, flag = y >= 37.5, p >= 37.5
    cm = np.array([[(~fever & ~flag).sum(), (~fever & flag).sum()], [(fever & ~flag).sum(), (fever & flag).sum()]])
    ax[2].imshow(cm, cmap="Blues")
    for i in range(2):
        for j in range(2):
            ax[2].text(j, i, int(cm[i, j]), ha="center", va="center", fontsize=14,
                       color="white" if cm[i, j] > cm.max() / 2 else "black")
    ax[2].set_xticks([0, 1]); ax[2].set_xticklabels(["not flagged", "flagged"])
    ax[2].set_yticks([0, 1]); ax[2].set_yticklabels(["oral < 37.5", "oral >= 37.5"])
    ax[2].set_title(f"T_Max, round 1, flag at 37.5 C\nsensitivity {cm[1, 1] / cm[1].sum():.2f}, specificity {cm[0, 0] / cm[0].sum():.2f}", fontsize=10)
    plt.tight_layout(); plt.savefig(IMG / "fda_regions.png", dpi=130); plt.close()


def charlotte_hotspots():
    from src.temperature.radiometric import raw_to_celsius
    OUT = Path("/content/cache/charlotte")
    if not (OUT / ".extracted_ok").exists():
        zipfile.ZipFile(DRIVE / "datasets/charlotte_thermal/Charlotte-ThermalFace.zip").extractall(OUT)
        (OUT / ".extracted_ok").touch()
    scan = pd.read_csv(DRIVE / "experiments/charlotte_scan_v2.csv"); scan["d"] = scan.distance.round(1)
    scan = scan[scan.subject.isin([1, 4, 5, 9]) & (scan.n_lm == 73) & (scan.d <= 1.4)]
    xc, yc = [f"x{i}" for i in range(73)], [f"y{i}" for i in range(73)]
    U, V, T = [], [], []
    for s in [1, 4, 5, 9]:
        sdir = next(q for q in OUT.rglob(f"S{s}") if q.is_dir())
        tiffs = {p.stem: p for p in sdir.glob("*.tiff")}
        ann = pd.read_csv(DRIVE / f"datasets/charlotte_thermal/annotations/S{s}.csv").dropna(subset=["ID"]).drop_duplicates("ID").set_index("ID")
        sub = scan[scan.subject == s]; sub = sub.sample(min(150, len(sub)), random_state=0)
        for r in sub.itertuples():
            if r.id not in tiffs or r.id not in ann.index:
                continue
            row = ann.loc[r.id]; xs, ys = row[xc].to_numpy(float), row[yc].to_numpy(float)
            if not (np.isfinite(xs).all() and np.isfinite(ys).all()):
                continue
            img = cv2.imread(str(tiffs[r.id]), cv2.IMREAD_UNCHANGED); h, w = img.shape
            pts = np.stack([np.clip(xs, 0, w - 1), np.clip(ys, 0, h - 1)], 1).astype(np.int32)
            mask = np.zeros((h, w), np.uint8); cv2.fillConvexPoly(mask, cv2.convexHull(pts), 1)
            c = raw_to_celsius(img); c[mask == 0] = -np.inf
            yy, xx = np.unravel_index(np.argmax(c), c.shape)
            x0, x1, y0, y1 = xs.min(), xs.max(), ys.min(), ys.max()
            U.append((xx - x0) / (x1 - x0)); V.append((yy - y0) / (y1 - y0))
            T.append(np.stack([(xs - x0) / (x1 - x0), (ys - y0) / (y1 - y0)], 1))
    assert len(U) > 50, f"too few usable frames: {len(U)}"
    U, V, T = np.array(U), np.array(V), np.mean(T, axis=0)
    fig, ax = plt.subplots(figsize=(5.8, 6.6))
    h2 = ax.hist2d(U, V, bins=[24, 28], range=[[0, 1], [0, 1]], cmap="inferno")
    ax.scatter(T[:, 0], T[:, 1], s=14, c="white", edgecolors="black", linewidths=.4)
    ax.invert_yaxis(); fig.colorbar(h2[3], ax=ax, label="frames")
    ax.set_xlabel("relative x inside the landmark box"); ax.set_ylabel("relative y (0 = top of the box)")
    ax.set_title(f"Where the hottest pixel falls (white dots: mean 73-landmark layout)\n{len(U)} frontal frames, 4 subjects, up to 1.4 m", fontsize=9)
    plt.tight_layout(); plt.savefig(IMG / "charlotte_hotspots.png", dpi=140); plt.close()
    json.dump({"n": int(len(U)), "median_x": float(np.median(U)), "median_y": float(np.median(V))},
              open(RES / "charlotte_hotspot_summary.json", "w"))


def face_alignment():
    import mediapipe as mp
    from src.alignment.face_aligner import FaceAligner
    SUB = DRIVE / "datasets/speaking_faces/processed/sample_trial1_every81"
    def idx(f): return {p.stem.rsplit("_", 1)[0]: p for p in (SUB / f).glob("*.png")}
    thr, ali = idx("thr_image"), idx("rgb_image_aligned")
    common = sorted(set(thr) & set(ali)); by_pos = {}
    for s in common:
        by_pos.setdefault(s.split("_")[3], s)
    picks = [by_pos[k] for k in sorted(by_pos)][:: max(len(by_pos) // 3, 1)][:3]
    fd = mp.solutions.face_detection.FaceDetection(model_selection=1, min_detection_confidence=0.3)
    aligner = FaceAligner(target_size=(256, 256), eye_target_ratio=0.35); rows = []
    for s in picks:
        rgb = cv2.cvtColor(cv2.imread(str(ali[s])), cv2.COLOR_BGR2RGB); th = cv2.cvtColor(cv2.imread(str(thr[s])), cv2.COLOR_BGR2RGB)
        h, w = rgb.shape[:2]; res = fd.process(rgb)
        if not res.detections:
            continue
        d = max(res.detections, key=lambda q: q.score[0])
        kp = [(k.x * w, k.y * h) for k in d.location_data.relative_keypoints]
        left, right = sorted(kp[:2], key=lambda p: p[0])
        a_rgb, M = aligner.align_face_by_eyes(rgb, left, right)
        rows.append((rgb, kp, d.location_data.relative_bounding_box, a_rgb, cv2.warpAffine(th, M, (256, 256), flags=cv2.INTER_LINEAR)))
    assert rows, "no face detected in the picked frames"
    fig, ax = plt.subplots(len(rows), 3, figsize=(10.5, 3.5 * len(rows)), squeeze=False)
    for i, (rgb, kp, bb, a_rgb, a_th) in enumerate(rows):
        h, w = rgb.shape[:2]
        ax[i, 0].imshow(rgb); ax[i, 1].imshow(a_rgb); ax[i, 2].imshow(a_th)
        ax[i, 0].add_patch(plt.Rectangle((bb.xmin * w, bb.ymin * h), bb.width * w, bb.height * h, fill=False, ec="lime", lw=2))
        ax[i, 0].scatter([p[0] for p in kp], [p[1] for p in kp], s=28, c="red", edgecolors="white", linewidths=.6)
        for a in ax[i]:
            a.axis("off")
    ax[0, 0].set_title("MediaPipe face box and 6 keypoints", fontsize=10)
    ax[0, 1].set_title("Eye-based alignment (RGB, 256x256)", fontsize=10)
    ax[0, 2].set_title("Same transform applied to thermal", fontsize=10)
    fig.suptitle("SpeakingFaces (ISSAI), CC BY 4.0: aligned RGB and 8-bit thermal frames from three camera positions", fontsize=8)
    plt.tight_layout(); plt.savefig(IMG / "face_alignment.png", dpi=130); plt.close()


if __name__ == "__main__":
    jobs = [("pipeline.png", diagram_pipeline), ("architecture.png", diagram_architecture), ("fda_regions.png", fda_regions),
            ("charlotte_hotspots.png", charlotte_hotspots), ("face_alignment.png", face_alignment)]
    failed = []
    for name, fn in jobs:
        try:
            fn(); print("ok  ", name)
        except Exception as e:
            failed.append(name); print("FAIL", name, "->", repr(e)); traceback.print_exc(limit=3)
    print("failed:", failed)
'''

# ======================= frontend/app.py =======================
APP = r'''import os
os.environ.setdefault("OPENCV_IO_MAX_IMAGE_PIXELS", str(2 ** 25))
import sys
import pathlib
sys.path.insert(0, str(pathlib.Path(__file__).resolve().parents[1]))

import datetime as dt
import cv2
import numpy as np
import pandas as pd
import streamlit as st
import matplotlib
matplotlib.use("Agg")
import matplotlib.pyplot as plt
from sklearn.linear_model import LinearRegression
from sklearn.metrics import roc_auc_score, roc_curve
from sklearn.model_selection import KFold, cross_val_predict

from frontend.helpers import render_overlay, reading_row
from src.pipeline.screener import Calibration, ScreenerConfig, ThermalScreener
from src.temperature.radiometric import raw_to_celsius

MAX_BYTES = 10 * 1024 * 1024
DATA = pathlib.Path(__file__).resolve().parent / "data" / "fda_flir_features.csv"
FEATURES = {"Whole-face maximum": "T_Max", "Inner-canthus maximum": "canthiMax", "Forehead maximum": "T_FH_Max"}

st.set_page_config(page_title="Thermal screening", page_icon="🌡️", layout="wide")
st.title("Thermal face-temperature screening")
st.caption("Research prototype, not a medical device. Face surface temperature is not core body temperature. "
           "Images are processed in memory and never stored.")
tab_scan, tab_fda, tab_method = st.tabs(["Screen an image", "FDA calibration explorer", "Method and limits"])


@st.cache_resource
def get_detector():
    from src.detection.face_detector import mediapipe_face_detector
    return mediapipe_face_detector()


@st.cache_data
def load_fda():
    return pd.read_csv(DATA)


@st.cache_data
def calibrated(base: str, rnd: str):
    df = load_fda()
    x = df[[f"{base}{k}" for k in range(1, 5)]].mean(axis=1) if rnd == "Mean of 4" else df[f"{base}{rnd}"]
    y = df["oral_c"]; ok = x.notna() & y.notna()
    x, y = x[ok].to_numpy(), y[ok].to_numpy()
    p = cross_val_predict(LinearRegression(), x.reshape(-1, 1), y, cv=KFold(5, shuffle=True, random_state=42))
    return x, y, p


def decode(upload):
    data = upload.getvalue()
    if len(data) > MAX_BYTES:
        st.error("File too large (max 10 MB)."); return None
    img = cv2.imdecode(np.frombuffer(data, np.uint8), cv2.IMREAD_UNCHANGED)
    if img is None:
        st.error("Could not decode the image."); return None
    return img


if "history" not in st.session_state:
    st.session_state.history = []

with tab_scan:
    left, right = st.columns([3, 2])
    with right:
        min_h = st.number_input("Minimum face height for a reliable reading (px)", 20, 400, 100, 5, key="minh")
        thr = st.number_input("Flag when the measured face maximum reaches (°C)", 30.0, 45.0, 37.5, 0.1, key="thr")
        st.caption("The flag applies to the measured surface temperature. No calibration is applied in this tab; "
                   "the next tab shows what calibration can and cannot do.")
    with left:
        up = st.file_uploader("Thermal image: 16-bit radiometric (temperature) or 8-bit (detection only)",
                              type=["tif", "tiff", "png", "jpg", "jpeg"], key="up")
    if up is None:
        st.info("Upload a 16-bit single-channel radiometric image (assumed 0.01 K per count) to get a reading. "
                "An 8-bit image only shows face detection because it carries no temperature.")
    else:
        img = decode(up)
        if img is not None and img.ndim == 2 and img.dtype == np.uint16:
            temp_c = raw_to_celsius(img)
            cfg = ScreenerConfig(min_face_height_px=int(min_h), alert_threshold_c=float(thr), calibration=Calibration(1.0, 0.0))
            readings = ThermalScreener(get_detector(), cfg).process_celsius(temp_c)
            a, b = st.columns([3, 2])
            a.image(render_overlay(temp_c, readings), caption="20 to 38 °C colormap. White box: reading, orange: unreliable, red: flagged")
            r0 = readings[0]; m1, m2 = b.columns(2)
            m1.metric("Face max (measured)", "n/a" if r0.face_max_c is None else f"{r0.face_max_c:.1f} °C")
            m2.metric("Status", r0.status.replace("_", " "))
            rows = [reading_row("upload_16bit", r, dt.datetime.now().strftime("%Y-%m-%d %H:%M:%S")) for r in readings]
            b.dataframe(pd.DataFrame(rows))
            if any(r.alert for r in readings):
                b.error("Flagged: the measured face maximum is at or above the threshold.")
            elif any(r.status == "unreliable_roi_small" for r in readings):
                b.warning("Face too small for a reliable reading at this resolution.")
            if b.button("Log this reading", key="log"):
                st.session_state.history.extend(rows)
        elif img is not None and img.dtype == np.uint8:
            gray = cv2.cvtColor(img, cv2.COLOR_BGR2GRAY) if img.ndim == 3 else img
            vis = cv2.cvtColor(gray, cv2.COLOR_GRAY2RGB); boxes = get_detector()(gray)
            for x0, y0, x1, y1 in boxes:
                cv2.rectangle(vis, (int(x0), int(y0)), (int(x1), int(y1)), (255, 255, 255), 2)
            st.image(vis, caption=f"{len(boxes)} face(s) detected. No temperature available from an 8-bit image.")
        elif img is not None:
            st.error("Expected a single-channel 16-bit image (temperature) or an 8-bit image (detection only).")
    if st.session_state.history:
        st.subheader("Session history (memory only)")
        hist = pd.DataFrame(st.session_state.history); st.dataframe(hist)
        st.download_button("Download report (CSV)", hist.to_csv(index=False).encode(), "screening_report.csv", "text/csv")
        if st.button("Clear history", key="clear"):
            st.session_state.history = []; st.rerun()

with tab_fda:
    st.markdown("How well does a facial temperature predict an oral thermometer? Data: 1,020 volunteers, one FLIR camera, "
                "about 0.8 m (FDA / PhysioNet, CC0, de-identified). The features are the dataset's own extracted values, "
                "so this shows the calibration problem, not image processing.")
    c1, c2, c3 = st.columns(3)
    feat = c1.selectbox("Facial measurement", list(FEATURES), key="feat")
    rnd = c2.selectbox("Round", ["1", "2", "3", "4", "Mean of 4"], key="rnd")
    thr_c = c3.slider("Flag when the oral-equivalent estimate reaches (°C)", 36.8, 38.0, 37.5, 0.1, key="thr_c")
    x, y, pred = calibrated(FEATURES[feat], rnd)
    err = pred - y; fever = y >= 37.5; flag = pred >= thr_c
    sens = float((fever & flag).sum() / max(fever.sum(), 1)); spec = float((~fever & ~flag).sum() / max((~fever).sum(), 1))
    m = st.columns(4)
    m[0].metric("MAE", f"{np.abs(err).mean():.2f} °C"); m[1].metric("RMSE", f"{np.sqrt((err ** 2).mean()):.2f} °C")
    m[2].metric("Sensitivity", f"{sens:.2f}"); m[3].metric("Specificity", f"{spec:.2f}")
    p1, p2 = st.columns(2)
    fig, ax = plt.subplots(figsize=(5.5, 4.2)); ax.scatter(y, pred, s=6, alpha=.45)
    lim = [min(y.min(), pred.min()) - .1, max(y.max(), pred.max()) + .1]; ax.plot(lim, lim, "k--", lw=1)
    ax.axvline(37.5, c="r", lw=1); ax.axhline(thr_c, c="orange", lw=1)
    ax.set_xlabel("Oral temperature (°C)"); ax.set_ylabel("Calibrated estimate (°C)")
    ax.set_title("Estimate vs oral (red: fever = oral >= 37.5)", fontsize=10); p1.pyplot(fig); plt.close(fig)
    fpr, tpr, _ = roc_curve(fever, x)
    fig, ax = plt.subplots(figsize=(5.5, 4.2)); ax.plot(fpr, tpr); ax.plot([0, 1], [0, 1], "k--", lw=1)
    ax.scatter([1 - spec], [sens], c="orange", zorder=3, label="chosen threshold"); ax.legend()
    ax.set_xlabel("False positive rate"); ax.set_ylabel("True positive rate")
    ax.set_title(f"ROC for fever detection (AUC {roc_auc_score(fever, x):.2f})", fontsize=10); p2.pyplot(fig); plt.close(fig)
    st.caption("Calibration is a 5-fold cross-validated linear fit. Thresholds chosen here are tuned on the same data, "
               "so treat the operating point as exploratory. The orange line and dot show your threshold.")

with tab_method:
    st.markdown("""
**Pipeline.** 16-bit radiometric frame, conversion to °C (assumed 0.01 K per count), face detection on a fixed 20 to 38 °C window,
maximum temperature inside the face box, optional calibration. No face, a face under the minimum height, invalid input or no
calibration gives a status instead of an alert.

**What was measured.**

| Question | Answer from public data |
|---|---|
| Raw facial temperature vs oral | Facial maximum reads about 0.9 to 1.6 °C below oral |
| After linear calibration (FDA camera, 0.8 m, expert ROI) | MAE 0.21 to 0.29 °C, RMSE 0.27 to 0.39 °C |
| Fever subjects | Under-estimated; sensitivity about 0.76 at 37.5 °C |
| Distance 1 m to 6.6 m | Face maximum falls by 1.15 to 1.49 °C |
| Two different cameras | About 0.6 °C apart at similar distance |

**Limits.** The scale is an assumption. Calibrations do not transfer between cameras. Controlled indoor data, one face per frame.
Reliable readings need a face of about 100 px or more (roughly 1.4 m on this sensor). Not a medical device.
""")
'''

# ======================= tests/test_frontend.py =======================
TESTS = r'''from pathlib import Path

import pandas as pd

from frontend.helpers import synthetic_frame, reading_row
from src.pipeline.screener import ThermalScreener, ScreenerConfig, Calibration

ROOT = Path(__file__).resolve().parents[1]
APP = str(ROOT / "frontend" / "app.py")
DATA = ROOT / "frontend" / "data" / "fda_flir_features.csv"


def test_synthetic_frame_box_and_max():
    t, box = synthetic_frame(36.7, 160)
    x0, y0, x1, y1 = box
    assert 0 <= x0 < x1 <= t.shape[1] and 0 <= y0 < y1 <= t.shape[0]
    assert abs(t[y0:y1, x0:x1].max() - 36.7) < 1e-9


def test_screener_alerts_only_above_threshold():
    cfg = ScreenerConfig(calibration=Calibration(1.0, 0.0), alert_threshold_c=37.5)
    for fmax, expect in [(37.0, False), (38.0, True)]:
        t, box = synthetic_frame(fmax, 160)
        assert ThermalScreener(lambda g: [box], cfg).process_celsius(t)[0].alert is expect


def test_reading_row_has_no_image_data():
    t, box = synthetic_frame(36.5, 160)
    r = ThermalScreener(lambda g: [box]).process_celsius(t)[0]
    assert set(reading_row("x", r, "t")) == {"time", "source", "status", "face_max_c", "roi_height_px", "alert"}


def test_fda_bundle_is_deidentified():
    df = pd.read_csv(DATA)
    assert "oral_c" in df.columns and len(df) > 900
    assert not {"SubjectID", "Gender", "Age", "Ethnicity"} & set(df.columns)


def test_app_default_run_shows_fda_metrics():
    from streamlit.testing.v1 import AppTest
    at = AppTest.from_file(APP, default_timeout=60).run()
    assert not at.exception
    assert len(at.metric) >= 4


def test_fda_threshold_changes_sensitivity():
    from streamlit.testing.v1 import AppTest
    at = AppTest.from_file(APP, default_timeout=60).run()
    sens = lambda: [m.value for m in at.metric if m.label == "Sensitivity"][0]
    at.slider(key="thr_c").set_value(37.0).run(); low = sens()
    at.slider(key="thr_c").set_value(37.8).run(); high = sens()
    assert not at.exception and low != high
'''

# ======================= README.md =======================
README = r'''# Thermal Face-Temperature Screening

![python](https://img.shields.io/badge/python-3.13-blue) ![license](https://img.shields.io/badge/license-MIT-green) ![status](https://img.shields.io/badge/status-research%20prototype-orange)

A research prototype that turns a radiometric thermal frame into a face-temperature reading and measures, honestly, how far that reading can be trusted. It combines a fail-safe pipeline with analyses on four public datasets. Every number below comes from a result file in `docs/results/` or from the cited dataset.

> **Not a medical device.** Face surface temperature is not core body temperature. The original brief asked for an error of at most 0.3 °C. That target is met only for expert-extracted face regions at about 0.8 m on one camera model. End-to-end accuracy with automatic face detection could not be validated, because no public dataset pairs thermal images with an oral-temperature reference.

**Contents:** [Key findings](#key-findings) · [Scope](#scope-against-the-original-brief) · [System design](#system-design) · [Results](#results) · [Dashboard and API](#dashboard-and-api) · [Limitations](#limitations) · [Datasets](#datasets-and-licences) · [Repository structure](#repository-structure) · [Quick start](#quick-start) · [Reproducibility](#reproducibility) · [Citations](#citations)

## Key findings

1. **Raw facial temperature is not body temperature.** On the FDA data the whole-face maximum reads about 0.9 °C below the oral thermometer and the inner canthus about 1.2 °C below. A linear calibration brings the mean absolute error to 0.21 to 0.25 °C.
2. **The 0.3 °C target holds only in a narrow regime:** single frame MAE 0.23 to 0.26 °C and RMSE 0.30 to 0.33 °C, expert ROI, 0.8 m, one camera. Subjects with fever (n = 111) are under-estimated (bias -0.27 °C); with four-round-averaged features and a 37.5 °C threshold sensitivity is 0.76, so roughly one in four fevers is missed.
3. **Distance matters more than the model.** On Charlotte-ThermalFace the measured face maximum falls by 1.15 to 1.49 °C between 1 m and 6.6 m for the same person. Down-sampling explains only about a quarter of that drop.
4. **Calibrations do not transfer between cameras.** Two cameras differ by about 0.6 °C at comparable distance, so the shipped FDA calibration is not applied by default.
5. **The pipeline fails safe.** No face, a face under 100 px, invalid input or no calibration produces a status, never an alert.

@@results_overview@@

## Scope against the original brief

| Brief item | Status |
|---|---|
| Person detection | Evaluated on LLVIP; optional module, not part of the thermal-only pipeline (ResNet50 needs about 6.9 s per frame on a Colab CPU) |
| Face detection | MediaPipe on thermal frames (in the pipeline) and on RGB frames (demonstrated on SpeakingFaces) |
| RGB-thermal alignment | `HomographyRegistration` (unit tested) and an eye-based `FaceAligner`, demonstrated on one SpeakingFaces subject; not wired into the thermal-only pipeline |
| Face to person association | Not implemented |
| Temperature from thermal data | Implemented (16-bit counts to °C, assumed 0.01 K per count) |
| Error of at most 0.3 °C | See key finding 2 |
| At most 1 s per frame | Face detection, ROI and reading take 4 to 22 ms on a Colab CPU (350x465 frames) |
| Live camera input | Not implemented or tested |
| History and report export | Session history in memory, CSV download |

## System design

@@architecture@@

@@pipeline@@

## Results

### 1. Facial temperature vs oral thermometer (FDA / PhysioNet)

One FLIR camera at about 0.8 m, 1,020 subjects, linear calibration with 5-fold cross-validation. Features are the dataset's own extracted values (expert ROI), not images.

| Feature | Raw bias (°C) | Calibrated MAE | Calibrated RMSE |
|---|---|---|---|
| T_Max, mean of 4 rounds | -0.94 | 0.210 (95% CI 0.200 to 0.220) | 0.271 |
| Inner-canthus max, mean of 4 rounds | -1.24 | 0.248 (95% CI 0.236 to 0.260) | 0.319 |
| T_Max, single round | not measured | 0.234 to 0.259 | 0.304 to 0.333 |
| Inner-canthus max, single round | not measured | 0.250 to 0.251 | 0.322 to 0.327 |

@@fda_regions@@

*Left: which facial measurement tracks the oral temperature best (variable names as in the dataset, whose Table 1 defines them). Middle: ROC curves for detecting fever (oral >= 37.5 °C). Right: confusion matrix for T_Max at a fixed 37.5 °C flag. The temperature estimate is a measured pixel maximum plus a linear fit, not a neural network, so there is no Grad-CAM; the question "which face region carries the signal" is answered by comparing regions directly (left) and by the hotspot map in section 2.*

* Threshold 37.2 °C gives sensitivity 0.95 and specificity 0.84 with four-round features, but it was chosen on the same data and is exploratory only.
* Round 4 sits about 0.4 °C below rounds 1 to 3 (per-round residual means +0.15, +0.07, +0.06, -0.29 °C). The cause was not investigated.
* The saved calibration (slope 0.587, intercept 15.84) is valid for the FDA camera and ROI definition only. Applied to Charlotte-ThermalFace it would flag healthy volunteers.

### 2. Distance effect and where the maximum comes from (Charlotte-ThermalFace)

Mean face maximum inside the landmark hull, all subjects pooled (the subject mix varies by distance):

| Distance | Face pixels | Face max (°C) |
|---|---|---|
| 1.0 m | about 14,900 | 36.47 |
| 1.4 m | about 7,600 | 36.42 |
| 1.8 m | about 4,600 | 36.12 |
| 2.8 m | about 1,900 | 35.97 |
| 4.0 m | about 900 | 35.75 |
| 6.6 m | about 375 | 35.33 |

For the 5 subjects recorded at both 1 m and 6.6 m the face maximum falls by 1.15 to 1.49 °C. Down-sampling the 1 m images to the 6.6 m face size reproduces only 19 to 32% of that drop; optics, camera distance compensation, landmark error and subject cooling are untested explanations. Readings are therefore flagged unreliable below a provisional face height of 100 px (about 1.4 m on this sensor).

@@charlotte_hotspots@@

*Position of the hottest pixel inside the landmark hull, relative to the landmark box, over frontal frames up to 1.4 m. It describes where the maximum tends to occur; it does not prove that the reading is correct.*

### 3. ROI placement (Charlotte-ThermalFace, 600 images, 4 subjects)

| Distance | ROI shifted by 10% of face height | Automatic box vs landmark ROI | IoU with landmark box |
|---|---|---|---|
| 1.0 m | 0.05 °C | 0.15 °C | 0.70 |
| 2.8 m | 0.04 °C | 0.08 °C | 0.59 |
| 6.6 m | 0.01 °C | 0.05 °C | 0.40 |

Shifting the ROI barely changes the face maximum (above 0.1 °C in 9% of images at a 10% shift). This holds for one warm face on a cold background and says nothing about crowds, rotation, scale error or camera parallax. The automatic box reads about 0.15 °C higher than the landmark ROI at 1 m, a bias on top of the calibration.

### 4. Face keypoints, alignment and RGB-thermal registration (SpeakingFaces)

@@face_alignment@@

*MediaPipe finds the face box and six keypoints (eyes, nose, mouth, ears). The eye keypoints drive a similarity transform that gives a canonical 256x256 crop, and the same transform is applied to the thermal frame. In this dataset the aligned RGB and thermal frames share one geometry. The thermal frames are 8-bit colourised PNGs, so they carry no temperature.*

@@alignment_demo@@

*A homography estimated once from one frame of a camera position is applied to another frame of the same position. The IoU in the figure title compares the mapped box with a box MediaPipe finds directly on the thermal image: a proxy, not ground truth. The homography is stable within a camera position (standard deviation 0.10 to 0.27 px) and differs between the nine positions.*

### 5. Person detection (LLVIP, first 500 consecutive test pairs, IoU 0.5)

Torchvision Faster R-CNN with COCO weights.

| Model / modality | AP50 | Precision | Recall | Median time (T4 GPU) |
|---|---|---|---|---|
| MobileNet / visible | 0.707 | 0.791 | 0.639 | 59 ms |
| MobileNet / infrared | 0.431 | 0.426 | 0.535 | 59 ms |
| ResNet50 / visible | 0.783 | 0.814 | 0.735 | 156 ms |
| ResNet50 / infrared | 0.836 | 0.829 | 0.780 | 155 ms |

On a Colab CPU MobileNet takes about 0.56 s and ResNet50 about 6.9 s per frame. The 500 frames come from a few scenes and there are no confidence intervals.

### 6. Error budget (estimate, not measured)

Calibration RMSE 0.30 to 0.33 °C, automatic-ROI bias about 0.15 °C and the 1.4 m distance effect about 0.05 °C combine to roughly 0.35 °C RMSE at 1.4 m or closer, assuming independent errors. It cannot be checked end to end because no public dataset pairs images with an oral reference.

## Dashboard and API

The Streamlit dashboard (`streamlit run frontend/app.py`) has three tabs: **Screen an image** (upload a 16-bit radiometric frame for a measured face maximum, status and CSV report, or an 8-bit frame for detection only), **FDA calibration explorer** (choose a facial measurement and a threshold, see MAE, RMSE, sensitivity, specificity, scatter and ROC on the 1,020 FDA subjects) and **Method and limits**. Nothing is stored. The bundled FDA table contains no subject identifiers or demographics.

The FastAPI endpoint (`uvicorn backend.app:app`) accepts a single-channel 16-bit image up to 10 MB at `POST /screen` and returns the status and face maximum per face. `calibrated` is false unless `CALIBRATION_JSON` points to a calibration file.

## Limitations

* The 0.01 K per count scale is an assumption. It gives plausible skin temperatures on Charlotte-ThermalFace and its face maxima are close to the FDA ones (35.8 °C vs 36.4 °C at about 1 m), but absolute accuracy was not checked against a blackbody reference.
* The two cameras differ by about 0.6 °C at comparable distance, so calibrations do not transfer.
* Public datasets are mostly controlled indoor scenes with one subject. Crowds, outdoor conditions, glasses, masks and fever-range subjects beyond FDA's 111 are untested.
* Close-range screening (about 1 to 1.4 m) is the validated regime. Distant crowd monitoring needs higher-resolution optics.
* Charlotte-ThermalFace: one annotation session (S3) is missing from the zip and S2 has corrupted distance values; affected rows were excluded.
* SpeakingFaces thermal frames provided here are 8-bit colourised images, so the dataset supports detection and registration only.
* Deployment to a hosted demo is paused; the repository contains the scripts for a Hugging Face Space.

## Datasets and licences

| Dataset | Used for | Terms |
|---|---|---|
| FDA / PhysioNet face and oral temperature data | Calibration and error analysis; de-identified table bundled with the dashboard | CC0 |
| Charlotte-ThermalFace | Distance and ROI analysis | Research use only, no licence file; images are not redistributed |
| LLVIP | Person-detector evaluation | Non-commercial; images are not redistributed |
| SpeakingFaces (sample, one subject) | Detection and registration figures | CC BY 4.0 (ISSAI); participants consented to public sharing |

Pretrained COCO weights come from torchvision; check their terms before any commercial use.

## Repository structure

```text
.
├── README.md
├── LICENSE
├── requirements.txt
├── configs/            calibration for the FDA camera only (not transferable)
├── src/
│   ├── pipeline/       ThermalScreener: fail-safe screening logic
│   ├── temperature/    radiometric conversion, FDA baseline analysis
│   ├── detection/      face detector wrapper, person detector (LLVIP evaluation)
│   ├── alignment/      HomographyRegistration, eye-based FaceAligner
│   └── utils/          StageTracker: resumable stages, atomic checkpoints
├── backend/            FastAPI /screen endpoint
├── frontend/           Streamlit dashboard and bundled FDA table
├── scripts/            make_figures.py, deploy_space.py
├── notebooks/          research log (Colab)
├── docs/
│   ├── images/         figures used in this README
│   └── results/        evaluation outputs (JSON and CSV)
├── deploy/             Hugging Face Space files (deployment paused)
└── tests/              unit, API and dashboard tests
```

## Quick start

```bash
pip install -r requirements.txt
python -m pytest -q
streamlit run frontend/app.py
uvicorn backend.app:app          # then: curl -F "file=@frame.tiff" http://localhost:8000/screen
```

## Reproducibility

* Research ran on Google Colab with datasets, checkpoints and results on Google Drive. `StageTracker` records finished stages in `progress.json` with atomic writes, so a disconnected runtime resumes where it stopped. Stages whose output lives on the volatile Colab disk must be re-run after a reset (the tracker supports a `verify` check).
* `scripts/make_figures.py` regenerates the diagrams, the FDA region figure, the hotspot map and the keypoint figure from the datasets on Drive. The registration figure and the results overview come from the research notebook in `notebooks/`, whose outputs are cleared because dataset images may not be redistributed.

## Privacy and ethics

The tool performs no identification or face recognition. The dashboard keeps readings in session memory only and the API processes uploads in memory. Thermal screening in public spaces raises consent, retention and discrimination questions that this prototype does not solve.

## Citations

* Charlotte-ThermalFace: "A Novel Fully Annotated Thermal Infrared Face Dataset: Recorded in Various Environment Conditions and Distances From The Camera", arXiv:2205.02093.
* LLVIP: Jia et al., "LLVIP: A Visible-infrared Paired Dataset for Low-light Vision", ICCV Workshops 2021.
* FDA / PhysioNet: https://physionet.org/content/face-oral-temp-data/1.0.0/
* SpeakingFaces: Abdrakhmanova, Kuzdeuov, Jarju, Khassanov, Lewis, Varol, "SpeakingFaces: A Large-Scale Multimodal Dataset of Voice Commands with Visual and Thermal Video Streams", Sensors 21(10):3465, 2021, doi:10.3390/s21103465. ISSAI dataset: doi:10.48333/SMGD-YJ77.

## Licence

Code: MIT (see `LICENSE`). Dataset terms above apply separately.
'''

# ======================= write everything =======================
put("scripts/make_figures.py", MAKE_FIGURES)
put("frontend/app.py", APP)
put("tests/test_frontend.py", TESTS)
put(".streamlit/config.toml", '[theme]\nbase = "dark"\nprimaryColor = "#ff6b35"\nbackgroundColor = "#0e1117"\nsecondaryBackgroundColor = "#1a1f2b"\ntextColor = "#fafafa"\n')
put("notebooks/README.md", "Research log (Google Colab). Outputs are cleared before committing because dataset images must not be redistributed.\n")

fl = pd.read_csv(ROOT/"datasets/fda_physionet/clean_FLIR_groups1and2.csv")
cols = ["aveOralM"] + [f"{b}{k}" for b in ["T_Max", "canthiMax", "T_FH_Max"] for k in range(1, 5)]
missing = [c for c in cols if c not in fl.columns]; assert not missing, missing
(REPO/"frontend/data").mkdir(parents=True, exist_ok=True)
fl[cols].rename(columns={"aveOralM": "oral_c"}).round(2).to_csv(REPO/"frontend/data/fda_flir_features.csv", index=False)
put("frontend/data/NOTICE.txt", "Source: FDA / PhysioNet face and oral temperature data (CC0). De-identified: no subject IDs or demographics.\n")

req = REPO/"deploy/space_requirements.txt"
if req.exists():
    have = req.read_text()
    for pkg in ["scikit-learn", "matplotlib"]:
        if pkg not in have: have += f"{pkg}=={md.version(pkg)}\n"
    req.write_text(have)

# ======================= figures, README, tests, commit =======================
r = subprocess.run([sys.executable, "scripts/make_figures.py", str(ROOT)], cwd=REPO, capture_output=True, text=True)
print(r.stdout[-1500:]); print(r.stderr[-1500:])

IMG = REPO/"docs/images"
text = README
for key, alt in [("results_overview", "Results overview"), ("architecture", "System architecture"), ("pipeline", "Screening pipeline"),
                 ("fda_regions", "FDA regions, ROC and confusion matrix"), ("charlotte_hotspots", "Hotspot map"),
                 ("face_alignment", "Face keypoints and alignment"), ("alignment_demo", "RGB-thermal registration")]:
    ok = (IMG/f"{key}.png").exists()
    text = text.replace(f"@@{key}@@", f"![{alt}](docs/images/{key}.png)" if ok else f"*(figure {key}.png is missing)*")
put("README.md", text)

t = subprocess.run([sys.executable, "-m", "pytest", "-q"], cwd=REPO, capture_output=True, text=True)
print(t.stdout[-600:])

token = userdata.get('GITHUB_TOKEN'); cwd = str(REPO)
def sh(*a): return subprocess.run(a, cwd=cwd, capture_output=True, text=True)
if t.returncode != 0:
    print("TESTS FAILED: nothing committed. Paste the pytest output.")
else:
    sh("git", "rm", "-f", "--ignore-unmatch", "docs/images/screening_logic.png", "docs/images/dashboard_synthetic.png")
    sh("git", "add", "README.md", "LICENSE", "frontend", "scripts", "tests", "docs", "deploy", ".streamlit", "notebooks", "src", "backend", "configs", "requirements.txt")
    print(sh("git", "status", "--short").stdout)
    print(sh("git", "commit", "-m", "docs: redesign README with figures; feat: new dashboard with FDA explorer; add figure script").stdout)
    p = sh("git", "push", f"https://{token}@github.com/NidaKhaan/intelligent-body-temperature-monitoring.git", "main")
    print(p.returncode, (p.stdout + p.stderr).replace(token, "***"))

In [ ]:
%%bash
pip -q install mediapipe==1.2.2 streamlit==1.67.0 2>&1 | tail -2

In [ ]:
import subprocess, sys
from google.colab import userdata
REPO = ROOT/"repo"

r = subprocess.run([sys.executable, "scripts/make_figures.py", str(ROOT)], cwd=REPO, capture_output=True, text=True)
print(r.stdout[-900:]); print(r.stderr[-700:])

rd = REPO/"README.md"; s = rd.read_text(encoding="utf-8")
miss = "*(figure face_alignment.png is missing)*"
if miss in s and (REPO/"docs/images/face_alignment.png").exists():
    rd.write_text(s.replace(miss, "![Face keypoints and alignment](docs/images/face_alignment.png)"), encoding="utf-8")
s = rd.read_text(encoding="utf-8")
print("README mein baaqi placeholders:", [k for k in ["is missing", "@@"] if k in s])

t = subprocess.run([sys.executable, "-m", "pytest", "-q"], cwd=REPO, capture_output=True, text=True)
print(t.stdout[-1200:])

token = userdata.get('GITHUB_TOKEN'); cwd = str(REPO)
def sh(*a): return subprocess.run(a, cwd=cwd, capture_output=True, text=True)
if t.returncode != 0:
    print("TESTS FAILED: kuch commit nahi hua. Upar ka pytest output paste karein.")
else:
    sh("git", "rm", "-f", "--ignore-unmatch", "docs/images/screening_logic.png", "docs/images/dashboard_synthetic.png")
    sh("git", "add", "README.md", "LICENSE", "frontend", "scripts", "tests", "docs", "deploy", ".streamlit", "notebooks", "src", "backend", "configs", "requirements.txt")
    print(sh("git", "status", "--short").stdout)
    print(sh("git", "commit", "-m", "docs: redesign README with figures; feat: new dashboard with FDA explorer; add figure script").stdout)
    p = sh("git", "push", f"https://{token}@github.com/NidaKhaan/intelligent-body-temperature-monitoring.git", "main")
    print(p.returncode, (p.stdout + p.stderr).replace(token, "***"))